# Manuscript evidence pack — v53 split GPU energy scales

GPU energy is now split into two benchmark regimes so figures do not mix very different joule scales. Figure 2a1 covers Dolly and GSM8K; Figure 2a2 covers HellaSwag and MMLU. Figure 3b1 and Figure 3b2 show the corresponding request-energy distributions as grouped boxplots. Figure 3a remains the last main GPU resource figure.


## 1. Configuration and input discovery

The output directory is version-specific so obsolete plots from earlier notebook versions are not mixed with manuscript artifacts.


In [ ]:
from pathlib import Path
import json
import re
import shutil
import textwrap
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.lines import Line2D
from matplotlib import colors as mcolors
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

SCORED_DATASETS = ["mmlu", "gsm8k", "hellaswag"]
ALL_WORKLOADS = ["mmlu", "gsm8k", "hellaswag", "dolly"]

# Alphabetical workload order used consistently in categorical figures.
BENCHMARK_ORDER_ALPHABETICAL = [
    "dolly",
    "gsm8k",
    "hellaswag",
    "mmlu",
]
# Display order: Dolly, GSM8K, HellaSwag, MMLU.
SCORED_BENCHMARK_ORDER_ALPHABETICAL = [
    "gsm8k",
    "hellaswag",
    "mmlu",
]

DATASET_DISPLAY = {
    "mmlu": "MMLU",
    "gsm8k": "GSM8K",
    "hellaswag": "HellaSwag",
    "dolly": "Dolly",
}
CONTEXT_ORDER = ["edge", "local_cpu", "gpu_server"]
CONTEXT_DISPLAY = {
    "edge": "Edge",
    "local_cpu": "Local CPU",
    "gpu_server": "GPU server",
}
PLATFORM_TO_CONTEXT = {
    "pi5": "edge",
    "cpu": "local_cpu",
    "vllm": "gpu_server",
}
PLATFORM_ALIASES = {
    "gpu": "vllm",
    "gpu_vllm": "vllm",
    "cuda": "vllm",
    "raspberry_pi_5": "pi5",
    "raspberrypi5": "pi5",
    "raspberry_pi": "pi5",
    "local_cpu": "cpu",
}
ACCURACY_RETENTION_THRESHOLD = 0.95
COMPACT_TOTAL_PARAMETER_THRESHOLD_B = 7.0
SAVE_DPI = 600
OUTPUT_SUBDIR = Path("results") / "manuscript_evidence_pack_v53"
CLEAN_OUTPUT_ON_RUN = True

SINGLE_COLUMN_WIDTH_IN = 3.45
DOUBLE_COLUMN_WIDTH_IN = 7.16
SHORT_PANEL_HEIGHT_IN = 2.35
MEDIUM_PANEL_HEIGHT_IN = 2.85
TALL_PANEL_HEIGHT_IN = 3.55
BASE_FONT_SIZE = 8

# Colorblind-safe publication palette.
# Dataset colors follow the Okabe–Ito family; contexts use a separate,
# equally distinguishable palette because dataset and context encodings
# appear in different figure families.
DATASET_COLORS = {
    "mmlu": "#0072B2",       # blue
    "gsm8k": "#D55E00",      # vermillion
    "hellaswag": "#009E73",  # bluish green
    "dolly": "#CC79A7",      # reddish purple
}

# Redundant encodings for color-vision accessibility and grayscale printing.
# The palette is color-blind-friendly, while hatches and marker shapes ensure
# that no category or Pareto status is communicated by color alone.
DATASET_HATCHES = {
    "mmlu": "xx",
    "gsm8k": "///",
    "hellaswag": r"\\\\",
    "dolly": "",
}

CONTEXT_HATCHES = {
    "edge": "///",
    "local_cpu": r"\\\\",
    "gpu_server": "xx",
}

DATASET_MARKERS = {
    "mmlu": "o",
    "gsm8k": "s",
    "hellaswag": "^",
    "dolly": "D",
}
CONTEXT_COLORS = {
    "edge": "#E69F00",       # amber
    "local_cpu": "#0072B2",  # blue
    "gpu_server": "#009E73", # green
}
NEUTRAL_DARK = "#2F3B46"
NEUTRAL_MID = "#7A8793"
NEUTRAL_LIGHT = "#D9E0E6"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": BASE_FONT_SIZE,
    "axes.labelsize": 8,
    "axes.titlesize": 8.5,
    "axes.titleweight": "semibold",
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.titlesize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.20,
    "lines.linewidth": 1.2,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

cwd = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / "results").exists():
            return candidate
    return start.parents[0] if start.name == "notebooks" else start

# Exact root confirmed by the executed CPU, Pi 5, and GPU notebooks.
CONFIRMED_ANALYSIS_ROOT = Path(
    "/home/stephane/resource-efficiency-benchmark"
)

ROOT = (
    CONFIRMED_ANALYSIS_ROOT
    if CONFIRMED_ANALYSIS_ROOT.exists()
    else find_project_root(cwd)
)

PLATFORM_PROCESSED_SAMPLE_PATHS = {
    "cpu": ROOT / "results" / "cpu" / "processed" / "cpu" / "samples.csv",
    "pi5": ROOT / "results" / "pi5" / "processed" / "pi5" / "samples.csv",
    "vllm": ROOT / "results" / "processed" / "vllm" / "samples.csv",
}

PLATFORM_MODEL_SUMMARY_PATHS = {
    "cpu": (
        ROOT / "results" / "cpu" / "processed" / "cpu"
        / "model_dataset_summary.csv"
    ),
    "pi5": (
        ROOT / "results" / "pi5" / "processed" / "pi5"
        / "model_dataset_summary.csv"
    ),
    "vllm": (
        ROOT / "results" / "processed" / "vllm"
        / "model_dataset_summary.csv"
    ),
}

CANDIDATE_DIRS = [
    ROOT / "results" / "report_grade_visualizations",
    ROOT / "results",
    ROOT / "notebooks",
    ROOT,
    cwd,
    Path("/mnt/data"),
]


PLATFORM_PROCESSED_SUMMARY_PATHS = {
    "cpu": (
        ROOT
        / "results"
        / "cpu"
        / "processed"
        / "cpu"
        / "summary.csv"
    ),
    "pi5": (
        ROOT
        / "results"
        / "pi5"
        / "processed"
        / "pi5"
        / "summary.csv"
    ),
    "vllm": (
        ROOT
        / "results"
        / "processed"
        / "vllm"
        / "summary.csv"
    ),
}


def find_file_optional(filename: str) -> Path | None:
    """Return an existing project file without raising when it is absent."""
    for directory in CANDIDATE_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate

    for search_root in [ROOT / "results", ROOT, cwd]:
        if not search_root.exists():
            continue
        matches = sorted(search_root.rglob(filename))
        if matches:
            return matches[0]

    return None


def normalize_platform_token(value) -> str:
    token = str(value).strip().lower().replace("-", "_").replace(" ", "_")
    return PLATFORM_ALIASES.get(token, token)


def discover_processed_summary(
    platform: str,
) -> Path | None:
    """Locate the processed platform summary exported by analysis notebooks."""
    explicit = PLATFORM_PROCESSED_SUMMARY_PATHS.get(platform)
    if explicit is not None and explicit.exists():
        return explicit

    results_root = ROOT / "results"
    if not results_root.exists():
        return None

    candidates = []
    for match in sorted(results_root.rglob("summary.csv")):
        match_text = str(match).lower()
        if (
            "manuscript_evidence_pack" in match_text
            or "/raw/" in match_text
        ):
            continue

        try:
            preview = pd.read_csv(match, nrows=50)
        except Exception:
            continue

        if "platform" not in preview.columns:
            continue

        platforms = {
            normalize_platform_token(value)
            for value in preview["platform"].dropna().unique()
        }
        if platform in platforms:
            candidates.append(match)

    return candidates[0] if candidates else None


def build_normalized_summary_from_platform_exports(
) -> tuple[Path, pd.DataFrame, pd.DataFrame]:
    """Build the manuscript input directly from platform `summary.csv` files.

    This replaces the former dependency on a separate report-grade notebook.
    Each source row is preserved. The context-aware aggregation later in this
    notebook applies means to ordinary metrics and maxima to peak metrics.
    """
    frames = []
    audit_rows = []

    for platform in ["cpu", "pi5", "vllm"]:
        source_path = discover_processed_summary(platform)

        if source_path is None:
            audit_rows.append({
                "platform": platform,
                "source_file": None,
                "rows": 0,
                "columns": 0,
                "status": "not found",
            })
            continue

        try:
            frame = pd.read_csv(source_path)
        except Exception as exc:
            audit_rows.append({
                "platform": platform,
                "source_file": str(source_path),
                "rows": 0,
                "columns": 0,
                "status": f"read error: {exc}",
            })
            continue

        if frame.empty:
            audit_rows.append({
                "platform": platform,
                "source_file": str(source_path),
                "rows": 0,
                "columns": len(frame.columns),
                "status": "empty",
            })
            continue

        if "platform" not in frame.columns:
            frame["platform"] = platform
        else:
            frame["platform"] = frame["platform"].map(
                normalize_platform_token
            )

        if "dataset" in frame.columns:
            frame["dataset"] = (
                frame["dataset"]
                .astype(str)
                .str.strip()
                .str.lower()
            )

        # Keep the originating processed file for reproducibility.
        frame["normalized_source_file"] = str(source_path)
        frames.append(frame)

        audit_rows.append({
            "platform": platform,
            "source_file": str(source_path),
            "rows": len(frame),
            "columns": len(frame.columns),
            "status": "loaded",
        })

    audit = pd.DataFrame(audit_rows)

    if not frames:
        expected = "\n".join(
            f"  - {platform}: {path}"
            for platform, path
            in PLATFORM_PROCESSED_SUMMARY_PATHS.items()
        )
        raise FileNotFoundError(
            "No processed platform summary files were found. "
            "Run the CPU, Pi 5, and GPU analysis notebooks first. "
            "Expected locations include:\n"
            + expected
        )

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    required = {"platform", "dataset", "model"}
    missing = sorted(required - set(combined.columns))
    if missing:
        raise KeyError(
            "The combined processed summaries are missing required "
            f"columns {missing}. Available columns: "
            f"{sorted(combined.columns)}"
        )

    destination_dir = (
        ROOT
        / "results"
        / "report_grade_visualizations"
    )
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / "normalized_summary_all.csv"
    combined.to_csv(destination, index=False)

    return destination, combined, audit


def load_or_build_normalized_summary(
) -> tuple[Path, pd.DataFrame, pd.DataFrame, str]:
    """Load an existing normalized file or rebuild it when needed."""
    existing_path = find_file_optional("normalized_summary_all.csv")

    processed_paths = {
        platform: discover_processed_summary(platform)
        for platform in ["cpu", "pi5", "vllm"]
    }
    available_processed = [
        path for path in processed_paths.values()
        if path is not None and path.exists()
    ]

    should_rebuild = existing_path is None

    # Rebuild when a platform notebook has produced newer processed results.
    if existing_path is not None and available_processed:
        normalized_mtime = existing_path.stat().st_mtime
        should_rebuild = any(
            path.stat().st_mtime > normalized_mtime
            for path in available_processed
        )

    if should_rebuild:
        path, frame, audit = (
            build_normalized_summary_from_platform_exports()
        )
        return path, frame, audit, "built from processed platform summaries"

    frame = pd.read_csv(existing_path)
    audit = pd.DataFrame([
        {
            "platform": "combined",
            "source_file": str(existing_path),
            "rows": len(frame),
            "columns": len(frame.columns),
            "status": "loaded existing normalized summary",
        }
    ])
    return existing_path, frame, audit, "loaded existing normalized summary"


(
    SUMMARY_PATH,
    summary_raw,
    normalized_summary_source_audit,
    SUMMARY_SOURCE_MODE,
) = load_or_build_normalized_summary()

OUTPUT_DIR = ROOT / OUTPUT_SUBDIR
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
TEXT_DIR = OUTPUT_DIR / "text"
CHECK_DIR = OUTPUT_DIR / "checks"

if CLEAN_OUTPUT_ON_RUN and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for directory in [
    FIGURE_DIR,
    TABLE_DIR,
    TEXT_DIR,
    CHECK_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

normalized_summary_source_audit.to_csv(
    CHECK_DIR / "normalized_summary_source_audit.csv",
    index=False,
)

print("Confirmed analysis root:", ROOT)
print("Normalized summary:", SUMMARY_PATH)
print("Summary mode:", SUMMARY_SOURCE_MODE)
print("Rows:", len(summary_raw))
print("Processed platform summaries:")
display(normalized_summary_source_audit)

print("Confirmed processed sample inputs:")
for platform_name, platform_path in (
    PLATFORM_PROCESSED_SAMPLE_PATHS.items()
):
    print(
        f" - {platform_name}: {platform_path} "
        f"(exists={platform_path.exists()})"
    )

missing_confirmed_inputs = {
    platform_name: platform_path
    for platform_name, platform_path
    in PLATFORM_PROCESSED_SAMPLE_PATHS.items()
    if (
        platform_name in {"cpu", "pi5"}
        and not platform_path.exists()
    )
}
if missing_confirmed_inputs:
    warnings.warn(
        "Confirmed processed Dolly sample inputs are missing: "
        + "; ".join(
            f"{platform_name}={platform_path}"
            for platform_name, platform_path
            in missing_confirmed_inputs.items()
        )
        + ". The notebook will then search original per-run "
          "dolly_samples.csv files."
    )


# Dolly was collected in split acquisitions for some llama.cpp configurations
# (for example, 100 samples on one date and the remaining samples later).
# The acquisitions are considered parts of the same experimental point only
# because the protocol-equivalence decision has been confirmed by the author.
DOLLY_SPLIT_PROTOCOL_EQUIVALENCE_CONFIRMED = True


def model_join_key(value) -> str:
    """Normalize model identifiers for joins across raw and processed files."""
    token = re.sub(r"[^a-z0-9]+", "", str(value).lower())
    return token[:-4] if token.endswith("gguf") else token


def first_non_null_column(
    frame: pd.DataFrame,
    candidates: list[str],
    default=np.nan,
) -> pd.Series:
    existing = [column for column in candidates if column in frame.columns]
    if not existing:
        return pd.Series(default, index=frame.index)
    return frame[existing].bfill(axis=1).iloc[:, 0]


def load_platform_dolly_samples(
    platform: str,
) -> tuple[pd.DataFrame, str]:
    """Load Dolly samples without requiring the analysis notebooks to be rerun.

    Priority:
    1. processed platform ``samples.csv``;
    2. original per-run ``dolly_samples.csv`` files;
    3. any compatible processed ``samples.csv`` found below ``results``.
    """
    processed_path = PLATFORM_PROCESSED_SAMPLE_PATHS.get(platform)
    if processed_path is not None and processed_path.exists():
        processed = pd.read_csv(processed_path)
        if {"platform", "dataset"}.issubset(processed.columns):
            processed_platform = (
                processed["platform"].astype(str).str.strip().str.lower()
            )
            processed_dataset = (
                processed["dataset"].astype(str).str.strip().str.lower()
            )
            selected = processed[
                processed_platform.eq(platform)
                & processed_dataset.eq("dolly")
            ].copy()
            if not selected.empty:
                return selected, str(processed_path)

    # Read the original per-run sample files directly. This is the important
    # fallback for split acquisitions such as 100 samples on one date and 900
    # on another date.
    raw_roots = {
        "cpu": [
            ROOT / "results" / "cpu",
            ROOT / "results" / "raw" / "cpu",
        ],
        "pi5": [
            ROOT / "results" / "pi5",
            ROOT / "results" / "raw" / "pi5",
        ],
        "vllm": [
            ROOT / "results" / "vllm",
            ROOT / "results" / "raw" / "vllm",
        ],
    }

    raw_paths = []
    seen = set()
    for raw_root in raw_roots.get(platform, []):
        if not raw_root.exists():
            continue
        for match in sorted(raw_root.rglob("dolly_samples.csv")):
            match_text = str(match).lower()
            if (
                "/processed/" in match_text
                or "manuscript_evidence_pack" in match_text
            ):
                continue
            key = str(match.resolve())
            if key not in seen:
                seen.add(key)
                raw_paths.append(match)

    raw_parts = []
    for sample_path in raw_paths:
        try:
            raw = pd.read_csv(sample_path)
        except Exception as exc:
            warnings.warn(f"Could not read {sample_path}: {exc}")
            continue
        if raw.empty:
            continue

        raw = raw.copy()
        run_dir = sample_path.parent
        model_dir = run_dir.parent.name

        raw["platform"] = platform
        raw["dataset"] = "dolly"
        raw["run_id"] = raw.get(
            "run_id",
            pd.Series(run_dir.name, index=raw.index),
        )
        raw["run_dir"] = str(run_dir)
        raw["source_file"] = str(sample_path)
        raw["model_dir"] = model_dir

        if "model" not in raw.columns:
            raw["model"] = first_non_null_column(
                raw,
                [
                    "model_name",
                    "Model name",
                    "model_alias",
                    "Model alias",
                    "served_model_id",
                    "model_path",
                    "repo_id",
                    "Repo id",
                    "model_dir",
                ],
                default=model_dir,
            )
        else:
            missing_model = raw["model"].isna()
            raw.loc[missing_model, "model"] = model_dir

        # The directory name is the most stable identifier in the llama.cpp
        # experiment layout and normally matches normalized_summary_all.csv.
        raw["model"] = raw["model"].fillna(model_dir).astype(str)
        raw["model_join_key"] = raw["model"].map(model_join_key)
        directory_key = model_join_key(model_dir)
        if directory_key:
            raw["model_join_key"] = directory_key

        if "sample_idx" not in raw.columns:
            raw["sample_idx"] = np.arange(len(raw), dtype=int)

        # Normalize the sample-level metric names expected by the pooling code.
        if "latency_ms" not in raw.columns:
            raw["latency_ms"] = first_non_null_column(
                raw,
                ["end_to_end_latency_ms", "latency_ms"],
            )
        if "ttft_ms" not in raw.columns:
            raw["ttft_ms"] = first_non_null_column(
                raw,
                ["ttft_ms", "time_to_first_token_ms"],
            )
        if "tokens_per_sec" not in raw.columns:
            raw["tokens_per_sec"] = first_non_null_column(
                raw,
                ["tokens_per_sec", "token_throughput"],
            )

        raw_parts.append(raw)

    if raw_parts:
        combined = pd.concat(raw_parts, ignore_index=True, sort=False)
        return combined, "; ".join(map(str, raw_paths))

    # Last-resort discovery of another processed samples.csv location.
    results_root = ROOT / "results"
    if results_root.exists():
        for match in sorted(results_root.rglob("samples.csv")):
            match_text = str(match).lower()
            if "manuscript_evidence_pack" in match_text:
                continue
            try:
                candidate = pd.read_csv(match)
            except Exception:
                continue
            if not {"platform", "dataset"}.issubset(candidate.columns):
                continue
            candidate_platform = (
                candidate["platform"].astype(str).str.strip().str.lower()
            )
            candidate_dataset = (
                candidate["dataset"].astype(str).str.strip().str.lower()
            )
            selected = candidate[
                candidate_platform.eq(platform)
                & candidate_dataset.eq("dolly")
            ].copy()
            if not selected.empty:
                return selected, str(match)

    return pd.DataFrame(), "not found"

def acquisition_date_from_row(row: pd.Series) -> str:
    """Extract an acquisition date from run metadata when available."""
    for field in ["run_id", "run_dir", "source_file"]:
        value = row.get(field, "")
        match = re.search(r"(20\d{2}-\d{2}-\d{2})", str(value))
        if match:
            return match.group(1)
    return "unknown"


def pool_split_dolly_acquisitions(
    normalized: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Pool split Dolly acquisitions at sample level.

    A 100-sample acquisition and a 900-sample acquisition are not averaged as
    two equally weighted observations. The underlying samples are concatenated,
    deduplicated by run/sample identity, and summarized once. This naturally
    gives the two batches weights of 100/1000 and 900/1000.

    The function also records whether the data represent independent
    100-sample runs or a different structure such as one 100-sample run plus
    one 900-sample run. The latter must not be described as ten independent
    measured runs.
    """
    if not DOLLY_SPLIT_PROTOCOL_EQUIVALENCE_CONFIRMED:
        raise RuntimeError(
            "Dolly split acquisitions cannot be pooled until protocol "
            "equivalence across acquisition dates is confirmed."
        )

    pooled = normalized.copy()
    for column in [
        "tokens_per_sec",
        "latency_ms",
        "ttft_ms",
        "ram_used_mb_peak",
        "process_ram_mb_peak",
        "cpu_percent_peak",
        "temperature_c_peak",
    ]:
        if column not in pooled.columns:
            pooled[column] = np.nan

    pooled["tokens_per_sec_backfilled_from_samples"] = 0
    pooled["dolly_split_sample_pool_used"] = 0
    pooled["dolly_peak_metric_pool_used"] = 0
    pooled["dolly_sample_count"] = np.nan
    pooled["dolly_run_count"] = np.nan
    pooled["dolly_acquisition_date_count"] = np.nan

    acquisition_audits = []
    pooling_audits = []

    numeric_columns = [
        "tokens_per_sec",
        "latency_ms",
        "ttft_ms",
        "output_tokens",
        "ram_used_mb_peak",
        "process_ram_mb_peak",
        "cpu_percent_peak",
        "temperature_c_peak",
    ]

    for platform in ["cpu", "pi5"]:
        samples, samples_source = load_platform_dolly_samples(platform)
        print(
            f"Dolly input for {platform}: "
            f"{len(samples)} rows from {samples_source}"
        )
        if samples.empty:
            pooling_audits.append({
                "platform": platform,
                "model": None,
                "sample_count": 0,
                "run_count": 0,
                "acquisition_date_count": 0,
                "run_sample_counts": "",
                "pooling_method": "sample-level weighted pooling",
                "protocol_structure": "Dolly sample files not found",
                "source_file": samples_source,
            })
            continue
        required = {"platform", "dataset", "model"}
        if not required.issubset(samples.columns):
            raise KeyError(
                f"{samples_path} is missing required columns "
                f"{sorted(required - set(samples.columns))}."
            )

        samples["platform"] = (
            samples["platform"].astype(str).str.strip().str.lower()
        )
        samples["dataset"] = (
            samples["dataset"].astype(str).str.strip().str.lower()
        )
        if "model" not in samples.columns:
            samples["model"] = first_non_null_column(
                samples,
                ["model_dir", "served_model_id", "model_path"],
                default="unknown",
            )
        samples["model"] = samples["model"].astype(str)
        if "model_join_key" not in samples.columns:
            samples["model_join_key"] = samples["model"].map(model_join_key)

        dolly = samples[
            samples["platform"].eq(platform)
            & samples["dataset"].eq("dolly")
        ].copy()

        if dolly.empty:
            pooling_audits.append({
                "platform": platform,
                "model": None,
                "sample_count": 0,
                "run_count": 0,
                "acquisition_date_count": 0,
                "run_sample_counts": "",
                "pooling_method": "sample-level weighted pooling",
                "protocol_structure": "no Dolly rows",
                "source_file": samples_source,
            })
            continue

        if "run_id" not in dolly.columns:
            dolly["run_id"] = "unknown_run"
        if "sample_idx" not in dolly.columns:
            dolly["sample_idx"] = np.arange(len(dolly), dtype=int)

        for column in numeric_columns:
            if column in dolly.columns:
                dolly[column] = pd.to_numeric(
                    dolly[column],
                    errors="coerce",
                )

        dolly["acquisition_date"] = dolly.apply(
            acquisition_date_from_row,
            axis=1,
        )

        identity_columns = [
            "platform",
            "dataset",
            "model",
            "run_id",
            "sample_idx",
        ]

        duplicate_count = int(
            dolly.duplicated(identity_columns, keep="first").sum()
        )
        dolly = dolly.drop_duplicates(
            identity_columns,
            keep="first",
        ).copy()

        run_group = [
            "platform",
            "model_join_key",
            "model",
            "run_id",
            "acquisition_date",
        ]
        run_aggregations = {
            "sample_count": ("sample_idx", "count"),
        }
        if "tokens_per_sec" in dolly.columns:
            run_aggregations["tokens_per_sec_mean"] = (
                "tokens_per_sec",
                "mean",
            )
        if "latency_ms" in dolly.columns:
            run_aggregations["latency_ms_mean"] = (
                "latency_ms",
                "mean",
            )
        if "ttft_ms" in dolly.columns:
            run_aggregations["ttft_ms_mean"] = (
                "ttft_ms",
                "mean",
            )

        for peak_column in [
            "ram_used_mb_peak",
            "process_ram_mb_peak",
            "cpu_percent_peak",
            "temperature_c_peak",
        ]:
            if peak_column in dolly.columns:
                run_aggregations[f"{peak_column}_max"] = (
                    peak_column,
                    "max",
                )

        run_summary = (
            dolly.groupby(run_group, dropna=False)
            .agg(**run_aggregations)
            .reset_index()
        )
        run_summary["source_file"] = samples_source
        run_summary["duplicate_samples_removed"] = duplicate_count
        acquisition_audits.append(run_summary)

        for model_key, group in dolly.groupby(
            "model_join_key",
            dropna=False,
        ):
            model = group["model"].dropna().astype(str).iloc[0]
            model_runs = run_summary[
                run_summary["model_join_key"].eq(model_key)
            ].copy()
            run_counts = (
                model_runs["sample_count"]
                .astype(int)
                .sort_values()
                .tolist()
            )
            total_samples = int(len(group))
            run_count = int(model_runs["run_id"].nunique())
            date_count = int(
                model_runs["acquisition_date"].nunique()
            )

            all_runs_are_100 = (
                bool(run_counts)
                and all(count == 100 for count in run_counts)
            )
            has_100_900_split = (
                100 in run_counts and 900 in run_counts
            )

            if all_runs_are_100:
                structure = (
                    f"{run_count} independent-looking 100-sample run IDs"
                )
            elif has_100_900_split:
                structure = (
                    "split acquisition: one 100-sample batch and one "
                    "900-sample batch"
                )
            else:
                structure = (
                    "unequal acquisition sizes: "
                    + ", ".join(map(str, run_counts))
                )

            pool_record = {
                "platform": platform,
                "dataset": "dolly",
                "model": model,
                "model_join_key": model_key,
                "sample_count": total_samples,
                "run_count": run_count,
                "acquisition_date_count": date_count,
                "acquisition_dates": "; ".join(
                    sorted(
                        model_runs["acquisition_date"]
                        .astype(str)
                        .unique()
                    )
                ),
                "run_sample_counts": "; ".join(
                    map(str, run_counts)
                ),
                "pooling_method": (
                    "sample-level pooled mean; batches weighted by "
                    "their sample counts"
                ),
                "protocol_structure": structure,
                "supports_ten_independent_run_claim": (
                    all_runs_are_100 and run_count == 10
                ),
                "source_file": samples_source,
                "duplicate_samples_removed": duplicate_count,
            }

            mean_metrics = [
                "tokens_per_sec",
                "latency_ms",
                "ttft_ms",
            ]
            peak_metrics = [
                "ram_used_mb_peak",
                "process_ram_mb_peak",
                "cpu_percent_peak",
                "temperature_c_peak",
            ]

            for metric in mean_metrics:
                if metric in group.columns:
                    pool_record[metric] = pd.to_numeric(
                        group[metric],
                        errors="coerce",
                    ).mean()

            for metric in peak_metrics:
                if metric in group.columns:
                    pool_record[metric] = pd.to_numeric(
                        group[metric],
                        errors="coerce",
                    ).max()

            pooling_audits.append(pool_record)

    acquisition_audit = (
        pd.concat(acquisition_audits, ignore_index=True)
        if acquisition_audits
        else pd.DataFrame()
    )
    pooling_audit = pd.DataFrame(pooling_audits)

    if not pooling_audit.empty:
        pooled["model_join_key"] = pooled["model"].map(model_join_key)

        mean_metric_columns = [
            column
            for column in ["tokens_per_sec", "latency_ms", "ttft_ms"]
            if column in pooling_audit.columns
        ]
        peak_metric_columns = [
            column
            for column in [
                "ram_used_mb_peak",
                "process_ram_mb_peak",
                "cpu_percent_peak",
                "temperature_c_peak",
            ]
            if column in pooling_audit.columns
        ]
        metric_columns = mean_metric_columns + peak_metric_columns
        merge_columns = [
            "platform",
            "dataset",
            "model_join_key",
            "sample_count",
            "run_count",
            "acquisition_date_count",
            *metric_columns,
        ]
        sidecar = pooling_audit[merge_columns].copy()
        sidecar = sidecar.rename(columns={
            "sample_count": "dolly_sample_count__pooled",
            "run_count": "dolly_run_count__pooled",
            "acquisition_date_count": (
                "dolly_acquisition_date_count__pooled"
            ),
            **{
                column: f"{column}__pooled"
                for column in metric_columns
            },
        })

        pooled = pooled.merge(
            sidecar,
            on=["platform", "dataset", "model_join_key"],
            how="left",
        )

        cpu_like_dolly = (
            pooled["platform"].astype(str).str.lower().isin(
                ["cpu", "pi5"]
            )
            & pooled["dataset"].astype(str).str.lower().eq("dolly")
        )
        matched = (
            cpu_like_dolly
            & pooled["dolly_sample_count__pooled"].notna()
        )

        for metric in metric_columns:
            pooled_column = f"{metric}__pooled"
            replace_mask = matched & pooled[pooled_column].notna()

            if metric in peak_metric_columns:
                # Peak metrics use the maximum across every split
                # acquisition and any pre-existing normalized value.
                existing_values = pd.to_numeric(
                    pooled.loc[replace_mask, metric],
                    errors="coerce",
                )
                pooled_values = pd.to_numeric(
                    pooled.loc[replace_mask, pooled_column],
                    errors="coerce",
                )
                pooled.loc[replace_mask, metric] = pd.concat(
                    [existing_values, pooled_values],
                    axis=1,
                ).max(axis=1, skipna=True)
            else:
                # Mean metrics are pooled from all underlying samples, so
                # acquisition batches are weighted by their sample counts.
                pooled.loc[replace_mask, metric] = pooled.loc[
                    replace_mask,
                    pooled_column,
                ]

            pooled = pooled.drop(columns=[pooled_column])

        pooled.loc[
            matched,
            "tokens_per_sec_backfilled_from_samples",
        ] = 1
        pooled.loc[
            matched,
            "dolly_split_sample_pool_used",
        ] = 1
        pooled.loc[
            matched
            & pooled["ram_used_mb_peak"].notna(),
            "dolly_peak_metric_pool_used",
        ] = 1
        pooled.loc[
            matched,
            "dolly_sample_count",
        ] = pooled.loc[
            matched,
            "dolly_sample_count__pooled",
        ]
        pooled.loc[
            matched,
            "dolly_run_count",
        ] = pooled.loc[
            matched,
            "dolly_run_count__pooled",
        ]
        pooled.loc[
            matched,
            "dolly_acquisition_date_count",
        ] = pooled.loc[
            matched,
            "dolly_acquisition_date_count__pooled",
        ]

        pooled = pooled.drop(columns=[
            "dolly_sample_count__pooled",
            "dolly_run_count__pooled",
            "dolly_acquisition_date_count__pooled",
        ])

    pooled = pooled.drop(columns=["model_join_key"], errors="ignore")
    return pooled, acquisition_audit, pooling_audit


(
    summary_raw,
    dolly_split_acquisition_audit,
    dolly_split_pooling_audit,
) = pool_split_dolly_acquisitions(summary_raw)


def integrate_processed_dolly_peak_metrics(
    normalized: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Merge Dolly peak metrics exported by the platform notebooks.

    ``model_dataset_summary.csv`` already contains model/dataset maxima from
    sample records and expanded system traces. These values are used as a
    robust secondary source for publication figures.
    """
    integrated = normalized.copy()
    integrated["model_join_key"] = integrated["model"].map(model_join_key)

    peak_map = {
        "ram_used_mb_peak_max": "ram_used_mb_peak",
        "process_ram_mb_peak_max": "process_ram_mb_peak",
        "cpu_percent_peak_max": "cpu_percent_peak",
        "temperature_c_peak_max": "temperature_c_peak",
    }
    audit_rows = []

    for platform in ["cpu", "pi5"]:
        path = PLATFORM_MODEL_SUMMARY_PATHS.get(platform)
        if path is None or not path.exists():
            audit_rows.append({
                "platform": platform,
                "source_file": str(path) if path is not None else None,
                "matched_models": 0,
                "ram_models_available": 0,
                "status": "model summary not found",
            })
            continue

        platform_summary = pd.read_csv(path)
        required = {"platform", "dataset", "model"}
        if not required.issubset(platform_summary.columns):
            audit_rows.append({
                "platform": platform,
                "source_file": str(path),
                "matched_models": 0,
                "ram_models_available": 0,
                "status": "required columns missing",
            })
            continue

        platform_summary["platform"] = (
            platform_summary["platform"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
        platform_summary["dataset"] = (
            platform_summary["dataset"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
        platform_summary = platform_summary[
            platform_summary["platform"].eq(platform)
            & platform_summary["dataset"].eq("dolly")
        ].copy()
        platform_summary["model_join_key"] = (
            platform_summary["model"].map(model_join_key)
        )

        available_map = {
            source: target
            for source, target in peak_map.items()
            if source in platform_summary.columns
        }
        if not available_map:
            audit_rows.append({
                "platform": platform,
                "source_file": str(path),
                "matched_models": 0,
                "ram_models_available": 0,
                "status": "peak columns missing",
            })
            continue

        for source in available_map:
            platform_summary[source] = pd.to_numeric(
                platform_summary[source],
                errors="coerce",
            )

        sidecar = (
            platform_summary.groupby(
                ["platform", "dataset", "model_join_key"],
                dropna=False,
            )[list(available_map)]
            .max()
            .reset_index()
        )
        sidecar = sidecar.rename(columns={
            source: f"{target}__processed_peak"
            for source, target in available_map.items()
        })

        integrated = integrated.merge(
            sidecar,
            on=["platform", "dataset", "model_join_key"],
            how="left",
        )

        platform_dolly = (
            integrated["platform"].astype(str).str.lower().eq(platform)
            & integrated["dataset"].astype(str).str.lower().eq("dolly")
        )
        matched_models = 0
        for target in available_map.values():
            source_column = f"{target}__processed_peak"
            if target not in integrated.columns:
                integrated[target] = np.nan

            replace_mask = (
                platform_dolly
                & integrated[source_column].notna()
            )
            matched_models = max(matched_models, int(replace_mask.sum()))

            existing_values = pd.to_numeric(
                integrated.loc[replace_mask, target],
                errors="coerce",
            )
            source_values = pd.to_numeric(
                integrated.loc[replace_mask, source_column],
                errors="coerce",
            )
            integrated.loc[replace_mask, target] = pd.concat(
                [existing_values, source_values],
                axis=1,
            ).max(axis=1, skipna=True)
            integrated = integrated.drop(columns=[source_column])

        ram_models_available = int(
            platform_summary["ram_used_mb_peak_max"].notna().sum()
        ) if "ram_used_mb_peak_max" in platform_summary.columns else 0

        audit_rows.append({
            "platform": platform,
            "source_file": str(path),
            "matched_models": matched_models,
            "ram_models_available": ram_models_available,
            "status": "loaded",
        })

    integrated = integrated.drop(
        columns=["model_join_key"],
        errors="ignore",
    )
    return integrated, pd.DataFrame(audit_rows)


(
    summary_raw,
    dolly_processed_peak_audit,
) = integrate_processed_dolly_peak_metrics(summary_raw)

dolly_processed_peak_audit.to_csv(
    CHECK_DIR / "dolly_processed_peak_audit.csv",
    index=False,
)

dolly_split_acquisition_audit.to_csv(
    CHECK_DIR / "dolly_split_acquisition_audit.csv",
    index=False,
)
dolly_split_pooling_audit.to_csv(
    CHECK_DIR / "dolly_split_pooling_audit.csv",
    index=False,
)

print("Dolly split-acquisition audit:")
display(dolly_split_pooling_audit)
print("Dolly processed peak-metric audit:")
display(dolly_processed_peak_audit)

print("Source:", SUMMARY_PATH)
print("Output:", OUTPUT_DIR)
print("Rows:", len(summary_raw))
print("Columns:", sorted(summary_raw.columns))


In [ ]:
print('LaTeX and Markdown table export is dependency-free; Jinja2 is not required.')


## 2. Canonical model metadata

Filename parsing is used only as a fallback. The explicit registry prevents version numbers such as `qwen2_5_1_5b` from being misread as 5B, and distinguishes **total** from **active** parameters for mixture-of-experts models.


In [ ]:

def canonical_token(value) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


MODEL_REGISTRY = [
    {
        "display_name": "Llama 3.2 1B (Q8)",
        "aliases": ["Llama-3.2-1B-Instruct-Q8_0.gguf"],
        "family": "Llama 3.2",
        "total_parameters_b": 1.0,
        "active_parameters_b": 1.0,
        "architecture": "dense",
        "precision": "Q8_0",
    },
    {
        "display_name": "Llama 3.2 1B (Q4)",
        "aliases": ["llama-3.2-1b-instruct-q4_k_m.gguf"],
        "family": "Llama 3.2",
        "total_parameters_b": 1.0,
        "active_parameters_b": 1.0,
        "architecture": "dense",
        "precision": "Q4_K_M",
    },
    {
        "display_name": "Llama 3.2 1B (bf16)",
        "aliases": ["llama3_2_1b_bf16"],
        "family": "Llama 3.2",
        "total_parameters_b": 1.0,
        "active_parameters_b": 1.0,
        "architecture": "dense",
        "precision": "bf16",
    },
    {
        "display_name": "Gemma 3 1B (Q8)",
        "aliases": ["gemma-3-1b-it-Q8_0.gguf"],
        "family": "Gemma 3",
        "total_parameters_b": 1.0,
        "active_parameters_b": 1.0,
        "architecture": "dense",
        "precision": "Q8_0",
    },
    {
        "display_name": "Qwen 2.5 1.5B (bf16)",
        "aliases": ["qwen2_5_1_5b_bf16"],
        "family": "Qwen 2.5",
        "total_parameters_b": 1.54,
        "active_parameters_b": 1.54,
        "architecture": "dense",
        "precision": "bf16",
    },
    {
        "display_name": "SmolLM3 3B (Q4)",
        "aliases": ["SmolLM3-3B-Q4_K_M.gguf"],
        "family": "SmolLM3",
        "total_parameters_b": 3.0,
        "active_parameters_b": 3.0,
        "architecture": "dense",
        "precision": "Q4_K_M",
    },
    {
        "display_name": "Ministral 3 3B (Q4)",
        "aliases": ["Ministral-3-3B-Instruct-2512-Q4_K_M.gguf"],
        "family": "Ministral 3",
        "total_parameters_b": 3.0,
        "active_parameters_b": 3.0,
        "architecture": "dense",
        "precision": "Q4_K_M",
    },
    {
        "display_name": "Ministral 3 3B (AWQ)",
        "aliases": ["ministral3_3b_awq"],
        "family": "Ministral 3",
        "total_parameters_b": 3.0,
        "active_parameters_b": 3.0,
        "architecture": "dense",
        "precision": "AWQ",
    },
    {
        "display_name": "Phi-4 Mini 3.8B (Q4)",
        "aliases": ["Phi-4-mini-instruct-Q4_K_M.gguf"],
        "family": "Phi-4 Mini",
        "total_parameters_b": 3.8,
        "active_parameters_b": 3.8,
        "architecture": "dense",
        "precision": "Q4_K_M",
    },
    {
        "display_name": "Phi-4 Mini 3.8B (bf16)",
        "aliases": ["phi4mini_bf16"],
        "family": "Phi-4 Mini",
        "total_parameters_b": 3.8,
        "active_parameters_b": 3.8,
        "architecture": "dense",
        "precision": "bf16",
    },
    {
        "display_name": "Phi-4 Mini 3.8B (GPTQ-4)",
        "aliases": ["phi4mini_gptq_4bit"],
        "family": "Phi-4 Mini",
        "total_parameters_b": 3.8,
        "active_parameters_b": 3.8,
        "architecture": "dense",
        "precision": "GPTQ-4",
    },
    {
        "display_name": "Qwen 3 4B (AWQ)",
        "aliases": ["qwen3_4b_awq"],
        "family": "Qwen 3",
        "total_parameters_b": 4.0,
        "active_parameters_b": 4.0,
        "architecture": "dense",
        "precision": "AWQ",
    },
    {
        "display_name": "Granite 4 H-Tiny (bf16)",
        "aliases": ["granite4_htiny_bf16"],
        "family": "Granite 4",
        "total_parameters_b": 7.0,
        "active_parameters_b": 1.0,
        "architecture": "moe",
        "precision": "bf16",
    },
    {
        "display_name": "Qwen 2.5 7B (AWQ)",
        "aliases": ["qwen2_5_7b_awq"],
        "family": "Qwen 2.5",
        "total_parameters_b": 7.62,
        "active_parameters_b": 7.62,
        "architecture": "dense",
        "precision": "AWQ",
    },
    {
        "display_name": "Ministral 3 8B (AWQ)",
        "aliases": ["ministral3_8b_awq"],
        "family": "Ministral 3",
        "total_parameters_b": 8.0,
        "active_parameters_b": 8.0,
        "architecture": "dense",
        "precision": "AWQ",
    },
    {
        "display_name": "Ministral 3 14B",
        "aliases": ["ministral3_14b"],
        "family": "Ministral 3",
        "total_parameters_b": 14.0,
        "active_parameters_b": 14.0,
        "architecture": "dense",
        "precision": "bf16",
    },
    {
        "display_name": "Phi-4 14B (bf16)",
        "aliases": ["phi4_14b_bf16"],
        "family": "Phi-4",
        "total_parameters_b": 14.0,
        "active_parameters_b": 14.0,
        "architecture": "dense",
        "precision": "bf16",
    },
    {
        "display_name": "Qwen 3 30B-A3B (bf16)",
        "aliases": ["qwen3_30b_instruct_bf16"],
        "family": "Qwen 3",
        "total_parameters_b": 30.5,
        "active_parameters_b": 3.3,
        "architecture": "moe",
        "precision": "bf16",
    },
    {
        "display_name": "Kimi Linear 48B-A3B (bf16)",
        "aliases": ["kimi_linear_48b_bf16"],
        "family": "Kimi Linear",
        "total_parameters_b": 48.0,
        "active_parameters_b": 3.0,
        "architecture": "moe",
        "precision": "bf16",
    },
    {
        "display_name": "Llama 3.3 70B (FP8)",
        "aliases": ["llama3_3_70b_fp8"],
        "family": "Llama 3.3",
        "total_parameters_b": 70.0,
        "active_parameters_b": 70.0,
        "architecture": "dense",
        "precision": "FP8",
    },
    {
        "display_name": "Apertus 70B (bf16)",
        "aliases": ["apertus_70b_instruct_bf16"],
        "family": "Apertus",
        "total_parameters_b": 70.0,
        "active_parameters_b": 70.0,
        "architecture": "dense",
        "precision": "bf16",
    },
]

ALIAS_TO_METADATA = {}
for record in MODEL_REGISTRY:
    for alias in record["aliases"]:
        ALIAS_TO_METADATA[canonical_token(alias)] = record


def fallback_parameter_count(model_name: str) -> float:
    """Fallback only; explicit registry values take precedence."""
    text = str(model_name).lower().replace("_", "-")
    moe = re.search(r"(?:^|[^0-9])(\d+)\s*[x×]\s*(\d+(?:\.\d+)?)\s*b(?:[^a-z]|$)", text)
    if moe:
        return float(moe.group(1)) * float(moe.group(2))
    matches = re.findall(r"(?:^|[^0-9])(\d+(?:\.\d+)?)\s*b(?:[^a-z]|$)", text)
    return max(map(float, matches)) if matches else np.nan


def resolve_metadata(row: pd.Series) -> pd.Series:
    candidate_fields = [
        "model_id_norm", "model", "model_label", "model_display", "repo_id"
    ]
    record = None
    matched_value = None
    for field in candidate_fields:
        if field in row.index and pd.notna(row[field]):
            key = canonical_token(row[field])
            if key in ALIAS_TO_METADATA:
                record = ALIAS_TO_METADATA[key]
                matched_value = row[field]
                break

    if record is None:
        inferred = fallback_parameter_count(row.get("model", ""))
        return pd.Series({
            "model_display_canonical": str(row.get("model_display", row.get("model", "unknown"))),
            "model_family_canonical": str(row.get("model_family", "unknown")),
            "total_parameters_b": inferred,
            "active_parameters_b": inferred,
            "architecture": "unknown",
            "precision": "unknown",
            "metadata_source": "filename fallback",
            "metadata_match": np.nan,
        })

    return pd.Series({
        "model_display_canonical": record["display_name"],
        "model_family_canonical": record["family"],
        "total_parameters_b": record["total_parameters_b"],
        "active_parameters_b": record["active_parameters_b"],
        "architecture": record["architecture"],
        "precision": record["precision"],
        "metadata_source": "explicit registry",
        "metadata_match": matched_value,
    })


def normalize_platform(value) -> str:
    token = str(value).strip().lower().replace("-", "_").replace(" ", "_")
    return PLATFORM_ALIASES.get(token, token)


def normalize_dataset(value) -> str:
    token = re.sub(r"[^a-z0-9]+", "", str(value).lower())
    aliases = {
        "mmlu": "mmlu",
        "mmlutest": "mmlu",
        "gsm8k": "gsm8k",
        "gsm8ktest": "gsm8k",
        "hellaswag": "hellaswag",
        "hellaswagtest": "hellaswag",
        "dolly": "dolly",
        "databricksdolly15k": "dolly",
    }
    return aliases.get(token, token)


summary = summary_raw.copy()
required = {"dataset", "platform", "model"}
missing = sorted(required - set(summary.columns))
if missing:
    raise KeyError(
        f"normalized_summary_all.csv is missing required columns {missing}. "
        f"Available columns: {sorted(summary.columns)}"
    )

summary["dataset"] = summary["dataset"].map(normalize_dataset)
summary["platform"] = summary["platform"].map(normalize_platform)
summary["context"] = summary["platform"].map(PLATFORM_TO_CONTEXT)
summary["context_display"] = summary["context"].map(CONTEXT_DISPLAY)
summary["model"] = summary["model"].astype(str)

metadata = summary.apply(resolve_metadata, axis=1)
summary = pd.concat([summary, metadata], axis=1)
summary["compact_total"] = summary["total_parameters_b"].le(COMPACT_TOTAL_PARAMETER_THRESHOLD_B)

metadata_audit = (
    summary[
        [
            "model", "model_display_canonical", "model_family_canonical",
            "total_parameters_b", "active_parameters_b", "architecture",
            "precision", "metadata_source", "metadata_match",
        ]
    ]
    .drop_duplicates()
    .sort_values(["total_parameters_b", "model"], na_position="last")
)

metadata_audit.to_csv(CHECK_DIR / "model_metadata_audit.csv", index=False)
display(metadata_audit)

unknown_metadata = metadata_audit[
    metadata_audit["metadata_source"].ne("explicit registry")
]
if not unknown_metadata.empty:
    print("WARNING: models using fallback metadata:")
    display(unknown_metadata)


## 3. Normalize metrics and construct context-aware summaries

For GPU, deployment memory means peak VRAM. For edge and local CPU, it means peak system RAM. These values are never pooled into a single unqualified “RAM” axis.


In [ ]:

NUMERIC_CANDIDATES = [
    "accuracy", "accuracy_sd",
    "latency_ms", "latency_ms_sd",
    "ttft_ms", "ttft_ms_sd",
    "tokens_per_sec", "tokens_per_sec_sd",
    "sustained_tokens_per_sec", "sustained_tokens_per_sec_sd",
    "samples_per_sec", "samples_per_sec_sd",
    "ram_used_mb_peak", "process_ram_mb_peak",
    "vram_used_mb_peak",
    "cpu_percent_avg", "cpu_percent_peak",
    "gpu_util_percent_avg", "gpu_util_percent_peak",
    "temperature_c_avg", "temperature_c_peak",
    "power_w_avg", "power_w_peak",
    "energy_j", "energy_per_token_j",
    "tokens_per_sec_backfilled_from_samples",
    "dolly_split_sample_pool_used",
    "dolly_peak_metric_pool_used",
    "dolly_sample_count",
    "dolly_run_count",
    "dolly_acquisition_date_count",
]
NUMERIC_METRICS = [c for c in NUMERIC_CANDIDATES if c in summary.columns]
for column in NUMERIC_METRICS:
    summary[column] = pd.to_numeric(summary[column], errors="coerce")

if "energy_j" not in summary.columns and {"power_w_avg", "latency_ms"}.issubset(summary.columns):
    summary["energy_j"] = summary["power_w_avg"] * summary["latency_ms"] / 1000.0
    NUMERIC_METRICS.append("energy_j")

IDENTITY = [
    "dataset", "platform", "context", "context_display", "model",
    "model_display_canonical", "model_family_canonical",
    "total_parameters_b", "active_parameters_b", "architecture", "precision",
    "compact_total",
]

PEAK_NUMERIC_METRICS = {
    "ram_used_mb_peak",
    "process_ram_mb_peak",
    "vram_used_mb_peak",
    "cpu_percent_peak",
    "gpu_util_percent_peak",
    "temperature_c_peak",
    "power_w_peak",
    "tokens_per_sec_backfilled_from_samples",
    "dolly_split_sample_pool_used",
    "dolly_peak_metric_pool_used",
    "dolly_sample_count",
    "dolly_run_count",
    "dolly_acquisition_date_count",
}

metric_aggregation = {
    metric: (
        "max"
        if metric in PEAK_NUMERIC_METRICS
        else "mean"
    )
    for metric in NUMERIC_METRICS
}

agg = (
    summary.groupby(IDENTITY, dropna=False)
    .agg(metric_aggregation)
    .reset_index()
)

agg["latency_s"] = agg["latency_ms"] / 1000.0 if "latency_ms" in agg else np.nan
agg["ttft_s"] = agg["ttft_ms"] / 1000.0 if "ttft_ms" in agg else np.nan

agg["deployment_memory_mb"] = np.where(
    agg["context"].eq("gpu_server"),
    agg.get("vram_used_mb_peak", np.nan),
    agg.get("ram_used_mb_peak", np.nan),
)
agg["deployment_memory_gib"] = agg["deployment_memory_mb"] / 1024.0
agg["deployment_memory_type"] = np.where(
    agg["context"].eq("gpu_server"), "VRAM", "system RAM"
)

# Short-output cost-effectiveness indicator.
#
# Definition:
# expected correct responses/s = accuracy × measured samples/s
#
# Do not substitute token throughput or accuracy/latency for this metric.
# When samples_per_sec is unavailable, the indicator remains missing so that
# the manuscript cannot accidentally report a differently defined quantity.
if "samples_per_sec" in agg.columns:
    agg["expected_correct_per_second"] = (
        agg["accuracy"] * agg["samples_per_sec"]
    )
else:
    agg["samples_per_sec"] = np.nan
    agg["expected_correct_per_second"] = np.nan

# A latency-derived diagnostic is retained for internal audit only and is not
# presented as "accuracy per second" in the manuscript.
agg["accuracy_over_latency_s"] = agg["accuracy"] / agg["latency_s"]


# Context-specific throughput reporting.
#
# Edge and Local CPU:
#   use active-generation tokens/s (`tokens_per_sec`), because llama.cpp
#   measurements describe decoding while the request is actively generating.
#   On the Raspberry Pi, the deliberate inter-request cooling pause is not
#   included in this value.
#
# GPU server:
#   use run-level sustained tokens/s (`sustained_tokens_per_sec`), because
#   the vLLM experiments report aggregate serving throughput over the
#   measured run.
#
# These values share the same unit but not the same operational denominator.
# They are therefore compared within context and displayed in separate
# panels, with the basis stated explicitly.
agg["reported_throughput_tokens_per_sec"] = np.nan
agg["throughput_measurement_basis"] = pd.Series(
    pd.NA,
    index=agg.index,
    dtype="object",
)
agg["throughput_source_column"] = pd.Series(
    pd.NA,
    index=agg.index,
    dtype="object",
)

cpu_like_mask = agg["context"].isin(["edge", "local_cpu"])
gpu_mask = agg["context"].eq("gpu_server")

agg.loc[
    cpu_like_mask,
    "reported_throughput_tokens_per_sec",
] = agg.loc[cpu_like_mask, "tokens_per_sec"]
agg.loc[
    cpu_like_mask,
    "throughput_measurement_basis",
] = "Active generation"
agg.loc[
    cpu_like_mask,
    "throughput_source_column",
] = np.where(
    agg.loc[
        cpu_like_mask,
        "dolly_split_sample_pool_used",
    ].fillna(0).gt(0),
    "tokens_per_sec (sample-level pooled split acquisitions)",
    "tokens_per_sec (normalized summary)",
)

agg.loc[
    gpu_mask,
    "reported_throughput_tokens_per_sec",
] = agg.loc[gpu_mask, "sustained_tokens_per_sec"]
agg.loc[
    gpu_mask,
    "throughput_measurement_basis",
] = "Run-level sustained"
agg.loc[
    gpu_mask,
    "throughput_source_column",
] = "sustained_tokens_per_sec"

# No silent fallback is applied on GPU. Missing sustained throughput remains
# missing and is surfaced in the audit table.

print("Datasets:", sorted(agg["dataset"].dropna().unique()))
print("Contexts:", sorted(agg["context"].dropna().unique()))
print("Rows after aggregation:", len(agg))
display(agg.head())


## 4. Export and plotting helpers

In [ ]:

def latex_escape(value: str) -> str:
    """Escape ordinary text for LaTeX while preserving selected math symbols."""
    text = str(value)
    placeholders = {
        "×": "@@TIMES@@", "≤": "@@LEQ@@", "≥": "@@GEQ@@",
        "∼": "@@SIM@@", "≈": "@@APPROX@@", "°": "@@DEGREE@@",
        "→": "@@ARROW@@",
    }
    for symbol, placeholder in placeholders.items():
        text = text.replace(symbol, placeholder)

    text = (
        text.replace("–", "--")
        .replace("—", "---")
        .replace("−", "-")
        .replace("’", "'")
        .replace("“", "``")
        .replace("”", "''")
    )

    text = text.replace("\\", "@@BACKSLASH@@")
    for old, new in [
        ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
        ("#", r"\#"), ("_", r"\_"), ("{", r"\{"),
        ("}", r"\}"), ("~", r"\textasciitilde{}"),
        ("^", r"\textasciicircum{}"),
    ]:
        text = text.replace(old, new)

    text = text.replace("@@BACKSLASH@@", r"\textbackslash{}")
    latex_symbols = {
        "@@TIMES@@": r"$\times$", "@@LEQ@@": r"$\leq$",
        "@@GEQ@@": r"$\geq$", "@@SIM@@": r"$\sim$",
        "@@APPROX@@": r"$\approx$", "@@DEGREE@@": r"$^\circ$",
        "@@ARROW@@": r"$\rightarrow$",
    }
    for placeholder, latex in latex_symbols.items():
        text = text.replace(placeholder, latex)
    return text


def normalize_float_format(float_format: str) -> str:
    text = str(float_format)
    return text[1:] if text.startswith("%") else text


def format_table_value(value, float_format: str = ".3f", output: str = "latex") -> str:
    if pd.isna(value):
        rendered = "--"
    elif isinstance(value, (bool, np.bool_)):
        rendered = "Yes" if bool(value) else "No"
    elif isinstance(value, (float, np.floating)):
        rendered = format(float(value), normalize_float_format(float_format))
    elif isinstance(value, (int, np.integer)):
        rendered = str(int(value))
    else:
        rendered = str(value)
    return latex_escape(rendered) if output == "latex" else rendered


def markdown_table(df: pd.DataFrame, float_format: str = ".3f") -> str:
    headers = [str(column) for column in df.columns]
    rows = [
        [format_table_value(value, float_format=float_format, output="markdown") for value in row]
        for row in df.itertuples(index=False, name=None)
    ]
    widths = []
    for index, header in enumerate(headers):
        row_widths = [len(row[index]) for row in rows]
        widths.append(max([len(header)] + row_widths))
    header_line = "| " + " | ".join(header.ljust(widths[i]) for i, header in enumerate(headers)) + " |"
    separator = "|-" + "-|-".join("-" * width for width in widths) + "-|"
    body = [
        "| " + " | ".join(value.ljust(widths[i]) for i, value in enumerate(row)) + " |"
        for row in rows
    ]
    return "\n".join([header_line, separator] + body)


def table_is_wide(df: pd.DataFrame) -> bool:
    """Select table* when the structure will not fit one IEEE column."""
    text_columns = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    longest_text = max(
        [len(str(c)) for c in df.columns]
        + [
            int(df[c].astype(str).str.len().max())
            for c in text_columns
            if not df.empty
        ]
    )
    return len(df.columns) >= 5 or len(text_columns) >= 3 or longest_text > 28


def dataframe_to_latex_string(
    df: pd.DataFrame,
    caption: str,
    label: str,
    float_format: str = ".3f",
    position: str = "t",
    wide: bool | None = None,
) -> str:
    """Render an IEEE-friendly tabularx table without optional dependencies."""
    if len(df.columns) == 0:
        raise ValueError("Cannot export a table with zero columns.")

    wide = table_is_wide(df) if wide is None else bool(wide)
    environment = "table*" if wide else "table"
    width = r"\textwidth" if wide else r"\columnwidth"
    font_size = r"\scriptsize" if wide else r"\footnotesize"

    column_spec = []
    for index, column in enumerate(df.columns):
        if pd.api.types.is_numeric_dtype(df[column]):
            column_spec.append("r")
        elif index == 0 and df[column].astype(str).str.len().max() <= 14:
            column_spec.append("l")
        else:
            column_spec.append("L")
    column_spec = "".join(column_spec)

    lines = [
        rf"\begin{{{environment}}}[{position}]",
        r"\centering",
        rf"\caption{{{latex_escape(caption)}}}",
        rf"\label{{{latex_escape(label)}}}",
        font_size,
        r"\setlength{\tabcolsep}{3.2pt}",
        r"\renewcommand{\arraystretch}{1.08}",
        rf"\begin{{tabularx}}{{{width}}}{{@{{}}{column_spec}@{{}}}}",
        r"\toprule",
        " & ".join(rf"\textbf{{{latex_escape(column)}}}" for column in df.columns) + r" \\",
        r"\midrule",
    ]

    for row in df.itertuples(index=False, name=None):
        lines.append(
            " & ".join(
                format_table_value(value, float_format=float_format, output="latex")
                for value in row
            ) + r" \\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabularx}",
        rf"\end{{{environment}}}",
        "",
    ])
    return "\n".join(lines)


def export_table(df, stem, caption, label, float_format="%.3f", wide=None):
    csv_path = TABLE_DIR / f"{stem}.csv"
    md_path = TABLE_DIR / f"{stem}.md"
    tex_path = TABLE_DIR / f"{stem}.tex"
    df.to_csv(csv_path, index=False)
    md_path.write_text(
        f"### {caption}\n\n{markdown_table(df, float_format=float_format)}\n",
        encoding="utf-8",
    )
    tex_path.write_text(
        dataframe_to_latex_string(
            df=df,
            caption=caption,
            label=label,
            float_format=float_format,
            position="t",
            wide=wide,
        ),
        encoding="utf-8",
    )
    return {"csv": csv_path, "md": md_path, "tex": tex_path}


def save_figure(fig, stem: str):
    """Export editable SVG, vector PDF, and high-resolution PNG."""
    png = FIGURE_DIR / f"{stem}.png"
    pdf = FIGURE_DIR / f"{stem}.pdf"
    svg = FIGURE_DIR / f"{stem}.svg"
    fig.savefig(png, dpi=SAVE_DPI, facecolor="white")
    fig.savefig(pdf, facecolor="white")
    fig.savefig(svg, format="svg", facecolor="white")
    print("saved:", png)
    print("saved:", pdf)
    print("saved:", svg)
    plt.show()
    return png, pdf, svg


def pareto_mask(df, x, y, maximize_x, maximize_y=True):
    values = df[[x, y]].to_numpy(dtype=float)
    efficient = np.ones(len(df), dtype=bool)
    for i, point in enumerate(values):
        if np.isnan(point).any():
            efficient[i] = False
            continue
        for j, other in enumerate(values):
            if i == j or np.isnan(other).any():
                continue
            x_no_worse = other[0] >= point[0] if maximize_x else other[0] <= point[0]
            y_no_worse = other[1] >= point[1] if maximize_y else other[1] <= point[1]
            x_strict = other[0] > point[0] if maximize_x else other[0] < point[0]
            y_strict = other[1] > point[1] if maximize_y else other[1] < point[1]
            if x_no_worse and y_no_worse and (x_strict or y_strict):
                efficient[i] = False
                break
    return pd.Series(efficient, index=df.index)


def short_name(value: str, max_chars: int = 26) -> str:
    text = str(value)
    replacements = {
        " (bf16)": "", " (Q4_K_M)": "", " (Q8_0)": "",
        " (GPTQ-4)": "", " (AWQ)": "", "-Instruct": "",
        "-instruct": "", "_instruct": "", ".gguf": "",
        "Qwen 3 30B-A3B": "Qwen3 30B-A3B",
        "Kimi Linear 48B-A3B": "Kimi Linear 48B",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text if len(text) <= max_chars else text[: max_chars - 1] + "…"


def style_axis(ax, grid_axis="both"):
    ax.grid(axis=grid_axis, which="major", alpha=0.20, linewidth=0.5)
    ax.tick_params(length=2.5, width=0.7)
    ax.spines["left"].set_linewidth(0.75)
    ax.spines["bottom"].set_linewidth(0.75)


def add_panel_label(ax, letter: str, title: str):
    ax.text(
        0.0, 1.03, f"({letter}) {title}",
        transform=ax.transAxes,
        ha="left", va="bottom", fontsize=8.2,
        fontweight="semibold",
    )


def shared_legend(fig, handles, labels, ncol=4, y=-0.08):
    fig.legend(
        handles, labels,
        loc="lower center", bbox_to_anchor=(0.5, y),
        ncol=ncol, frameon=False,
        columnspacing=1.0, handletextpad=0.45,
    )


def context_subset(context: str, dataset: str | None = None) -> pd.DataFrame:
    frame = agg[agg["context"].eq(context)].copy()
    if dataset is not None:
        frame = frame[frame["dataset"].eq(dataset)].copy()
    return frame





def order_models_by_gsm8k(
    context_frame: pd.DataFrame,
) -> list[str]:
    """Return a stable model order based on GSM8K accuracy.

    Higher GSM8K accuracy appears first. Configurations without a GSM8K
    measurement are placed last and ordered alphabetically. The resulting
    order is reused by categorical Figures 2–5.
    """
    available_models = (
        context_frame[
            ["model_display_canonical"]
        ]
        .drop_duplicates()
    )

    gsm8k_rows = (
        context_frame[
            context_frame["dataset"].eq("gsm8k")
        ][
            [
                "model_display_canonical",
                "accuracy",
            ]
        ]
        .drop_duplicates(
            subset=["model_display_canonical"]
        )
        .rename(columns={"accuracy": "gsm8k_accuracy"})
    )

    ordered = available_models.merge(
        gsm8k_rows,
        on="model_display_canonical",
        how="left",
    )
    ordered["gsm8k_missing"] = (
        ordered["gsm8k_accuracy"].isna()
    )

    ordered = ordered.sort_values(
        [
            "gsm8k_missing",
            "gsm8k_accuracy",
            "model_display_canonical",
        ],
        ascending=[True, False, True],
        na_position="last",
    )

    return ordered[
        "model_display_canonical"
    ].tolist()


SHARED_CONTEXT_MODEL_ORDER = {
    context: order_models_by_gsm8k(
        agg[agg["context"].eq(context)]
    )
    for context in CONTEXT_ORDER
}

shared_model_order_audit = pd.DataFrame(
    [
        {
            "context": context,
            "context_display": CONTEXT_DISPLAY[context],
            "model_order_position": position + 1,
            "model_display_canonical": model,
        }
        for context in CONTEXT_ORDER
        for position, model in enumerate(
            SHARED_CONTEXT_MODEL_ORDER[context]
        )
    ]
)
shared_model_order_audit.to_csv(
    CHECK_DIR / "shared_model_order_gsm8k.csv",
    index=False,
)



def build_shared_context_model_positions(
    context_gap: float = 1.15,
) -> tuple[
    pd.DataFrame,
    dict[tuple[str, str], float],
    list[float],
    dict[str, float],
]:
    """Build the categorical x-axis shared by Figures 4 and 5.

    Contexts follow Edge, Local CPU, and GPU server order. Models within
    each context follow SHARED_CONTEXT_MODEL_ORDER, which is based on
    descending GSM8K accuracy.
    """
    records = []
    position = 0.0
    boundaries = []
    midpoints = {}

    for context_index, context in enumerate(CONTEXT_ORDER):
        context_start = position

        for model_order_index, model in enumerate(
            SHARED_CONTEXT_MODEL_ORDER[context]
        ):
            records.append({
                "context_order_position": context_index + 1,
                "context": context,
                "context_display": CONTEXT_DISPLAY[context],
                "model_order_position": model_order_index + 1,
                "model_display_canonical": model,
                "x_position": position,
            })
            position += 1.0

        context_end = position - 1.0
        if context_end >= context_start:
            midpoints[context] = (
                context_start + context_end
            ) / 2.0

        if context_index < len(CONTEXT_ORDER) - 1:
            boundaries.append(
                position - 0.5 + context_gap / 2.0
            )
            position += context_gap

    positions = pd.DataFrame(records)
    positions["model_key"] = list(
        zip(
            positions["context"],
            positions["model_display_canonical"],
        )
    )
    position_map = dict(
        zip(
            positions["model_key"],
            positions["x_position"],
        )
    )

    return (
        positions,
        position_map,
        boundaries,
        midpoints,
    )




# Canonical categorical layout consumed by both Figures 4 and 5.
# Building this table once prevents accidental order or spacing drift.
(
    SHARED_FIGURE_4_5_MODEL_POSITIONS,
    SHARED_FIGURE_4_5_POSITION_MAP,
    SHARED_FIGURE_4_5_CONTEXT_BOUNDARIES,
    SHARED_FIGURE_4_5_CONTEXT_MIDPOINTS,
) = build_shared_context_model_positions()

SHARED_FIGURE_4_5_MODEL_POSITIONS[
    [
        "context_order_position",
        "context",
        "context_display",
        "model_order_position",
        "model_display_canonical",
        "x_position",
    ]
].to_csv(
    CHECK_DIR / "figure_04_05_shared_model_positions.csv",
    index=False,
)

def publication_model_label(
    value: str,
    width: int = 20,
    max_lines: int = 2,
) -> str:
    """Wrap a complete model-configuration label for publication figures."""
    text = str(value).strip()
    text = text.replace("_", " ")

    # Keep precision/quantization visible, but move it to a second line.
    if " (" in text:
        model_part, suffix = text.rsplit(" (", 1)
        suffix = "(" + suffix
        model_lines = textwrap.wrap(
            model_part,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )
        lines = model_lines[: max(1, max_lines - 1)] + [suffix]
    else:
        lines = textwrap.wrap(
            text,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )[:max_lines]

    return "\n".join(lines)



def spread_label_positions(
    values,
    lower: float,
    upper: float,
    minimum_separation: float,
) -> np.ndarray:
    """Spread ordered label positions while preserving their vertical order."""
    positions = np.sort(
        np.asarray(values, dtype=float)
    )
    count = len(positions)

    if count == 0:
        return positions
    if count == 1:
        return np.array([
            float(np.clip(positions[0], lower, upper))
        ])

    available = max(upper - lower, 1e-9)
    separation = min(
        minimum_separation,
        available / (count - 1),
    )

    positions = np.clip(positions, lower, upper)

    # Forward pass: enforce a minimum distance.
    for index in range(1, count):
        positions[index] = max(
            positions[index],
            positions[index - 1] + separation,
        )

    # Move the group down if the top label exceeded the available range.
    if positions[-1] > upper:
        positions -= positions[-1] - upper

    # Backward pass: restore separation after the downward shift.
    for index in range(count - 2, -1, -1):
        positions[index] = min(
            positions[index],
            positions[index + 1] - separation,
        )

    # Move the group up if the bottom label exceeded the available range.
    if positions[0] < lower:
        positions += lower - positions[0]

    return positions


def annotate_frontier(
    ax,
    frontier: pd.DataFrame,
    x: str,
    y: str,
    label_side: str = "auto",
):
    """Label every context-local Pareto point.

    Edge and Local CPU use compact alternating callouts. The GPU panel uses
    a dedicated label column to the right of all observations, preventing
    its denser frontier labels from covering the plotted points.
    """
    if frontier.empty:
        return

    ordered = frontier.sort_values(x).copy()

    if label_side == "right":
        # The axis margins must be set before this function is called.
        original_left, original_right = ax.get_xlim()
        x_span = max(original_right - original_left, 1e-9)

        # Reserve a right-side annotation column inside the axes. Keeping the
        # labels inside the enlarged x-limits avoids clipping in SVG/PDF.
        label_x = original_right + 0.055 * x_span
        expanded_right = original_right + 0.72 * x_span
        ax.set_xlim(original_left, expanded_right)

        y_lower, y_upper = ax.get_ylim()
        y_span = max(y_upper - y_lower, 1e-9)
        label_lower = y_lower + 0.075 * y_span
        label_upper = y_upper - 0.075 * y_span

        # Sort by accuracy so leader lines do not cross one another.
        label_rows = ordered.sort_values(y).copy()
        label_y_positions = spread_label_positions(
            label_rows[y].to_numpy(dtype=float),
            lower=label_lower,
            upper=label_upper,
            minimum_separation=0.085 * y_span,
        )

        for (_, row), label_y in zip(
            label_rows.iterrows(),
            label_y_positions,
        ):
            ax.annotate(
                publication_model_label(
                    row["model_display_canonical"],
                    width=21,
                    max_lines=2,
                ),
                xy=(row[x], row[y]),
                xytext=(label_x, label_y),
                xycoords="data",
                textcoords="data",
                ha="left",
                va="center",
                fontsize=6.05,
                linespacing=0.92,
                bbox={
                    "boxstyle": "round,pad=0.16",
                    "facecolor": "white",
                    "edgecolor": "0.72",
                    "linewidth": 0.50,
                    "alpha": 0.98,
                },
                arrowprops={
                    "arrowstyle": "-",
                    "linewidth": 0.50,
                    "color": "0.38",
                    "shrinkA": 1.5,
                    "shrinkB": 1.5,
                    "connectionstyle": "arc3,rad=0",
                },
                annotation_clip=False,
                zorder=7,
            )
        return

    # Compact alternating callouts for the less crowded Edge/CPU panels.
    offset_cycle = [
        (6, 10),
        (6, -16),
        (-6, 10),
        (-6, -16),
        (0, 15),
        (0, -21),
    ]

    for position, (_, row) in enumerate(ordered.iterrows()):
        dx, dy = offset_cycle[
            position % len(offset_cycle)
        ]

        if position == 0:
            dx = 6
        elif position == len(ordered) - 1:
            dx = -6

        horizontal_alignment = (
            "left"
            if dx > 0
            else "right"
            if dx < 0
            else "center"
        )
        vertical_alignment = (
            "bottom"
            if dy > 0
            else "top"
        )

        ax.annotate(
            publication_model_label(
                row["model_display_canonical"],
                width=18,
                max_lines=2,
            ),
            xy=(row[x], row[y]),
            xytext=(dx, dy),
            textcoords="offset points",
            ha=horizontal_alignment,
            va=vertical_alignment,
            fontsize=6.15,
            linespacing=0.92,
            bbox={
                "boxstyle": "round,pad=0.14",
                "facecolor": "white",
                "edgecolor": "0.78",
                "linewidth": 0.45,
                "alpha": 0.97,
            },
            arrowprops={
                "arrowstyle": "-",
                "linewidth": 0.45,
                "color": "0.42",
                "shrinkA": 1.5,
                "shrinkB": 1.5,
            },
            annotation_clip=False,
            zorder=7,
        )


def heatmap_text_color(cmap, norm, value):
    rgba = cmap(norm(value))
    luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
    return "black" if luminance > 0.57 else "white"


def plot_context_metric_heatmaps(
    frame: pd.DataFrame,
    metric: str,
    datasets: list[str],
    stem: str,
    cmap_name: str,
    norm,
    value_formatter,
    colorbar_label: str,
    context_title_map: dict[str, str] | None = None,
    dataset_label_map: dict[str, str] | None = None,
    figure_note: str | None = None,
):
    """Stack one compact heatmap per deployment context."""
    pivots = []
    ratios = []
    for context in CONTEXT_ORDER:
        sub = frame[frame["context"].eq(context)].copy()
        pivot = sub.pivot_table(
            index="model_display_canonical",
            columns="dataset",
            values=metric,
            aggfunc="mean",
        ).reindex(columns=datasets)
        pivot = pivot.loc[pivot.mean(axis=1, skipna=True).sort_values(ascending=False).index]
        pivots.append((context, pivot))
        ratios.append(max(len(pivot), 1))

    total_rows = sum(ratios)
    figure_height = max(4.8, 0.235 * total_rows + 1.35)
    fig, axes = plt.subplots(
        3, 1,
        figsize=(DOUBLE_COLUMN_WIDTH_IN, figure_height),
        gridspec_kw={"height_ratios": ratios},
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("0.94")
    image = None

    for panel_index, (ax, (context, pivot)) in enumerate(zip(axes, pivots)):
        values = pivot.to_numpy(dtype=float)
        masked = np.ma.masked_invalid(values)
        image = ax.imshow(masked, aspect="auto", cmap=cmap, norm=norm, interpolation="none")

        ax.set_yticks(np.arange(len(pivot)))
        ax.set_yticklabels([short_name(model, 32) for model in pivot.index], fontsize=6.8)
        ax.set_xticks(np.arange(len(datasets)))
        if panel_index == len(axes) - 1:
            displayed_datasets = [
                (
                    dataset_label_map.get(
                        dataset,
                        DATASET_DISPLAY[dataset],
                    )
                    if dataset_label_map
                    else DATASET_DISPLAY[dataset]
                )
                for dataset in datasets
            ]
            ax.set_xticklabels(displayed_datasets, fontsize=7.4)
            ax.set_xlabel("Workload")
        else:
            ax.set_xticklabels([])

        for row in range(values.shape[0]):
            for col in range(values.shape[1]):
                value = values[row, col]
                if not np.isfinite(value):
                    ax.text(col, row, "--", ha="center", va="center", fontsize=6.0, color="0.45")
                    continue
                ax.text(
                    col, row, value_formatter(value),
                    ha="center", va="center", fontsize=6.0,
                    color=heatmap_text_color(cmap, norm, value),
                )

        panel_title = (
            context_title_map.get(context, CONTEXT_DISPLAY[context])
            if context_title_map
            else CONTEXT_DISPLAY[context]
        )
        ax.set_title(
            panel_title,
            loc="left",
            pad=2.5,
            fontsize=8.3,
            fontweight="semibold",
        )
        ax.set_xticks(np.arange(-0.5, len(datasets), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(pivot), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=0.8)
        ax.tick_params(which="minor", bottom=False, left=False)
        ax.tick_params(which="major", length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)

    colorbar = fig.colorbar(
        image, ax=axes.tolist(), orientation="horizontal",
        fraction=0.025, pad=0.025, aspect=45,
    )
    colorbar.set_label(colorbar_label, fontsize=7.5)
    colorbar.ax.tick_params(labelsize=6.8, length=2)
    if figure_note:
        fig.text(
            0.01,
            -0.015,
            figure_note,
            ha="left",
            va="top",
            fontsize=6.2,
            color="0.35",
        )
    save_figure(fig, stem)


## 5. Static manuscript tables: deployment contexts and hardware

In [ ]:

table_01 = pd.DataFrame([
    {
        "Context": "Edge",
        "Hardware envelope": "Raspberry Pi 5; 4 cores; 8 GB RAM; no GPU",
        "Backend": "llama.cpp",
        "Model scale": "up to ≈4B, quantized",
        "Operational objective": "feasibility first, then accuracy",
        "Workload profile": "short interactive requests",
        "Binding constraint": "RAM, then thermal/latency",
    },
    {
        "Context": "Local CPU",
        "Hardware envelope": "Laptop CPU; 12 threads; 30 GB RAM; no discrete GPU",
        "Backend": "llama.cpp (Podman)",
        "Model scale": "≈1B–4B, quantized in current pool",
        "Operational objective": "responsiveness under memory budget",
        "Workload profile": "short-to-mid generation",
        "Binding constraint": "latency",
    },
    {
        "Context": "GPU server",
        "Hardware envelope": "2× RTX PRO 6000 Blackwell; ≈96 GiB VRAM each",
        "Backend": "vLLM",
        "Model scale": "up to ≈70B; bf16/fp8/AWQ/GPTQ",
        "Operational objective": "accuracy and throughput",
        "Workload profile": "mixed, including long generation",
        "Binding constraint": "accuracy–latency/throughput trade-off",
    },
])

export_table(
    table_01,
    "table_01_deployment_contexts",
    "Deployment contexts used to organize the evaluation.",
    "tab:deployment-contexts",
)

table_03 = pd.DataFrame([
    {
        "Platform": "Raspberry Pi 5",
        "CPU": "Cortex-A76, 4 cores",
        "RAM": "7.9 GiB",
        "GPU": "none",
        "VRAM": "—",
        "OS": "Debian 13 (trixie)",
        "Backend": "llama.cpp",
    },
    {
        "Platform": "Laptop CPU",
        "CPU": "Intel i5-1345U, 10c/12t",
        "RAM": "30 GiB",
        "GPU": "integrated Intel",
        "VRAM": "—",
        "OS": "OpenSUSE Tumbleweed",
        "Backend": "llama.cpp (Podman)",
    },
    {
        "Platform": "GPU server",
        "CPU": "2× Xeon 6520P, 96 threads",
        "RAM": "251 GiB",
        "GPU": "2× RTX PRO 6000 Blackwell",
        "VRAM": "97,887 MiB per GPU",
        "OS": "Ubuntu 24.04.3 LTS",
        "Backend": "vLLM 0.20.0",
    },
])

export_table(
    table_03,
    "table_03_hardware_platforms",
    "Hardware platforms used for evaluation.",
    "tab:hardware-platforms",
)

display(table_01)
display(table_03)


## 6. Table 2: evaluated model configurations and metadata

This table is generated from the models actually present in `normalized_summary_all.csv`; the notebook does not hard-code “nineteen” configurations.


In [ ]:

# Table 2 uses a precise counting unit:
# one configuration = one distinct model artifact and precision/quantization
# variant, regardless of whether that artifact appears in more than one
# deployment context.

context_rank = {
    CONTEXT_DISPLAY["edge"]: 0,
    CONTEXT_DISPLAY["local_cpu"]: 1,
    CONTEXT_DISPLAY["gpu_server"]: 2,
}

def join_contexts(values):
    unique_values = sorted(
        set(values),
        key=lambda value: context_rank.get(value, 99),
    )
    return "; ".join(unique_values)


table_02 = (
    agg[
        [
            "context_display",
            "model_display_canonical",
            "model_family_canonical",
            "total_parameters_b",
            "active_parameters_b",
            "architecture",
            "precision",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "model_display_canonical",
            "model_family_canonical",
            "total_parameters_b",
            "active_parameters_b",
            "architecture",
            "precision",
        ],
        dropna=False,
        as_index=False,
    )
    .agg({"context_display": join_contexts})
    .rename(columns={
        "context_display": "Deployment context(s)",
        "model_display_canonical": "Model configuration",
        "model_family_canonical": "Family",
        "total_parameters_b": "Total parameters (B)",
        "active_parameters_b": "Active parameters (B)",
        "architecture": "Architecture",
        "precision": "Precision / quantization",
    })
    .sort_values(
        ["Total parameters (B)", "Model configuration"],
        na_position="last",
    )
    .reset_index(drop=True)
)

export_table(
    table_02,
    "table_02_evaluated_model_configurations",
    (
        "Evaluated model and precision/quantization configurations. "
        "A configuration is counted once even when evaluated in multiple "
        "deployment contexts."
    ),
    "tab:evaluated-models",
)

# Audited counting units used by the manuscript-support documents.
model_configuration_count = len(table_02)
context_deployment_count = (
    agg[["context", "model_display_canonical"]]
    .drop_duplicates()
    .shape[0]
)
model_family_count = table_02["Family"].nunique()
gpu_configuration_count = (
    agg.loc[
        agg["context"].eq("gpu_server"),
        "model_display_canonical",
    ]
    .drop_duplicates()
    .shape[0]
)

parameter_values = pd.to_numeric(
    table_02["Total parameters (B)"],
    errors="coerce",
).dropna()
parameter_min_b = float(parameter_values.min()) if not parameter_values.empty else np.nan
parameter_max_b = float(parameter_values.max()) if not parameter_values.empty else np.nan

print("Distinct model/precision configurations:", model_configuration_count)
print("Context-specific deployments:", context_deployment_count)
print("Unique model families:", model_family_count)
print("GPU configurations:", gpu_configuration_count)
display(table_02)


## 7. Figure 1: benchmarking framework architecture

In [ ]:

# Figure 1 — compact architecture aligned with the manuscript caption.
fig, ax = plt.subplots(figsize=(DOUBLE_COLUMN_WIDTH_IN, 2.25))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

FONT = "DejaVu Sans"
NAVY = "#18364f"
BLUE = "#2f6f9f"
TEAL = "#2a7f82"
PALE_BLUE = "#eaf2f8"
PALE_TEAL = "#e9f5f4"
PALE_GREY = "#f4f6f8"
LINE_GREY = "#7d8891"
TEXT_GREY = "#46525d"
WHITE = "#ffffff"

def card(
    x, y, w, h,
    title, subtitle,
    facecolor=WHITE,
    edgecolor=LINE_GREY,
    title_size=6.45,
    subtitle_size=5.25,
):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.009,rounding_size=0.014",
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=0.9,
        zorder=3,
    )
    ax.add_patch(patch)

    ax.text(
        x + w / 2,
        y + h * 0.66,
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        color=NAVY,
        fontfamily=FONT,
        linespacing=1.0,
        zorder=4,
    )
    ax.text(
        x + w / 2,
        y + h * 0.28,
        subtitle,
        ha="center",
        va="center",
        fontsize=subtitle_size,
        color=TEXT_GREY,
        fontfamily=FONT,
        linespacing=1.05,
        zorder=4,
    )
    return patch

def arrow(x1, y1, x2, y2, color=LINE_GREY, lw=0.85):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops={
            "arrowstyle": "-|>",
            "linewidth": lw,
            "color": color,
            "shrinkA": 2,
            "shrinkB": 2,
            "mutation_scale": 7,
        },
        zorder=2,
    )

# ------------------------------------------------------------------
# Execution path
# ------------------------------------------------------------------
ax.text(
    0.025, 0.925,
    "EXECUTION PATH",
    ha="left", va="center",
    fontsize=6.15, fontweight="bold",
    color="#66737e", fontfamily=FONT,
)

top_y, top_h, top_w = 0.625, 0.235, 0.158
top_x = [0.025, 0.223, 0.421, 0.619, 0.817]

top_cards = [
    ("YAML\nconfiguration", "model · dataset\nbackend · decoding"),
    ("Python\norchestration", "fresh server / run\nwarm-up · repeats"),
    ("Inference\nbackend", "llama.cpp: edge / CPU\nvLLM: GPU"),
    ("Streaming\nAPI", "OpenAI-compatible\nsequential token stream"),
    ("Evaluation\nloop", "generate · parse\nscore · retain output"),
]

for index, (x, (title, subtitle)) in enumerate(zip(top_x, top_cards)):
    highlighted = index in {1, 2, 3}
    card(
        x, top_y, top_w, top_h,
        title, subtitle,
        facecolor=PALE_BLUE if highlighted else WHITE,
        edgecolor=BLUE if highlighted else LINE_GREY,
    )

    if index:
        previous_x = top_x[index - 1]
        arrow(
            previous_x + top_w,
            top_y + top_h / 2,
            x,
            top_y + top_h / 2,
            color=BLUE if index >= 2 else LINE_GREY,
        )

# ------------------------------------------------------------------
# Inline measurement band
# ------------------------------------------------------------------
ax.text(
    0.421, 0.500,
    "INLINE MEASUREMENT",
    ha="left", va="center",
    fontsize=5.95, fontweight="bold",
    color="#66737e", fontfamily=FONT,
)

measure_x, measure_y, measure_w, measure_h = 0.421, 0.155, 0.356, 0.265
measurement = FancyBboxPatch(
    (measure_x, measure_y), measure_w, measure_h,
    boxstyle="round,pad=0.009,rounding_size=0.014",
    facecolor=PALE_TEAL,
    edgecolor=TEAL,
    linewidth=0.95,
    zorder=3,
)
ax.add_patch(measurement)

column_labels = [
    ("SYSTEM", "CPU/GPU\nRAM/VRAM"),
    ("TIMING", "TTFT · latency\ntokens/s"),
    ("TASK", "output · answer\ncorrectness"),
]

for fraction in (1 / 3, 2 / 3):
    x_sep = measure_x + measure_w * fraction
    ax.plot(
        [x_sep, x_sep],
        [measure_y + 0.035, measure_y + measure_h - 0.035],
        color="#a9c8c6",
        linewidth=0.7,
        zorder=4,
    )

for index, (heading, details) in enumerate(column_labels):
    cx = measure_x + measure_w * (index + 0.5) / 3
    ax.text(
        cx,
        measure_y + measure_h * 0.68,
        heading,
        ha="center", va="center",
        fontsize=5.65, fontweight="bold",
        color="#356b6e", fontfamily=FONT,
        zorder=4,
    )
    ax.text(
        cx,
        measure_y + measure_h * 0.33,
        details,
        ha="center", va="center",
        fontsize=5.05,
        color=TEXT_GREY,
        fontfamily=FONT,
        linespacing=1.10,
        zorder=4,
    )

# Direct, aligned drops: backend→system, API→timing, evaluation→task.
drop_sources = [
    top_x[2] + top_w / 2,
    top_x[3] + top_w / 2,
    top_x[4] + top_w / 2,
]
drop_targets = [
    measure_x + measure_w / 6,
    measure_x + measure_w / 2,
    measure_x + measure_w * 5 / 6,
]

for source_x, target_x in zip(drop_sources, drop_targets):
    arrow(
        source_x,
        top_y,
        target_x,
        measure_y + measure_h,
        color=TEAL,
        lw=0.78,
    )

# ------------------------------------------------------------------
# Structured output
# ------------------------------------------------------------------
output_x, output_y, output_w, output_h = 0.817, 0.155, 0.158, 0.265
card(
    output_x, output_y, output_w, output_h,
    "Structured\nresults",
    "config snapshot\nsample + summary CSV\nserver logs",
    facecolor=PALE_GREY,
    edgecolor=LINE_GREY,
    title_size=6.35,
    subtitle_size=5.05,
)

arrow(
    measure_x + measure_w,
    measure_y + measure_h / 2,
    output_x,
    output_y + output_h / 2,
    color=TEAL,
)

save_figure(fig, "figure_01_framework_architecture")


## 8. Figures 2 and 3: context-specific memory and GPU-resource figures

Figures 2 and 3 report system-RAM capacity requirements for edge and CPU contexts. GPU resource figures are deferred to the final notebook task: Figure 2a1 reports mean GPU energy per request for Dolly and GSM8K; Figure 2a2 reports mean GPU energy per request for HellaSwag and MMLU; Figure 3a reports absolute VRAM deployment footprint; Figure 3b1 and Figure 3b2 report GPU energy-per-request distributions as grouped boxplots.


In [ ]:

def plot_context_memory_bars(
    context: str,
    stem: str,
    metric_column: str,
    y_label: str,
    *,
    require_complete_dolly: bool,
    device_limit_gib: float | None = None,
):
    """Plot grouped peak-memory bars ordered by increasing Dolly peak memory.

    Workloads always follow BENCHMARK_ORDER_ALPHABETICAL. Model order is
    determined independently within each deployment context by increasing
    Dolly peak memory, with missing Dolly measurements placed last.
    """
    frame = context_subset(context)

    if metric_column not in frame.columns:
        raise ValueError(
            f"{metric_column} unavailable; cannot generate {stem}."
        )

    frame = frame.copy()
    frame[metric_column] = pd.to_numeric(
        frame[metric_column],
        errors="coerce",
    )

    expected_models = sorted(
        frame["model_display_canonical"]
        .dropna()
        .astype(str)
        .unique()
    )

    any_memory_models = set(
        frame.loc[
            frame[metric_column].notna(),
            "model_display_canonical",
        ]
        .dropna()
        .astype(str)
    )
    missing_all_memory_models = sorted(
        set(expected_models) - any_memory_models
    )
    if missing_all_memory_models:
        raise ValueError(
            f"{stem} has no {metric_column} value for: "
            + "; ".join(missing_all_memory_models)
        )

    dolly_memory = frame[
        frame["dataset"].eq("dolly")
        & frame[metric_column].notna()
    ].copy()

    available_dolly_models = set(
        dolly_memory["model_display_canonical"]
        .dropna()
        .astype(str)
    )
    missing_dolly_models = sorted(
        set(expected_models) - available_dolly_models
    )

    if require_complete_dolly and missing_dolly_models:
        raise ValueError(
            f"{stem} cannot be fully ordered by Dolly because Dolly "
            f"{metric_column} is missing for: "
            + "; ".join(missing_dolly_models)
        )

    if dolly_memory.empty:
        raise ValueError(
            f"{stem} requires at least one Dolly {metric_column} "
            "measurement for model ordering."
        )

    available_benchmarks = [
        dataset
        for dataset in BENCHMARK_ORDER_ALPHABETICAL
        if dataset in frame["dataset"].unique()
    ]

    pivot = (
        frame.pivot_table(
            index="model_display_canonical",
            columns="dataset",
            values=metric_column,
            aggfunc="max",
        )
        .reindex(columns=available_benchmarks)
        / 1024.0
    )

    if pivot.empty:
        raise ValueError(
            f"No plottable {metric_column} values are available for {stem}."
        )

    # Grouped bars and legend use the exact alphabetical workload order.
    assert list(pivot.columns) == available_benchmarks

    if "dolly" not in pivot.columns:
        raise ValueError(
            f"{stem} cannot be ordered by Dolly because the Dolly "
            "column is unavailable after pivoting."
        )

    # Sort smallest Dolly peak first. A canonical-name tie-breaker makes
    # the order deterministic when Dolly peaks are equal.
    order_table = pd.DataFrame({
        "model_configuration": pivot.index.astype(str),
        "dolly_peak_memory_gib": pivot["dolly"].to_numpy(dtype=float),
    })
    order_table["dolly_measurement_missing"] = (
        order_table["dolly_peak_memory_gib"].isna()
    )
    order_table = (
        order_table.sort_values(
            [
                "dolly_measurement_missing",
                "dolly_peak_memory_gib",
                "model_configuration",
            ],
            ascending=[True, True, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )
    order_table["model_order_position"] = (
        np.arange(len(order_table)) + 1
    )
    order_table["sort_direction"] = (
        "increasing Dolly peak memory"
    )
    order_table["memory_metric_column"] = metric_column
    order_table["memory_resource"] = y_label

    model_order = order_table[
        "model_configuration"
    ].tolist()
    pivot = pivot.reindex(model_order)

    model_count = len(pivot)
    workload_count = max(len(pivot.columns), 1)
    x_positions = np.arange(model_count)
    group_width = 0.82
    bar_width = group_width / workload_count

    is_gpu = context == "gpu_server"
    figure_width = (
        DOUBLE_COLUMN_WIDTH_IN
        if is_gpu
        else SINGLE_COLUMN_WIDTH_IN
    )
    figure_height = 4.15 if is_gpu else 3.30

    fig, ax = plt.subplots(
        figsize=(figure_width, figure_height),
        constrained_layout=True,
    )

    for workload_index, dataset in enumerate(pivot.columns):
        values = pivot[dataset].to_numpy(dtype=float)
        offsets = (
            x_positions
            - group_width / 2
            + bar_width / 2
            + workload_index * bar_width
        )
        mask = np.isfinite(values)

        ax.bar(
            offsets[mask],
            values[mask],
            width=bar_width * 0.92,
            label=DATASET_DISPLAY[dataset],
            color=DATASET_COLORS[dataset],
            hatch=DATASET_HATCHES[dataset],
            edgecolor="0.20",
            linewidth=0.60,
            zorder=3,
        )

    maximum_value = float(
        np.nanmax(pivot.to_numpy(dtype=float))
    )

    if device_limit_gib is not None:
        ax.axhline(
            device_limit_gib,
            color="0.35",
            linestyle="--",
            linewidth=0.85,
            zorder=2,
        )
        ax.text(
            model_count - 0.45,
            device_limit_gib + 0.06,
            f"{device_limit_gib:g} GiB device limit",
            ha="right",
            va="bottom",
            fontsize=6.2,
            color="0.35",
        )
        y_limit = max(
            device_limit_gib * 1.07,
            maximum_value * 1.14,
        )
    else:
        y_limit = maximum_value * (
            1.22 if is_gpu else 1.18
        )

    ax.set_ylim(0, y_limit)
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel(y_label)
    ax.set_xlabel(
        "Model configuration "
        "(increasing Dolly peak memory)"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=15 if not is_gpu else 13,
                max_lines=3,
            )
            for model in pivot.index
        ],
        rotation=58 if is_gpu else 24,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15 if is_gpu else 6.25,
    )

    style_axis(ax, grid_axis="y")
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=4 if is_gpu else 2,
        frameon=False,
        columnspacing=0.8,
        handletextpad=0.35,
        handlelength=1.8,
        handleheight=1.05,
    )

    # Export the exact values in visual model and workload order.
    value_export = (
        pivot.reset_index()
        .rename(
            columns={
                "model_display_canonical": "model_configuration"
            }
        )
    )
    value_export.insert(
        0,
        "model_order_position",
        np.arange(len(value_export)) + 1,
    )
    value_export.insert(
        1,
        "model_order_basis",
        "increasing Dolly peak memory",
    )
    value_export.to_csv(
        TABLE_DIR / f"{stem}_values.csv",
        index=False,
    )

    order_table.to_csv(
        CHECK_DIR / f"{stem}_model_order_dolly.csv",
        index=False,
    )

    pd.DataFrame({
        "benchmark_order_position": np.arange(
            1,
            len(available_benchmarks) + 1,
        ),
        "benchmark": available_benchmarks,
        "benchmark_display": [
            DATASET_DISPLAY[dataset]
            for dataset in available_benchmarks
        ],
    }).to_csv(
        CHECK_DIR / f"{stem}_benchmark_order.csv",
        index=False,
    )

    missingness_rows = []
    for model in pivot.index:
        for dataset in pivot.columns:
            missingness_rows.append({
                "model_configuration": model,
                "benchmark": dataset,
                "memory_value_gib": pivot.loc[model, dataset],
                "measurement_missing": pd.isna(
                    pivot.loc[model, dataset]
                ),
            })
    pd.DataFrame(missingness_rows).to_csv(
        CHECK_DIR / f"{stem}_missingness.csv",
        index=False,
    )

    save_figure(fig, stem)
    return pivot, order_table


In [ ]:
figure_02_values, figure_02_order = plot_context_memory_bars(
    "edge",
    "figure_02_pi5_peak_system_ram",
    "ram_used_mb_peak",
    "Peak system RAM (GiB)",
    require_complete_dolly=True,
    device_limit_gib=8.0,
)

figure_03_values, figure_03_order = plot_context_memory_bars(
    "local_cpu",
    "figure_03_cpu_peak_system_ram",
    "ram_used_mb_peak",
    "Peak system RAM (GiB)",
    require_complete_dolly=True,
)

display(figure_02_order)
display(figure_03_order)


### Figure 3a is deferred

The GPU VRAM box-plot definitions and execution are placed near the end of the notebook. Running this section therefore generates Figures 2 and 3 without loading the large GPU trace table.


## 9. Figure 4: accuracy by benchmark and model

Figure 4 uses three benchmark-specific bar-chart panels in alphabetical order: GSM8K, HellaSwag, and MMLU. Accuracy is on the y-axis. The complete model labels are repeated beneath every panel, while Edge, Local CPU, and GPU-server configurations retain identical categorical positions and the shared GSM8K-based model order.


In [ ]:

scored = agg[
    agg["dataset"].isin(SCORED_DATASETS)
    & agg["accuracy"].notna()
].copy()

if scored.empty:
    raise ValueError(
        "No scored accuracy values are available for Figure 4."
    )

FIGURE_4_BENCHMARK_ORDER = (
    SCORED_BENCHMARK_ORDER_ALPHABETICAL
)

figure_4_model_positions = (
    SHARED_FIGURE_4_5_MODEL_POSITIONS.copy()
)
figure_4_position_map = (
    SHARED_FIGURE_4_5_POSITION_MAP
)
figure_4_context_boundaries = (
    SHARED_FIGURE_4_5_CONTEXT_BOUNDARIES
)
figure_4_context_midpoints = (
    SHARED_FIGURE_4_5_CONTEXT_MIDPOINTS
)

fig, axes = plt.subplots(
    len(FIGURE_4_BENCHMARK_ORDER),
    1,
    figsize=(DOUBLE_COLUMN_WIDTH_IN, 11.20),
    sharex=False,
    sharey=True,
    constrained_layout=False,
)
axes = np.atleast_1d(axes)

figure_4_export_rows = []

for panel_index, (ax, dataset) in enumerate(
    zip(
        axes,
        FIGURE_4_BENCHMARK_ORDER,
    )
):
    dataset_frame = scored[
        scored["dataset"].eq(dataset)
    ].copy()

    dataset_frame["model_key"] = list(
        zip(
            dataset_frame["context"],
            dataset_frame["model_display_canonical"],
        )
    )
    dataset_frame["x_position"] = (
        dataset_frame["model_key"].map(
            figure_4_position_map
        )
    )
    dataset_frame = dataset_frame[
        dataset_frame["x_position"].notna()
    ].copy()

    for context in CONTEXT_ORDER:
        context_rows = dataset_frame[
            dataset_frame["context"].eq(context)
        ].copy()

        if context_rows.empty:
            continue

        bars = ax.bar(
            context_rows["x_position"],
            context_rows["accuracy"],
            width=0.74,
            color=CONTEXT_COLORS[context],
            hatch=CONTEXT_HATCHES[context],
            edgecolor="0.20",
            linewidth=0.55,
            label=CONTEXT_DISPLAY[context],
            zorder=3,
        )

        # Small numeric labels support precise reading without cluttering
        # the model axis. Values remain inside the [0, 1] plotting range.
        for bar, accuracy_value in zip(
            bars,
            context_rows["accuracy"],
        ):
            ax.annotate(
                f"{accuracy_value:.2f}",
                xy=(
                    bar.get_x() + bar.get_width() / 2.0,
                    bar.get_height(),
                ),
                xytext=(0, 2),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=4.75,
                color="0.25",
                rotation=90,
                annotation_clip=False,
                zorder=5,
            )

    for boundary in figure_4_context_boundaries:
        ax.axvline(
            boundary,
            color="0.72",
            linewidth=0.55,
            linestyle="--",
            zorder=1,
        )

    for context in CONTEXT_ORDER:
        if context not in figure_4_context_midpoints:
            continue
        ax.text(
            figure_4_context_midpoints[context],
            0.965,
            CONTEXT_DISPLAY[context],
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=6.35,
            fontweight="semibold",
            color="0.28",
            bbox={
                "boxstyle": "round,pad=0.12",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.88,
            },
            zorder=6,
        )

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accuracy [0, 1]")
    ax.set_title(
        f"({chr(ord('a') + panel_index)}) "
        f"{DATASET_DISPLAY[dataset]}",
        loc="left",
        fontsize=8.0,
        fontweight="bold",
        pad=4,
    )
    style_axis(ax, grid_axis="y")

    dataset_export = dataset_frame.copy()
    dataset_export["panel_order_position"] = panel_index + 1
    dataset_export["panel_label"] = (
        f"({chr(ord('a') + panel_index)}) "
        f"{DATASET_DISPLAY[dataset]}"
    )
    dataset_export["benchmark_order_position"] = (
        FIGURE_4_BENCHMARK_ORDER.index(dataset) + 1
    )
    dataset_export["x_tick_labels_repeated_on_each_panel"] = True
    dataset_export["model_axis_order_basis"] = (
        "descending GSM8K accuracy within context"
    )
    dataset_export["context_order_position"] = (
        dataset_export["context"].map({
            context_name: position + 1
            for position, context_name
            in enumerate(CONTEXT_ORDER)
        })
    )
    dataset_export["model_order_position"] = (
        dataset_export.apply(
            lambda row: (
                SHARED_CONTEXT_MODEL_ORDER[
                    row["context"]
                ].index(
                    row["model_display_canonical"]
                ) + 1
            ),
            axis=1,
        )
    )

    figure_4_export_rows.append(
        dataset_export[
            [
                "panel_order_position",
                "panel_label",
                "benchmark_order_position",
                "dataset",
                "x_tick_labels_repeated_on_each_panel",
                "model_axis_order_basis",
                "context_order_position",
                "context",
                "context_display",
                "model_order_position",
                "model_display_canonical",
                "accuracy",
                "x_position",
            ]
        ]
    )

# Repeat the full model labels below every benchmark panel. Independent
# x-axes prevent Matplotlib from suppressing labels on upper shared panels.
shared_tick_positions = figure_4_model_positions["x_position"].to_numpy()
shared_tick_labels = [
    publication_model_label(
        model,
        width=15,
        max_lines=2,
    )
    for model in figure_4_model_positions[
        "model_display_canonical"
    ]
]
shared_x_min = float(shared_tick_positions.min()) - 0.75
shared_x_max = float(shared_tick_positions.max()) + 0.75

for panel_index, ax in enumerate(axes):
    ax.set_xlim(shared_x_min, shared_x_max)
    ax.set_xticks(shared_tick_positions)
    ax.set_xticklabels(
        shared_tick_labels,
        rotation=62,
        ha="right",
        va="top",
        rotation_mode="anchor",
        fontsize=4.65,
    )
    ax.tick_params(
        axis="x",
        which="major",
        labelbottom=True,
        pad=1.5,
        length=2.5,
    )
    ax.set_xlabel(
        "Model configuration "
        "(shared GSM8K-based order)",
        fontsize=6.6,
        labelpad=3.0,
    )

legend_handles = [
    Patch(
        facecolor=CONTEXT_COLORS[context],
        edgecolor="0.20",
        hatch=CONTEXT_HATCHES[context],
        label=CONTEXT_DISPLAY[context],
    )
    for context in CONTEXT_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.992),
    ncol=3,
    frameon=False,
    columnspacing=1.15,
    handletextpad=0.45,
)

fig.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.955,
    bottom=0.070,
    hspace=0.92,
)

fig.text(
    0.5,
    0.012,
    (
        "Each panel isolates one scored benchmark. Accuracy is shown on "
        "the y-axis; model labels are repeated beneath every panel. "
        "Edge, Local CPU, and GPU-server configurations use the same "
        "categorical positions and GSM8K-based order as Figures 2, 3, "
        "and 5."
    ),
    ha="center",
    va="bottom",
    fontsize=6.05,
    color="0.35",
    wrap=True,
)

figure_4_export_long = (
    pd.concat(
        figure_4_export_rows,
        ignore_index=True,
    )
    if figure_4_export_rows
    else pd.DataFrame()
)

if not figure_4_export_long.empty:
    figure_4_export_long = (
        figure_4_export_long.sort_values(
            [
                "panel_order_position",
                "context_order_position",
                "model_order_position",
            ]
        )
        .reset_index(drop=True)
    )

figure_4_export_wide = (
    figure_4_export_long.pivot_table(
        index=[
            "context_order_position",
            "context",
            "context_display",
            "model_order_position",
            "model_display_canonical",
            "x_position",
        ],
        columns="dataset",
        values="accuracy",
        aggfunc="first",
    )
    .reindex(
        columns=FIGURE_4_BENCHMARK_ORDER
    )
    .reset_index()
)

figure_4_export_wide = figure_4_export_wide.rename(
    columns={
        dataset: f"{dataset}_accuracy"
        for dataset in FIGURE_4_BENCHMARK_ORDER
    }
)

figure_4_export_long.to_csv(
    TABLE_DIR / "figure_04_accuracy_panels_long.csv",
    index=False,
)
figure_4_export_wide.to_csv(
    TABLE_DIR / "figure_04_accuracy_panels_wide.csv",
    index=False,
)

# Descriptive aliases for the new Figure-5-style layout.
figure_4_export_long.to_csv(
    TABLE_DIR / "figure_04_accuracy_by_benchmark_long.csv",
    index=False,
)
figure_4_export_wide.to_csv(
    TABLE_DIR / "figure_04_accuracy_by_benchmark_wide.csv",
    index=False,
)

# Compatibility alias retained for earlier manuscript automation.
figure_4_export_long.to_csv(
    TABLE_DIR / "figure_04_accuracy_by_context_gsm8k_order.csv",
    index=False,
)

figure_4_export_schema = pd.DataFrame([
    {
        "export_file": "figure_04_accuracy_by_benchmark_long.csv",
        "format": "long",
        "one_row_per": "benchmark panel × context × model",
        "ordering": (
            "alphabetical scored-benchmark panel order, context order, "
            "shared GSM8K-based model order"
        ),
        "visual_mapping": (
            "x=model configuration in context blocks with labels "
            "repeated on every panel; y=accuracy; bar fill/hatch=context"
        ),
    },
    {
        "export_file": "figure_04_accuracy_by_benchmark_wide.csv",
        "format": "wide",
        "one_row_per": "context × model",
        "ordering": (
            "context order and shared GSM8K-based model order; "
            "columns GSM8K, HellaSwag, MMLU"
        ),
        "visual_mapping": "one accuracy column per benchmark panel",
    },
])
figure_4_export_schema.to_csv(
    CHECK_DIR / "figure_04_export_schema.csv",
    index=False,
)

save_figure(
    fig,
    "figure_04_accuracy_by_context_and_dataset",
)

display(figure_4_export_long)
display(figure_4_export_wide)


## 10. Figure 5: throughput by benchmark and model

Figure 5 uses four benchmark-specific bar-chart panels in alphabetical order. Throughput is on a shared logarithmic y-axis. The complete model names are repeated beneath every panel using the exact context blocks, x-positions, label formatting, and GSM8K-based model order used in Figure 4.


In [ ]:

throughput_metric = "reported_throughput_tokens_per_sec"
throughput_frame = agg[
    agg[throughput_metric].notna()
    & agg["dataset"].isin(BENCHMARK_ORDER_ALPHABETICAL)
].copy()

if throughput_frame.empty:
    raise ValueError(
        "No throughput values are available for Figure 5."
    )

# Use exactly the same categorical x-axis as Figure 4.
ordered_model_positions = (
    SHARED_FIGURE_4_5_MODEL_POSITIONS.copy()
)
model_position_map = (
    SHARED_FIGURE_4_5_POSITION_MAP
)
context_boundaries = (
    SHARED_FIGURE_4_5_CONTEXT_BOUNDARIES
)
context_midpoints = (
    SHARED_FIGURE_4_5_CONTEXT_MIDPOINTS
)

# Runtime guard: Figure 5 must use the exact context/model/order/position
# sequence exported for the shared Figure 4–5 categorical axis.
shared_order_columns = [
    "context_order_position",
    "context",
    "model_order_position",
    "model_display_canonical",
    "x_position",
]
assert ordered_model_positions[
    shared_order_columns
].reset_index(drop=True).equals(
    SHARED_FIGURE_4_5_MODEL_POSITIONS[
        shared_order_columns
    ].reset_index(drop=True)
)

# A logarithmic y-axis is necessary because the observed values span
# approximately one or more orders of magnitude.
positive_values = pd.to_numeric(
    throughput_frame[throughput_metric],
    errors="coerce",
)
positive_values = positive_values[
    positive_values.gt(0)
]
if positive_values.empty:
    raise ValueError(
        "Figure 5 requires positive throughput values."
    )

global_y_min = max(
    0.5,
    float(positive_values.min()) / 1.45,
)
global_y_max = float(positive_values.max()) * 1.65

fig, axes = plt.subplots(
    len(BENCHMARK_ORDER_ALPHABETICAL),
    1,
    figsize=(DOUBLE_COLUMN_WIDTH_IN, 15.20),
    sharex=False,
    sharey=True,
    constrained_layout=False,
)
axes = np.atleast_1d(axes)

figure_5_plot_rows = []
spread_rows = []

for panel_index, (ax, dataset) in enumerate(
    zip(
        axes,
        BENCHMARK_ORDER_ALPHABETICAL,
    )
):
    dataset_frame = throughput_frame[
        throughput_frame["dataset"].eq(dataset)
    ].copy()

    dataset_frame["model_key"] = list(
        zip(
            dataset_frame["context"],
            dataset_frame[
                "model_display_canonical"
            ],
        )
    )
    dataset_frame["x_position"] = (
        dataset_frame["model_key"].map(
            model_position_map
        )
    )
    dataset_frame = dataset_frame[
        dataset_frame["x_position"].notna()
    ].copy()

    for context in CONTEXT_ORDER:
        context_rows = dataset_frame[
            dataset_frame["context"].eq(context)
        ].copy()

        if context_rows.empty:
            continue

        ax.bar(
            context_rows["x_position"],
            context_rows[throughput_metric],
            width=0.74,
            color=CONTEXT_COLORS[context],
            hatch=CONTEXT_HATCHES[context],
            edgecolor="0.20",
            linewidth=0.55,
            label=CONTEXT_DISPLAY[context],
            zorder=3,
        )

        positive_context = context_rows[
            context_rows[throughput_metric].gt(0)
        ].copy()
        if not positive_context.empty:
            fastest = positive_context.loc[
                positive_context[
                    throughput_metric
                ].idxmax()
            ]
            slowest = positive_context.loc[
                positive_context[
                    throughput_metric
                ].idxmin()
            ]
            spread = (
                fastest[throughput_metric]
                / slowest[throughput_metric]
            )

            spread_rows.append({
                "benchmark": dataset,
                "benchmark_display": DATASET_DISPLAY[dataset],
                "context": context,
                "context_display": CONTEXT_DISPLAY[context],
                "fastest_model": fastest[
                    "model_display_canonical"
                ],
                "fastest_throughput_tokens_per_sec": fastest[
                    throughput_metric
                ],
                "slowest_model": slowest[
                    "model_display_canonical"
                ],
                "slowest_throughput_tokens_per_sec": slowest[
                    throughput_metric
                ],
                "within_context_spread_x": spread,
                "throughput_measurement_basis": fastest.get(
                    "throughput_measurement_basis",
                    None,
                ),
            })

    # Context separators and labels make the shared model ordering explicit.
    for boundary in context_boundaries:
        ax.axvline(
            boundary,
            color="0.72",
            linewidth=0.55,
            linestyle="--",
            zorder=1,
        )

    for context in CONTEXT_ORDER:
        if context not in context_midpoints:
            continue
        ax.text(
            context_midpoints[context],
            0.965,
            CONTEXT_DISPLAY[context],
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=6.35,
            fontweight="semibold",
            color="0.28",
            bbox={
                "boxstyle": "round,pad=0.12",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.88,
            },
            zorder=6,
        )

    panel_spreads = [
        row
        for row in spread_rows
        if row["benchmark"] == dataset
    ]
    spread_text = " | ".join(
        (
            f'{row["context_display"]}: '
            f'{row["within_context_spread_x"]:.1f}×'
        )
        for row in panel_spreads
    )

    ax.text(
        0.995,
        0.93,
        "Within-context fastest/slowest\n"
        + spread_text,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=5.75,
        color="0.30",
        bbox={
            "boxstyle": "round,pad=0.20",
            "facecolor": "white",
            "edgecolor": "0.80",
            "linewidth": 0.45,
            "alpha": 0.96,
        },
        zorder=5,
    )

    ax.set_yscale("log")
    ax.set_ylim(global_y_min, global_y_max)
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, _:
            f"{value:g}"
        )
    )
    ax.set_ylabel("Throughput\n(tokens/s)")
    ax.set_title(
        f"({chr(ord('a') + panel_index)}) "
        f"{DATASET_DISPLAY[dataset]}",
        loc="left",
        fontsize=8.0,
        fontweight="bold",
        pad=4,
    )
    style_axis(ax, grid_axis="y")

    dataset_export = dataset_frame.copy()
    dataset_export["panel_order_position"] = panel_index + 1
    dataset_export["panel_label"] = (
        f"({chr(ord('a') + panel_index)}) "
        f"{DATASET_DISPLAY[dataset]}"
    )
    dataset_export["benchmark_order_position"] = (
        BENCHMARK_ORDER_ALPHABETICAL.index(dataset) + 1
    )
    dataset_export["shares_model_axis_with_figure_4"] = True
    dataset_export["x_tick_labels_repeated_on_each_panel"] = True
    dataset_export["model_axis_order_basis"] = (
        "descending GSM8K accuracy within context"
    )
    dataset_export["context_order_position"] = (
        dataset_export["context"].map({
            context_name: position + 1
            for position, context_name
            in enumerate(CONTEXT_ORDER)
        })
    )
    dataset_export["model_order_position"] = (
        dataset_export.apply(
            lambda row: (
                SHARED_CONTEXT_MODEL_ORDER[
                    row["context"]
                ].index(
                    row["model_display_canonical"]
                ) + 1
            ),
            axis=1,
        )
    )

    figure_5_plot_rows.append(
        dataset_export[
            [
                column
                for column in [
                    "panel_order_position",
                    "panel_label",
                    "benchmark_order_position",
                    "dataset",
                    "shares_model_axis_with_figure_4",
                    "x_tick_labels_repeated_on_each_panel",
                    "model_axis_order_basis",
                    "context_order_position",
                    "context",
                    "context_display",
                    "model_order_position",
                    "model_display_canonical",
                    throughput_metric,
                    "throughput_measurement_basis",
                    "throughput_source_column",
                    "x_position",
                ]
                if column in dataset_export.columns
            ]
        ]
    )

# Repeat the complete Figure 4 model axis beneath every Figure 5 panel.
# Independent x-axes prevent Matplotlib from suppressing upper-panel labels.
figure_5_tick_positions = (
    ordered_model_positions["x_position"].to_numpy()
)
figure_5_tick_labels = [
    publication_model_label(
        model,
        width=15,
        max_lines=2,
    )
    for model in ordered_model_positions[
        "model_display_canonical"
    ]
]
figure_5_x_min = float(
    figure_5_tick_positions.min()
) - 0.75
figure_5_x_max = float(
    figure_5_tick_positions.max()
) + 0.75

for panel_index, ax in enumerate(axes):
    ax.set_xlim(
        figure_5_x_min,
        figure_5_x_max,
    )
    ax.set_xticks(
        figure_5_tick_positions
    )
    ax.set_xticklabels(
        figure_5_tick_labels,
        rotation=62,
        ha="right",
        va="top",
        rotation_mode="anchor",
        fontsize=4.65,
    )
    ax.tick_params(
        axis="x",
        which="major",
        labelbottom=True,
        pad=1.5,
        length=2.5,
    )
    ax.set_xlabel(
        "Model configuration "
        "(same positions and GSM8K-based order as Figure 4)",
        fontsize=6.6,
        labelpad=3.0,
    )

# One context legend for the complete multi-panel figure.
legend_handles = [
    Patch(
        facecolor=CONTEXT_COLORS[context],
        edgecolor="0.20",
        hatch=CONTEXT_HATCHES[context],
        label=CONTEXT_DISPLAY[context],
    )
    for context in CONTEXT_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.994),
    ncol=3,
    frameon=False,
    columnspacing=1.15,
    handletextpad=0.45,
)

fig.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.965,
    bottom=0.055,
    hspace=1.02,
)

fig.text(
    0.5,
    0.010,
    (
        "The complete model labels are repeated beneath every benchmark "
        "panel. The shared logarithmic y-axis exposes order-of-magnitude "
        "differences. Edge and Local CPU report active-generation "
        "tokens/s; GPU reports run-level sustained tokens/s. "
        "Cross-context gaps therefore combine hardware, backend, model, "
        "and denominator effects and are not a controlled backend "
        "comparison. Boxed ratios quantify variation within the same "
        "benchmark and deployment context."
    ),
    ha="center",
    va="bottom",
    fontsize=6.05,
    color="0.35",
    wrap=True,
)

figure_5_export_long = (
    pd.concat(
        figure_5_plot_rows,
        ignore_index=True,
    )
    if figure_5_plot_rows
    else pd.DataFrame()
)

if not figure_5_export_long.empty:
    figure_5_export_long = (
        figure_5_export_long.sort_values(
            [
                "panel_order_position",
                "context_order_position",
                "model_order_position",
            ]
        )
        .reset_index(drop=True)
    )

figure_5_spread = pd.DataFrame(
    spread_rows
)

if not figure_5_spread.empty:
    figure_5_spread["benchmark_order_position"] = (
        figure_5_spread["benchmark"].map({
            benchmark: position + 1
            for position, benchmark
            in enumerate(BENCHMARK_ORDER_ALPHABETICAL)
        })
    )
    figure_5_spread["context_order_position"] = (
        figure_5_spread["context"].map({
            context_name: position + 1
            for position, context_name
            in enumerate(CONTEXT_ORDER)
        })
    )
    figure_5_spread = (
        figure_5_spread.sort_values(
            [
                "benchmark_order_position",
                "context_order_position",
            ]
        )
        .reset_index(drop=True)
    )

# Wide export mirrors the four benchmark panels: one row per context/model
# and one throughput column per alphabetically ordered workload.
if not figure_5_export_long.empty:
    figure_5_export_wide = (
        figure_5_export_long.pivot_table(
            index=[
                "context_order_position",
                "context",
                "context_display",
                "model_order_position",
                "model_display_canonical",
                "x_position",
            ],
            columns="dataset",
            values=throughput_metric,
            aggfunc="first",
        )
        .reindex(
            columns=BENCHMARK_ORDER_ALPHABETICAL
        )
        .reset_index()
    )
    figure_5_export_wide = figure_5_export_wide.rename(
        columns={
            dataset: f"{dataset}_tokens_per_sec"
            for dataset in BENCHMARK_ORDER_ALPHABETICAL
        }
    )
else:
    figure_5_export_wide = pd.DataFrame()

figure_5_export_long.to_csv(
    TABLE_DIR / "figure_05_throughput_panels_long.csv",
    index=False,
)
figure_5_export_wide.to_csv(
    TABLE_DIR / "figure_05_throughput_panels_wide.csv",
    index=False,
)
figure_5_spread.to_csv(
    TABLE_DIR / "figure_05_throughput_spread_by_panel.csv",
    index=False,
)

# Compatibility aliases for previous evidence-pack versions.
figure_5_export_long.to_csv(
    TABLE_DIR / "figure_05_throughput_by_benchmark.csv",
    index=False,
)
figure_5_spread.to_csv(
    TABLE_DIR / "figure_05_throughput_spread_by_benchmark.csv",
    index=False,
)

figure_5_export_schema = pd.DataFrame([
    {
        "export_file": "figure_05_throughput_panels_long.csv",
        "format": "long",
        "one_row_per": "benchmark panel × context × model",
        "ordering": (
            "alphabetical benchmark panel order, context order, "
            "the exact canonical model positions shared with Figure 4; "
            "full model labels repeated on every panel"
        ),
    },
    {
        "export_file": "figure_05_throughput_panels_wide.csv",
        "format": "wide",
        "one_row_per": "context × model",
        "ordering": (
            "context order and the exact canonical model positions "
            "shared with Figure 4; columns Dolly, GSM8K, HellaSwag, MMLU"
        ),
    },
    {
        "export_file": "figure_05_throughput_spread_by_panel.csv",
        "format": "summary",
        "one_row_per": "benchmark panel × context",
        "ordering": "alphabetical benchmark order and context order",
    },
])
figure_5_export_schema.to_csv(
    CHECK_DIR / "figure_05_export_schema.csv",
    index=False,
)

# Generate a data-grounded interpretation note for the manuscript.
global_ratio = (
    float(positive_values.max())
    / float(positive_values.min())
)
interpretation_lines = [
    "# Figure 5 throughput interpretation",
    "",
    (
        "The observed reported-throughput range spans "
        f"{global_ratio:.1f}× from the smallest to the largest value, "
        "so Figure 5 uses a logarithmic y-axis."
    ),
    "",
    (
        "Cross-context differences must not be attributed to model size "
        "alone. Edge and Local CPU use llama.cpp active-generation "
        "tokens/s, while the GPU server uses vLLM run-level sustained "
        "tokens/s. Hardware, backend, precision/quantization, model "
        "architecture, and the throughput denominator all differ."
    ),
    "",
    (
        "Within a fixed benchmark and deployment context, the boxed "
        "fastest/slowest ratios provide the cleaner comparison. Those "
        "spreads reflect the observed combined effect of model scale, "
        "architecture, precision/quantization, runtime behavior, and "
        "generated-output behavior under the same workload."
    ),
    "",
    "## Within-benchmark, within-context spreads",
    "",
    markdown_table(
        figure_5_spread[
            [
                "benchmark_display",
                "context_display",
                "fastest_model",
                "fastest_throughput_tokens_per_sec",
                "slowest_model",
                "slowest_throughput_tokens_per_sec",
                "within_context_spread_x",
                "throughput_measurement_basis",
            ]
        ],
        float_format=".3f",
    ),
]
(
    TEXT_DIR / "figure_05_throughput_interpretation.md"
).write_text(
    "\n".join(interpretation_lines) + "\n",
    encoding="utf-8",
)

save_figure(
    fig,
    "figure_05_sustained_throughput_across_contexts",
)

display(figure_5_export_long)
display(figure_5_export_wide)
display(figure_5_spread)


## 11. Figures 6–7: GSM8K context-local Pareto frontiers

The panels are intentionally context-specific. A GPU point may globally dominate a CPU point, but that comparison is not actionable when GPU access is unavailable.


In [ ]:

gsm = agg[agg["dataset"].eq("gsm8k") & agg["accuracy"].notna()].copy()
pareto_records = []

def plot_context_frontier(frame, x, maximize_x, xlabel_by_context, stem):
    fig, axes = plt.subplots(
        1, 3,
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 3.35),
        sharey=True,
        constrained_layout=True,
    )
    letters = ["a", "b", "c"]

    for ax, context, letter in zip(axes, CONTEXT_ORDER, letters):
        sub = frame[frame["context"].eq(context)].dropna(subset=[x, "accuracy"]).copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        sub["pareto"] = pareto_mask(
            sub, x=x, y="accuracy",
            maximize_x=maximize_x,
            maximize_y=True,
        )

        for _, row in sub.iterrows():
            pareto_records.append({
                "frontier": stem,
                "context": context,
                "dataset": "gsm8k",
                "model": row["model_display_canonical"],
                "pareto": bool(row["pareto"]),
                "accuracy": row["accuracy"],
                "x_metric": x,
                "x_value": row[x],
            })

        dominated = sub[~sub["pareto"]]
        frontier = sub[sub["pareto"]].sort_values(x)

        ax.scatter(
            dominated[x], dominated["accuracy"],
            s=22, marker="o",
            facecolors="none", edgecolors="0.65",
            linewidths=0.7, alpha=0.75,
            zorder=2,
        )
        ax.plot(
            frontier[x], frontier["accuracy"],
            color=CONTEXT_COLORS[context], linewidth=1.15,
            zorder=3,
        )
        ax.scatter(
            frontier[x], frontier["accuracy"],
            s=38, marker="D",
            color=CONTEXT_COLORS[context],
            edgecolors="white", linewidths=0.45,
            zorder=4,
        )

        ax.set_xlabel(xlabel_by_context[context])
        add_panel_label(ax, letter, CONTEXT_DISPLAY[context])
        style_axis(ax, grid_axis="both")

        # Establish the data limits before adding labels. GPU receives a
        # dedicated right-side label column; Edge/CPU retain compact callouts.
        ax.margins(
            x=0.10 if context == "gpu_server" else 0.17,
            y=0.13,
        )
        annotate_frontier(
            ax,
            frontier,
            x=x,
            y="accuracy",
            label_side=(
                "right"
                if context == "gpu_server"
                else "auto"
            ),
        )
        ax.text(
            0.04, 0.94, "preferred",
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=6.2, color="0.45",
        )
        ax.annotate(
            "", xy=(0.04, 0.86), xytext=(0.16, 0.74),
            xycoords="axes fraction",
            arrowprops={"arrowstyle": "->", "linewidth": 0.55, "color": "0.45"},
        )

    axes[0].set_ylabel("GSM8K accuracy [0, 1]")
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="None", markersize=4.5,
               markerfacecolor="white", markeredgecolor="0.6"),
        Line2D([0], [0], marker="D", linestyle="-", markersize=4.5,
               color="0.35", markerfacecolor="0.35"),
    ]
    shared_legend(
        fig,
        legend_handles,
        ["dominated within context", "Pareto-relevant"],
        ncol=2,
        y=-0.10,
    )
    save_figure(fig, stem)


plot_context_frontier(
    gsm,
    x="latency_s",
    maximize_x=False,
    xlabel_by_context={context: "Mean end-to-end latency (s)" for context in CONTEXT_ORDER},
    stem="figure_06_gsm8k_accuracy_latency_frontiers",
)

plot_context_frontier(
    gsm,
    x="deployment_memory_gib",
    maximize_x=False,
    xlabel_by_context={
        "edge": "Peak system RAM (GiB)",
        "local_cpu": "Peak system RAM (GiB)",
        "gpu_server": "Peak VRAM (GiB)",
    },
    stem="figure_07_gsm8k_accuracy_memory_frontiers",
)

pareto_table = pd.DataFrame(pareto_records)
pareto_table.to_csv(TABLE_DIR / "gsm8k_context_pareto_membership.csv", index=False)
display(pareto_table[pareto_table["pareto"]])


## 12. Tables 4–7: empirical manuscript tables

Both full and selected versions are exported. The selected versions remain readable in a two-column IEEE layout.


In [ ]:

# Representative order used in the current manuscript.
REPRESENTATIVE_MODELS = OrderedDict({
    "edge": [
        "Llama 3.2 1B (Q4)",
        "Gemma 3 1B (Q8)",
        "SmolLM3 3B (Q4)",
        "Phi-4 Mini 3.8B (Q4)",
    ],
    "local_cpu": [
        "Llama 3.2 1B (Q8)",
        "Gemma 3 1B (Q8)",
        "Phi-4 Mini 3.8B (Q4)",
        "Ministral 3 3B (Q4)",
    ],
    "gpu_server": [
        "Llama 3.2 1B (bf16)",
        "Qwen 2.5 1.5B (bf16)",
        "Ministral 3 3B (AWQ)",
        "Phi-4 Mini 3.8B (bf16)",
        "Ministral 3 8B (AWQ)",
        "Ministral 3 14B",
        "Phi-4 14B (bf16)",
        "Qwen 3 30B-A3B (bf16)",
        "Llama 3.3 70B (FP8)",
    ],
})

representative_pairs = {
    (context, model)
    for context, models in REPRESENTATIVE_MODELS.items()
    for model in models
}

def representative_filter(frame):
    return frame.apply(
        lambda r: (r["context"], r["model_display_canonical"]) in representative_pairs,
        axis=1,
    )


# Table 4 — selected task accuracy.
table_04_source = scored[representative_filter(scored)].copy()
table_04 = (
    table_04_source.pivot_table(
        index=["context_display", "model_display_canonical"],
        columns="dataset",
        values="accuracy",
        aggfunc="mean",
    )
    .reset_index()
    .rename(columns={
        "context_display": "Context",
        "model_display_canonical": "Model",
        "mmlu": "MMLU",
        "gsm8k": "GSM8K",
        "hellaswag": "HellaSwag",
    })
)
for col in ["MMLU", "GSM8K", "HellaSwag"]:
    if col not in table_04:
        table_04[col] = np.nan
table_04 = table_04[["Context", "Model", "MMLU", "GSM8K", "HellaSwag"]]

export_table(
    table_04,
    "table_04_task_accuracy_selected",
    "Task-level accuracy per deployment context and dataset. Values are aggregated across repeated runs.",
    "tab:task-accuracy",
)

# Full accuracy table for appendix/audit.
table_04_full = (
    scored.pivot_table(
        index=["context_display", "model_display_canonical"],
        columns="dataset",
        values="accuracy",
        aggfunc="mean",
    )
    .reset_index()
)
table_04_full.to_csv(TABLE_DIR / "table_04_task_accuracy_full.csv", index=False)


# Table 5 — representative timing/throughput rows.
TIMING_SELECTION = [
    ("edge", "Llama 3.2 1B (Q4)", "mmlu"),
    ("edge", "Llama 3.2 1B (Q4)", "gsm8k"),
    ("edge", "Phi-4 Mini 3.8B (Q4)", "mmlu"),
    ("edge", "Phi-4 Mini 3.8B (Q4)", "gsm8k"),
    ("local_cpu", "Gemma 3 1B (Q8)", "mmlu"),
    ("local_cpu", "Phi-4 Mini 3.8B (Q4)", "mmlu"),
    ("local_cpu", "Phi-4 Mini 3.8B (Q4)", "gsm8k"),
    ("local_cpu", "Ministral 3 3B (Q4)", "mmlu"),
    ("gpu_server", "Llama 3.2 1B (bf16)", "dolly"),
    ("gpu_server", "Llama 3.2 1B (bf16)", "gsm8k"),
    ("gpu_server", "Qwen 2.5 1.5B (bf16)", "dolly"),
    ("gpu_server", "Ministral 3 3B (AWQ)", "dolly"),
    ("gpu_server", "Phi-4 Mini 3.8B (bf16)", "dolly"),
    ("gpu_server", "Ministral 3 8B (AWQ)", "dolly"),
    ("gpu_server", "Phi-4 14B (bf16)", "dolly"),
    ("gpu_server", "Ministral 3 14B", "dolly"),
    ("gpu_server", "Qwen 3 30B-A3B (bf16)", "dolly"),
    ("gpu_server", "Llama 3.3 70B (FP8)", "dolly"),
    ("gpu_server", "Apertus 70B (bf16)", "dolly"),
]
timing_keys = set(TIMING_SELECTION)
timing_selected = agg[
    agg.apply(
        lambda r: (
            r["context"], r["model_display_canonical"], r["dataset"]
        ) in timing_keys,
        axis=1,
    )
].copy()

table_05 = timing_selected[
    [
        "context_display",
        "model_display_canonical",
        "dataset",
        "ttft_ms",
        "latency_ms",
        "reported_throughput_tokens_per_sec",
        "throughput_measurement_basis",
    ]
].rename(columns={
    "context_display": "Context",
    "model_display_canonical": "Model",
    "dataset": "Dataset",
    "ttft_ms": "TTFT (ms)",
    "latency_ms": "Latency (ms)",
    "reported_throughput_tokens_per_sec": "Reported throughput (tokens/s)",
    "throughput_measurement_basis": "Throughput basis",
})
table_05["Dataset"] = table_05["Dataset"].map(DATASET_DISPLAY)

export_table(
    table_05,
    "table_05_timing_throughput_selected",
    (
        "Mean time to first token, end-to-end latency, and context-specific "
        "token throughput for representative configurations. Edge and Local "
        "CPU report active-generation tokens per second; the GPU server "
        "reports run-level sustained tokens per second."
    ),
    "tab:timing-throughput",
)


# Table 6 — selected resources.
RESOURCE_SELECTION = [
    ("edge", "Llama 3.2 1B (Q4)", "mmlu"),
    ("edge", "Gemma 3 1B (Q8)", "mmlu"),
    ("edge", "SmolLM3 3B (Q4)", "mmlu"),
    ("edge", "Phi-4 Mini 3.8B (Q4)", "mmlu"),
    ("local_cpu", "Llama 3.2 1B (Q8)", "gsm8k"),
    ("local_cpu", "Gemma 3 1B (Q8)", "mmlu"),
    ("local_cpu", "Phi-4 Mini 3.8B (Q4)", "mmlu"),
    ("local_cpu", "Ministral 3 3B (Q4)", "mmlu"),
    ("gpu_server", "Llama 3.2 1B (bf16)", "dolly"),
    ("gpu_server", "Phi-4 Mini 3.8B (bf16)", "dolly"),
    ("gpu_server", "Phi-4 14B (bf16)", "dolly"),
    ("gpu_server", "Ministral 3 14B", "dolly"),
    ("gpu_server", "Qwen 3 30B-A3B (bf16)", "dolly"),
    ("gpu_server", "Kimi Linear 48B-A3B (bf16)", "dolly"),
    ("gpu_server", "Llama 3.3 70B (FP8)", "dolly"),
    ("gpu_server", "Apertus 70B (bf16)", "dolly"),
]
resource_keys = set(RESOURCE_SELECTION)
resource_selected = agg[
    agg.apply(
        lambda r: (
            r["context"], r["model_display_canonical"], r["dataset"]
        ) in resource_keys,
        axis=1,
    )
].copy()

resource_cols = [
    "context_display", "model_display_canonical", "dataset",
    "cpu_percent_peak", "ram_used_mb_peak", "vram_used_mb_peak",
    "temperature_c_peak",
]
resource_cols = [c for c in resource_cols if c in resource_selected.columns]
table_06 = resource_selected[resource_cols].rename(columns={
    "context_display": "Context",
    "model_display_canonical": "Model",
    "dataset": "Dataset",
    "cpu_percent_peak": "CPU peak (%)",
    "ram_used_mb_peak": "RAM peak (MB)",
    "vram_used_mb_peak": "VRAM peak (MB)",
    "temperature_c_peak": "Temperature peak (°C)",
})
table_06["Dataset"] = table_06["Dataset"].map(DATASET_DISPLAY)

export_table(
    table_06,
    "table_06_peak_resources_selected",
    "Peak resource utilization for representative configurations.",
    "tab:peak-resources",
)


# Table 7 — corrected and explicitly defined cost-effectiveness metric.
mmlu = agg[
    agg["dataset"].eq("mmlu")
    & agg["accuracy"].notna()
].copy()
mmlu_selected = mmlu[representative_filter(mmlu)].copy()

table_07 = mmlu_selected[
    [
        "context_display",
        "model_display_canonical",
        "accuracy",
        "latency_s",
        "samples_per_sec",
        "expected_correct_per_second",
    ]
].rename(columns={
    "context_display": "Context",
    "model_display_canonical": "Model",
    "accuracy": "Accuracy [0, 1]",
    "latency_s": "Latency (s)",
    "samples_per_sec": "Samples/s",
    "expected_correct_per_second": "Expected correct responses/s",
})

table_07 = table_07.sort_values(
    ["Context", "Expected correct responses/s"],
    ascending=[True, False],
    na_position="last",
).reset_index(drop=True)

export_table(
    table_07,
    "table_07_cost_effectiveness_mmlu",
    (
        "Short-output cost-effectiveness on MMLU. Expected correct responses "
        "per second is benchmark accuracy [0, 1] multiplied by measured sample "
        "throughput under sequential inference."
    ),
    "tab:cost-effectiveness",
)


display(table_04)
display(table_05)
display(table_06)
display(table_07)


## 13. Supplementary evidence: compact-model deltas versus context-local best accuracy

These outputs support statements such as “a compact model gives up *x* accuracy points for *y*× throughput” without mixing baselines across contexts.


In [ ]:

def safe_ratio(numerator, denominator):
    return np.where(
        pd.notna(denominator) & (denominator != 0),
        numerator / denominator,
        np.nan,
    )


delta_parts = []
for (dataset, context), group in scored.groupby(["dataset", "context"], sort=False):
    g = group.copy()
    reference = g.loc[g["accuracy"].idxmax()]
    g["reference_model"] = reference["model_display_canonical"]
    g["reference_accuracy"] = reference["accuracy"]
    g["accuracy_loss_pp"] = 100.0 * (reference["accuracy"] - g["accuracy"])
    g["accuracy_retained_pct"] = 100.0 * safe_ratio(g["accuracy"], reference["accuracy"])
    if "tokens_per_sec" in g.columns:
        g["throughput_speedup_x"] = safe_ratio(g["tokens_per_sec"], reference["tokens_per_sec"])
    if "latency_ms" in g.columns:
        g["latency_reduction_pct"] = 100.0 * (
            1.0 - safe_ratio(g["latency_ms"], reference["latency_ms"])
        )
    g["competitive_95"] = (
        g["accuracy_retained_pct"] >= 100 * ACCURACY_RETENTION_THRESHOLD
    )
    delta_parts.append(g)

delta = pd.concat(delta_parts, ignore_index=True)
delta.to_csv(TABLE_DIR / "supplement_context_local_deltas.csv", index=False)

competitive_compact = delta[
    delta["compact_total"]
    & delta["competitive_95"]
    & (delta["accuracy_loss_pp"] > 1e-9)
].copy()
competitive_compact.to_csv(
    TABLE_DIR / "supplement_competitive_compact_models.csv",
    index=False,
)

for dataset in SCORED_DATASETS:
    sub = delta[delta["dataset"].eq(dataset)].copy()
    fig, axes = plt.subplots(
        1, 3,
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 2.75),
        sharey=True,
        constrained_layout=True,
    )
    letters = ["a", "b", "c"]
    threshold = 100 * (1 - ACCURACY_RETENTION_THRESHOLD)

    for ax, context, letter in zip(axes, CONTEXT_ORDER, letters):
        g = sub[sub["context"].eq(context)].dropna(
            subset=["accuracy_loss_pp", "latency_reduction_pct"]
        ).copy()
        if g.empty:
            ax.set_visible(False)
            continue

        preferred = g["competitive_95"] & (g["latency_reduction_pct"] >= 0)
        reference = g[g["accuracy_loss_pp"].abs() < 1e-9]
        others = g[~preferred & ~g.index.isin(reference.index)]

        y_min = min(float(g["latency_reduction_pct"].min()) - 5, -5)
        y_max = max(float(g["latency_reduction_pct"].max()) + 8, 10)
        ax.set_ylim(y_min, y_max)
        ax.axvspan(0, threshold, ymin=(0 - y_min) / (y_max - y_min), ymax=1.0,
                   color=CONTEXT_COLORS[context], alpha=0.08, linewidth=0)
        ax.axvline(threshold, linestyle="--", linewidth=0.75, color="0.45")
        ax.axhline(0, linestyle=":", linewidth=0.75, color="0.45")

        ax.scatter(
            others["accuracy_loss_pp"], others["latency_reduction_pct"],
            s=22, facecolors="none", edgecolors="0.65",
            linewidths=0.7, alpha=0.8,
        )
        ax.scatter(
            g.loc[preferred, "accuracy_loss_pp"],
            g.loc[preferred, "latency_reduction_pct"],
            s=38, marker="D",
            color=CONTEXT_COLORS[context],
            edgecolors="white", linewidths=0.45,
            zorder=4,
        )
        if not reference.empty:
            ax.scatter(
                reference["accuracy_loss_pp"], reference["latency_reduction_pct"],
                marker="*", s=52,
                color="0.20", zorder=5,
            )

        label_candidates = pd.concat([
            g[preferred].sort_values("latency_reduction_pct", ascending=False).head(2),
            g.sort_values("latency_reduction_pct", ascending=False).head(1),
        ]).drop_duplicates()
        for index, (_, row) in enumerate(label_candidates.head(3).iterrows()):
            dy = 7 if index % 2 == 0 else -11
            ax.annotate(
                short_name(row["model_display_canonical"], 18),
                xy=(row["accuracy_loss_pp"], row["latency_reduction_pct"]),
                xytext=(4, dy), textcoords="offset points",
                fontsize=6.2, ha="left", va="center",
                bbox={"boxstyle": "round,pad=0.10", "facecolor": "white", "edgecolor": "0.84", "alpha": 0.94},
                arrowprops={"arrowstyle": "-", "linewidth": 0.45, "color": "0.45"},
            )

        ax.set_xlim(left=-0.8)
        ax.set_xlabel("Accuracy sacrificed (pp)")
        add_panel_label(ax, letter, CONTEXT_DISPLAY[context])
        style_axis(ax, grid_axis="both")
        ax.text(
            0.03, 0.95, "preferred region",
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=6.1, color=CONTEXT_COLORS[context],
        )

    axes[0].set_ylabel("Latency reduction vs. accuracy leader (%)")
    legend_handles = [
        Line2D([0], [0], marker="*", linestyle="None", markersize=7,
               color="0.2", label="accuracy leader"),
        Line2D([0], [0], marker="D", linestyle="None", markersize=4.5,
               color="0.35", label="within 5 pp and faster"),
        Line2D([0], [0], marker="o", linestyle="None", markersize=4.5,
               markerfacecolor="none", markeredgecolor="0.6", label="other configuration"),
    ]
    shared_legend(
        fig,
        legend_handles,
        ["accuracy leader", "within 5 pp and faster", "other configuration"],
        ncol=3,
        y=-0.10,
    )
    save_figure(fig, f"supplement_{dataset}_compact_accuracy_latency_tradeoff")

display(competitive_compact[[
    "dataset", "context_display", "model_display_canonical",
    "total_parameters_b", "reference_model",
    "accuracy", "reference_accuracy", "accuracy_loss_pp",
    "accuracy_retained_pct", "throughput_speedup_x",
    "latency_reduction_pct",
]])


## 14. Canonical facts and manuscript text documents

The text files below are generated from the current CSV. They are intended as source material for revision—not as claims to copy without checking the confidence intervals and methodological caveats.


In [ ]:

def fmt(value, digits=3):
    return "NA" if pd.isna(value) else f"{value:.{digits}f}"


fact_rows = []

# Per-context and per-dataset leaders.
for context in CONTEXT_ORDER:
    for dataset in SCORED_DATASETS:
        g = agg[
            agg["context"].eq(context)
            & agg["dataset"].eq(dataset)
            & agg["accuracy"].notna()
        ].copy()
        if g.empty:
            continue

        accuracy_leader = g.loc[g["accuracy"].idxmax()]
        latency_leader = g.loc[g["latency_ms"].idxmin()]
        throughput_candidates = g.dropna(
            subset=["reported_throughput_tokens_per_sec"]
        )
        throughput_leader = (
            throughput_candidates.loc[
                throughput_candidates[
                    "reported_throughput_tokens_per_sec"
                ].idxmax()
            ]
            if not throughput_candidates.empty
            else None
        )

        fact_rows.extend([
            {
                "context": context,
                "dataset": dataset,
                "fact_type": "highest_accuracy",
                "model": accuracy_leader["model_display_canonical"],
                "value": accuracy_leader["accuracy"],
                "unit": "accuracy",
            },
            {
                "context": context,
                "dataset": dataset,
                "fact_type": "lowest_latency",
                "model": latency_leader["model_display_canonical"],
                "value": latency_leader["latency_ms"],
                "unit": "ms",
            },
            *(
                [{
                    "context": context,
                    "dataset": dataset,
                    "fact_type": "highest_context_specific_throughput",
                    "model": throughput_leader["model_display_canonical"],
                    "value": throughput_leader[
                        "reported_throughput_tokens_per_sec"
                    ],
                    "unit": "tokens/s",
                    "measurement_basis": throughput_leader[
                        "throughput_measurement_basis"
                    ],
                }]
                if throughput_leader is not None
                else []
            ),
        ])

facts = pd.DataFrame(fact_rows)
facts.to_csv(TEXT_DIR / "canonical_results_facts.csv", index=False)


def context_paragraph(context: str) -> str:
    frame = agg[agg["context"].eq(context)].copy()
    model_count = frame["model_display_canonical"].nunique()
    scored_frame = frame[frame["dataset"].isin(SCORED_DATASETS)]

    lines = [
        f"### {CONTEXT_DISPLAY[context]} context",
        "",
        f"The evaluated set contains **{model_count} context-specific model configurations**.",
    ]

    if frame["deployment_memory_gib"].notna().any():
        memory = frame["deployment_memory_gib"].dropna()
        memory_type = frame["deployment_memory_type"].dropna().iloc[0]
        lines.append(
            f"Observed peak {memory_type} spans **{memory.min():.2f}–{memory.max():.2f} GiB** "
            "across workloads and configurations."
        )

    for dataset in SCORED_DATASETS:
        g = scored_frame[scored_frame["dataset"].eq(dataset)]
        if g.empty:
            continue
        acc = g.loc[g["accuracy"].idxmax()]
        fast = g.loc[g["latency_ms"].idxmin()]
        lines.append(
            f"On {DATASET_DISPLAY[dataset]}, **{acc['model_display_canonical']}** has the "
            f"highest observed accuracy ({acc['accuracy']:.3f}), while "
            f"**{fast['model_display_canonical']}** has the lowest mean end-to-end latency "
            f"({fast['latency_ms']/1000:.2f} s)."
        )

    return "\n".join(lines)


results_draft = "\n\n".join(context_paragraph(c) for c in CONTEXT_ORDER)

# GPU diagnostic facts.
gpu = agg[agg["context"].eq("gpu_server")]
gpu_ttft = gpu["ttft_ms"].dropna()
dolly_gpu = gpu[gpu["dataset"].eq("dolly")].dropna(
    subset=["reported_throughput_tokens_per_sec"]
)
if not gpu_ttft.empty:
    results_draft += (
        "\n\n### GPU timing interpretation\n\n"
        f"Across the GPU configurations and workloads represented in the normalized table, "
        f"mean TTFT spans **{gpu_ttft.min():.1f}–{gpu_ttft.max():.1f} ms**. "
        "End-to-end latency and sustained generation throughput should therefore carry more "
        "weight than TTFT when outputs differ substantially in length."
    )
if not dolly_gpu.empty:
    fastest = dolly_gpu.loc[
        dolly_gpu["reported_throughput_tokens_per_sec"].idxmax()
    ]
    slowest = dolly_gpu.loc[
        dolly_gpu["reported_throughput_tokens_per_sec"].idxmin()
    ]
    results_draft += (
        f"\n\nOn the GPU server, Dolly run-level sustained throughput ranges "
        f"from **{slowest['reported_throughput_tokens_per_sec']:.1f}** "
        f"tokens/s for **{slowest['model_display_canonical']}** to "
        f"**{fastest['reported_throughput_tokens_per_sec']:.1f}** tokens/s "
        f"for **{fastest['model_display_canonical']}**. Edge and Local CPU "
        "throughput is reported separately as active-generation tokens per "
        "second and should not be interpreted as the same serving-level "
        "denominator."
    )

(TEXT_DIR / "results_section_data_draft.md").write_text(results_draft + "\n", encoding="utf-8")


# Abstract evidence block.
best_by_context = []
for context in CONTEXT_ORDER:
    g = scored[scored["context"].eq(context)]
    if g.empty:
        continue
    best = g.loc[g["accuracy"].idxmax()]
    best_by_context.append(
        f"{CONTEXT_DISPLAY[context]}: {best['model_display_canonical']} "
        f"({DATASET_DISPLAY[best['dataset']]} accuracy {best['accuracy']:.3f})"
    )

abstract_block = f"""## Abstract evidence block

**Method facts.** The normalized analysis contains **{model_configuration_count} distinct model and precision/quantization configurations** instantiated as **{context_deployment_count} context-specific deployments** across three deployment contexts and four workloads. Accuracy is available for MMLU, GSM8K, and HellaSwag; Dolly is treated as an open-ended generation workload.

**Result facts.** The accuracy-leading configuration changes by context and dataset. Examples from the current data are: {"; ".join(best_by_context)}. Context-local Pareto frontiers identify multiple relevant configurations rather than one universal winner.

**Interpretation.** The evidence supports a deployment-context-aware claim: feasibility and memory dominate on edge hardware, latency dominates on local CPU, and the GPU context exposes an accuracy–throughput trade-off. Avoid claiming that one model is globally best.
"""
(TEXT_DIR / "abstract_evidence_block.md").write_text(abstract_block, encoding="utf-8")



# Manuscript-ready statement replacements.
parameter_span = (
    f"approximately {parameter_min_b:.1f}B to {parameter_max_b:.0f}B parameters"
    if pd.notna(parameter_min_b) and pd.notna(parameter_max_b)
    else "the parameter range represented in Table 2"
)

statement_replacements = f"""# Manuscript statement replacements

These passages use the counting definitions reported in Table 2.


## Accuracy notation

Accuracy is reported as a proportion on the closed interval **[0, 1]**.
For example, 0.64 corresponds to 64% correct responses. Figures and tables
use the label `Accuracy [0, 1]` to prevent ambiguity with percentage units.

## Counting definition

A **model configuration** is one distinct model artifact and
precision/quantization variant. The same artifact is counted once even when
it is evaluated in more than one deployment context. A
**context-specific deployment** is one model configuration instantiated in
one deployment context.

Current audited counts:

- Distinct model/precision configurations: **{model_configuration_count}**
- Context-specific deployments: **{context_deployment_count}**
- Unique model families: **{model_family_count}**
- GPU-server configurations: **{gpu_configuration_count}**

## Contribution bullet replacement

We report empirical Pareto frontiers per context across
**{model_configuration_count} distinct model and precision/quantization
configurations**, instantiated as **{context_deployment_count}
context-specific deployments**, and show that the preferred model changes
by context: no single model dominates all three environments, and the
selection changes when the operational objective shifts from accuracy to
latency or memory.

## Model-pool sentence replacement

The evaluated pool contains **{model_configuration_count} distinct
model and precision/quantization configurations**, spanning
{parameter_span} and including dense and mixture-of-experts architectures.
Because model assignment follows deployment feasibility, these
configurations correspond to **{context_deployment_count} context-specific
deployments** rather than a fully crossed model-by-platform design.

## GPU-results sentence replacement

On the GPU server, the model pool expands to
**{gpu_configuration_count} configurations**, ranging from compact models
to approximately {parameter_max_b:.0f}B parameters and including dense and
mixture-of-experts architectures.

## Conclusion sentence replacement

We evaluated **{model_configuration_count} distinct model and
precision/quantization configurations**, instantiated across
**{context_deployment_count} context-specific deployments**, using a
reproducible measurement instrument that combines task-level accuracy with
inline system-level metrics collected during streaming inference.

## Table 7 caption replacement

Short-output cost-effectiveness indicators on MMLU. Expected correct
responses per second is computed as benchmark accuracy [0, 1] multiplied by
measured sample throughput under sequential inference.

## Table 7 interpretation replacement

Table 7 reports a short-output cost-effectiveness indicator defined as
expected correct responses per second:

`expected correct responses/s = accuracy × samples/s`.

The indicator estimates the rate at which a configuration produces correct
benchmark responses under the measured sequential-request protocol. It is
most interpretable for short, similarly structured outputs such as MMLU.
It should not be treated as a general measure of reasoning efficiency,
because output length, parsing requirements, and workload structure differ
across datasets.
"""

(TEXT_DIR / "manuscript_statement_replacements.md").write_text(
    statement_replacements,
    encoding="utf-8",
)


print((TEXT_DIR / "results_section_data_draft.md").read_text(encoding="utf-8")[:5000])


## 15. Captions, document map, and consistency audit

In [ ]:

captions = OrderedDict({
    "Figure 1": (
        "Overall architecture of the benchmarking framework. Experiments are defined by "
        "configuration files and executed through a Python orchestration layer. Platform-specific "
        "vLLM and llama.cpp backends expose an OpenAI-compatible streaming API. Task-level and "
        "inline system-level metrics are recorded and exported as traceable structured artifacts."
    ),
    "Figure 2": (
        "Color-blind-safe grouped bars report peak system RAM on the Raspberry Pi 5 for each feasible model "
        "configuration and workload. Models are ordered by increasing Dolly peak RAM, and workload bars "
        "are alphabetical. The 8 GiB reference line makes the edge-context ceiling explicit; Dolly uses "
        "the maximum peak across its split acquisitions."
    ),
    "Figure 2a1": (
        "Grouped bars report mean GPU energy per request for GPU model configurations on Dolly and GSM8K only. "
        "The split avoids mixing long-generation and short-answer regimes on the same joule scale. Request energy "
        "is obtained by trapezoidal integration of recorded GPU power over request time, with audited fallbacks. "
        "Models are ordered by increasing Dolly mean energy."
    ),
    "Figure 2a2": (
        "Grouped bars report mean GPU energy per request for GPU model configurations on HellaSwag and MMLU only. "
        "These short-answer workloads are separated from Dolly and GSM8K so the joule scale remains interpretable. "
        "Figure 2a2 keeps the same model order as Figure 2a1."
    ),
    "Figure 3": (
        "Color-blind-safe grouped bars report peak system RAM on the local CPU platform for each model "
        "configuration and workload. Models are ordered by increasing Dolly peak RAM, workload bars are "
        "alphabetical, and Dolly uses the maximum peak across its split acquisitions."
    ),
    "Figure 3a": (
        "Absolute GPU VRAM deployment footprint by model configuration and workload. Markers show "
        "the median recorded VRAM, thick vertical intervals show the 5th–95th percentiles, and thin "
        "vertical intervals show the observed minimum–maximum range. This representation remains "
        "readable when the interquartile range collapses. Models are ordered by increasing Dolly "
        "median VRAM; trace measurements are descriptive repeated observations within requests."
    ),
    "Figure 3b1": (
        "Grouped boxplots report the distribution of GPU energy per request for Dolly and GSM8K only. "
        "Separating these workloads avoids compressing the higher-energy regime into the short-answer regime. "
        "Figure 3b1 keeps the same increasing Dolly mean-energy model order as Figure 2a1."
    ),
    "Figure 3b2": (
        "Grouped boxplots report the distribution of GPU energy per request for HellaSwag and MMLU only. "
        "These short-answer workloads use a separate joule scale and the same model order as Figure 2a2."
    ),
    "Figure 4": (
        "Accuracy [0, 1] by model and benchmark, separated into GSM8K, HellaSwag, and MMLU "
        "panels. Accuracy is shown on the y-axis, and the complete model labels are repeated "
        "beneath every panel. Edge, Local CPU, and GPU-server configurations occupy fixed "
        "context blocks and follow the shared GSM8K-based order used in Figure 5."
    ),
    "Figure 5": (
        "Reported token throughput by model and workload, separated into Dolly, GSM8K, "
        "HellaSwag, and MMLU panels. The complete model labels are repeated beneath every "
        "panel. The shared logarithmic y-axis exposes the observed order-of-magnitude range, "
        "and context blocks, model order, and x-coordinate positions are identical to Figure 4. "
        "Edge and Local CPU report active-generation tokens/s, whereas the GPU server reports "
        "run-level sustained tokens/s; boxed values quantify the fastest-to-slowest spread "
        "within each benchmark and context."
    ),
    "Figure 6": (
        "GSM8K accuracy [0, 1]–latency Pareto frontiers within each deployment context. Every "
        "Pareto-relevant configuration is labeled; the GPU panel uses a dedicated right-side label column so annotations do not obscure the data."
    ),
    "Figure 7": (
        "GSM8K accuracy [0, 1]–memory Pareto frontiers within each deployment context. Every "
        "Pareto-relevant configuration is labeled. The GPU panel uses a dedicated right-side label column. Memory denotes peak system RAM for edge/local CPU and peak VRAM for the GPU server."
    ),
})

caption_text = "# Manuscript figure captions\n\n" + "\n\n".join(
    f"**{name}.** {caption}" for name, caption in captions.items()
)
(TEXT_DIR / "manuscript_figure_captions.md").write_text(caption_text + "\n", encoding="utf-8")


count_warning = (
    f"Use {model_configuration_count} distinct model/precision configurations "
    f"and {context_deployment_count} context-specific deployments. "
    "The fixed phrase 'nineteen configurations' has been replaced in the "
    "generated manuscript-support text with these audited counting units."
)

audit_items = [
    ("Model-count statement", count_warning, "CORRECTED"),
    (
        "Table 7 metric definition",
        "Do not describe the indicator as accuracy divided by token throughput. This notebook "
        "defines expected correct responses per second as accuracy × measured samples/s. "
        "The metric is only suitable as a compact short-output sequential-workload indicator.",
        "CORRECTED",
    ),
    (
        "Cross-context Pareto language",
        "GPU points may dominate CPU/edge points globally, but those comparisons are not actionable "
        "when an accelerator is unavailable. Main claims and Figures 6–7 use context-local frontiers.",
        "CORRECTED",
    ),
    (
        "Memory terminology",
        "Use system RAM for edge/CPU and VRAM for GPU. Do not plot these as an unqualified common "
        "memory measure without identifying the resource type.",
        "CORRECTED",
    ),
    (
        "Memory-figure model ordering",
        "Figures 2 and 3 use increasing Dolly peak system RAM. Figure 3a uses grouped box plots of all recorded VRAM trace measurements, with model order based on increasing Dolly mean VRAM measurement. Figures 4 and 5 retain a separate "
        "shared GSM8K-accuracy order. The manuscript must not claim one common order across all figures.",
        "CORRECTED",
    ),
    (
        "SLM/LLM metadata",
        "Use total parameters for catalog/model-size classification, active parameters for MoE "
        "compute interpretation, and measured RAM/VRAM for deployment footprint.",
        "CORRECTED",
    ),
    (
        "Phi-4 Mini metadata",
        "All Phi-4 Mini variants are registered as 3.8B dense models and are no longer excluded "
        "from compact-model evidence due to missing filename parsing.",
        "CORRECTED",
    ),
    (
        "Qwen 2.5 1.5B metadata",
        "qwen2_5_1_5b_bf16 is registered as 1.54B, not 5B.",
        "CORRECTED",
    ),
    (
        "MoE metadata",
        "Granite H-Tiny, Qwen3-30B-A3B, and Kimi Linear include both total and active parameter counts.",
        "CORRECTED",
    ),
    (
        "GPU VRAM interpretation",
        "Figure 3a reports absolute deployment footprint using median VRAM with 5th–95th percentile "
        "and min–max intervals. Repeated trace points are descriptive, not independent observations. "
        "The supplementary run-min-adjusted figure is exploratory because the baseline is the minimum "
        "observed within a model run, not a separately measured idle baseline.",
        "CORRECTED",
    ),
    (
        "GPU energy definition",
        "Figures 2a and 3b use one energy value per request. The preferred value is the trapezoidal "
        "integral of recorded GPU power over trace time. Sample energy_j or average power × latency "
        "is used only as an audited fallback. Energy must not be described as instantaneous power.",
        "CORRECTED",
    ),
    (
        "Accuracy uncertainty",
        "Tables and prose report means. Add standard deviations or confidence intervals where space "
        "allows, especially for small differences and reduced 100-sample subsets.",
        "REQUIRES CHECK",
    ),
    (
        "Dolly split acquisitions",
        "For affected Edge/CPU configurations, Dolly samples collected on "
        "different dates are pooled at sample level. Acquisition sizes are "
        "preserved in checks/dolly_split_acquisition_audit.csv. A 100-sample "
        "batch and a 900-sample batch are not treated as equal observations. "
        "If the 900 samples were generated under one server session, Dolly "
        "must not be described as ten independent 100-sample runs.",
        "CORRECTED",
    ),
    (
        "Throughput denominator by context",
        "Figure 5 and Table 5 use active-generation tokens/s for Edge and Local CPU, "
        "and run-level sustained tokens/s for the GPU server. The panels state the "
        "basis explicitly, and checks/throughput_metric_audit.csv preserves both raw "
        "columns and the selected value.",
        "CORRECTED",
    ),
    (
        "Pi throughput caveat",
        "The one-second inter-request pause is excluded from active-generation throughput. "
        "Do not describe Raspberry Pi values as continuous serving throughput.",
        "CORRECTED",
    ),
    (
        "Backend and quantization confound",
        "vLLM/bf16-fp8-AWQ on GPU and llama.cpp/GGUF on CPU-edge are deployment-realistic but not a "
        "controlled backend comparison. Keep this limitation explicit.",
        "REQUIRES CHECK",
    ),
]

audit_df = pd.DataFrame(audit_items, columns=["Item", "Required action", "Status"])
audit_df.to_csv(CHECK_DIR / "manuscript_consistency_audit.csv", index=False)
audit_md = "# Manuscript consistency audit\n\n" + markdown_table(audit_df) + "\n"
(CHECK_DIR / "manuscript_consistency_audit.md").write_text(audit_md, encoding="utf-8")


document_map = """# Manuscript evidence-pack map

## Main manuscript figures
- `figures/figure_01_framework_architecture.{png,pdf,svg}`
- `figures/figure_02_pi5_peak_system_ram.{png,pdf,svg}`
- `figures/figure_02a1_gpu_mean_energy_per_request_dolly_gsm8k.{png,pdf,svg}`
- `figures/figure_02a2_gpu_mean_energy_per_request_hellaswag_mmlu.{png,pdf,svg}`
- `figures/figure_03_cpu_peak_system_ram.{png,pdf,svg}`
- `figures/figure_03a_gpu_vram_footprint_intervals.{png,pdf,svg}`
- `figures/figure_03b1_gpu_energy_per_request_boxplots_dolly_gsm8k.{png,pdf,svg}`
- `figures/figure_03b2_gpu_energy_per_request_boxplots_hellaswag_mmlu.{png,pdf,svg}`
- `figures/figure_04_accuracy_by_context_and_dataset.{png,pdf,svg}`
- `figures/figure_05_sustained_throughput_across_contexts.{png,pdf,svg}`
- `figures/figure_06_gsm8k_accuracy_latency_frontiers.{png,pdf,svg}`
- `figures/figure_07_gsm8k_accuracy_memory_frontiers.{png,pdf,svg}`

## Figure-selection data
- `tables/figure_02_pi5_peak_system_ram_values.csv`
- `tables/figure_02a1_gpu_mean_energy_per_request_dolly_gsm8k_values.csv`
- `tables/figure_02a2_gpu_mean_energy_per_request_hellaswag_mmlu_values.csv`
- `tables/gpu_request_energy_statistics.csv`
- `tables/figure_03_cpu_peak_system_ram_values.csv`
- `tables/figure_03a_gpu_vram_footprint_intervals_statistics.csv`
- `tables/figure_03b1_gpu_energy_per_request_boxplots_dolly_gsm8k_measurement_values.csv`
- `tables/figure_03b2_gpu_energy_per_request_boxplots_hellaswag_mmlu_measurement_values.csv`
- `tables/figure_s03a_gpu_run_min_adjusted_vram_boxplots_measurement_values.csv`
- `tables/figure_04_accuracy_by_benchmark_long.csv`
- `tables/figure_04_accuracy_by_benchmark_wide.csv`
- `tables/figure_04_accuracy_panels_long.csv` (compatibility alias)
- `tables/figure_04_accuracy_panels_wide.csv` (compatibility alias)
- `tables/figure_04_accuracy_by_context_gsm8k_order.csv` (compatibility alias)
- `tables/figure_05_throughput_panels_long.csv`
- `tables/figure_05_throughput_panels_wide.csv`
- `tables/figure_05_throughput_spread_by_panel.csv`
- `tables/figure_05_throughput_by_benchmark.csv` (compatibility alias)
- `tables/figure_05_throughput_spread_by_benchmark.csv` (compatibility alias)

## Main manuscript tables
Each table is exported as CSV, Markdown, and LaTeX.
- Table 1: deployment contexts
- Table 2: evaluated model configurations and corrected metadata
- Table 3: hardware platforms
- Table 4: selected task-level accuracy
- Table 5: selected timing and throughput
- Table 6: selected peak resources
- Table 7: explicitly defined cost-effectiveness indicator

## Text-support documents
- `text/manuscript_figure_captions.md`
- `text/results_section_data_draft.md`
- `text/abstract_evidence_block.md`
- `text/canonical_results_facts.csv`
- `text/manuscript_statement_replacements.md`
- `text/dolly_split_acquisition_protocol.md`
- `text/figure_05_throughput_interpretation.md`


## Figure accessibility
- Figures 2, 3, 3a, 4, and 5 use a color-blind-friendly palette with hatch patterns and dark outlines.
- Figures 6--7 use open versus filled markers, different marker shapes, connecting lines, and direct labels.
- The scientific interpretation therefore remains available in grayscale and never depends on hue alone.

## Checks
- `checks/model_metadata_audit.csv`
- `checks/shared_model_order_gsm8k.csv`
- `checks/figure_04_05_shared_model_positions.csv`
- `checks/figure_04_export_schema.csv`
- `checks/figure_05_export_schema.csv`
- `checks/figure_02_pi5_peak_system_ram_benchmark_order.csv`
- `checks/figure_03_cpu_peak_system_ram_benchmark_order.csv`
- `checks/figure_02_pi5_peak_system_ram_model_order_dolly.csv`
- `checks/figure_02a1_gpu_mean_energy_per_request_dolly_gsm8k_model_order_dolly.csv`
- `checks/figure_02a1_gpu_mean_energy_per_request_dolly_gsm8k_benchmark_order.csv`
- `checks/figure_02a2_gpu_mean_energy_per_request_hellaswag_mmlu_model_order_dolly.csv`
- `checks/figure_02a2_gpu_mean_energy_per_request_hellaswag_mmlu_benchmark_order.csv`
- `checks/figure_03b1_gpu_energy_per_request_boxplots_dolly_gsm8k_model_order_dolly.csv`
- `checks/figure_03b1_gpu_energy_per_request_boxplots_dolly_gsm8k_benchmark_order.csv`
- `checks/figure_03b2_gpu_energy_per_request_boxplots_hellaswag_mmlu_model_order_dolly.csv`
- `checks/figure_03b2_gpu_energy_per_request_boxplots_hellaswag_mmlu_benchmark_order.csv`
- `checks/figure_03_cpu_peak_system_ram_model_order_dolly.csv`
- `checks/figure_03a_gpu_vram_footprint_intervals_model_order_dolly.csv`
- `checks/figure_03a_gpu_vram_distribution_diagnostics.csv`
- `checks/gpu_request_energy_source_audit.csv`
- `checks/gpu_request_energy_source_counts.csv`
- `checks/gpu_request_energy_request_audit.csv`
- `checks/figure_s03a_gpu_run_min_baseline_audit.csv`
- `checks/figure_03a_gpu_vram_measurement_source_audit.csv`
- `checks/figure_03a_gpu_vram_measurement_source_counts.csv`
- `checks/figure_03a_gpu_vram_measurement_trace_audit.csv`
- `checks/normalized_summary_source_audit.csv`
- `checks/throughput_metric_audit.csv`
- `checks/dolly_split_acquisition_audit.csv`
- `checks/dolly_split_pooling_audit.csv`
- `checks/dolly_processed_peak_audit.csv`
- `checks/throughput_missingness_by_context.csv`
- `checks/manuscript_consistency_audit.{csv,md}`
- `checks/output_manifest.csv`

## Supplement
- `figures/figure_s03a_gpu_run_min_adjusted_vram_boxplots.{png,pdf,svg}`
- context-local compact-model delta figures;
- context-local delta CSV;
- competitive compact-model evidence CSV;
- GSM8K Pareto-membership CSV.
"""
(TEXT_DIR / "document_map.md").write_text(document_map, encoding="utf-8")



# Dolly split-acquisition protocol note.
if not dolly_split_pooling_audit.empty:
    split_rows = dolly_split_pooling_audit[
        dolly_split_pooling_audit["sample_count"].gt(0)
    ].copy()

    protocol_lines = [
        "# Dolly split-acquisition protocol note",
        "",
        "For some configurations, Dolly was collected in more than one "
        "acquisition batch on different dates. The analysis concatenates "
        "the underlying sample records and computes sample-level means. "
        "Consequently, a batch of 100 samples and a batch of 900 samples "
        "contribute in proportion to their sample counts; their batch means "
        "are not averaged with equal weight.",
        "",
        "This pooling is valid only because the model, hardware, backend, "
        "generation settings, prompt construction, and measurement procedure "
        "are treated as protocol-equivalent across the acquisition dates.",
        "",
        "The acquisition structure observed by the notebook is:",
        "",
        markdown_table(
            split_rows[
                [
                    "platform",
                    "model",
                    "sample_count",
                    "run_count",
                    "acquisition_date_count",
                    "acquisition_dates",
                    "run_sample_counts",
                    "protocol_structure",
                    "supports_ten_independent_run_claim",
                ]
            ],
            float_format=".3f",
        ),
        "",
        "Important: a single 900-sample server session is not statistically "
        "equivalent to nine independent 100-sample runs. The manuscript's "
        "ten-independent-run statement should therefore be restricted to "
        "workloads/configurations whose audit confirms ten distinct "
        "100-sample run identifiers.",
    ]
    (TEXT_DIR / "dolly_split_acquisition_protocol.md").write_text(
        "\n".join(protocol_lines) + "\n",
        encoding="utf-8",
    )


display(audit_df)
print(caption_text)


## 16. Deferred Figure 3a definitions

The following cells contain definitions only. They should execute quickly because they do not read trace files or create plots.


In [ ]:
def gpu_trace_source_candidates() -> dict[str, list[Path]]:
    """Locate all-measurement GPU trace sources in fallback priority."""
    preferred_dir = (
        ROOT / "results" / "processed" / "vllm"
    )
    results_root = ROOT / "results"

    candidates = {
        "system_traces": [
            preferred_dir / "system_traces.csv",
        ],
        "samples": [
            PLATFORM_PROCESSED_SAMPLE_PATHS["vllm"],
            preferred_dir / "samples.csv",
        ],
    }

    if results_root.exists():
        candidates["system_traces"].extend(
            sorted(
                results_root.rglob(
                    "system_traces.csv"
                )
            )
        )
        candidates["samples"].extend(
            sorted(results_root.rglob("samples.csv"))
        )

    for key, paths in candidates.items():
        unique_paths = []
        seen = set()

        for path in paths:
            path = Path(path)
            if not path.exists():
                continue

            resolved = str(path.resolve())
            if resolved in seen:
                continue

            seen.add(resolved)
            unique_paths.append(path)

        candidates[key] = unique_paths

    return candidates


def is_gpu_trace_frame(
    frame: pd.DataFrame,
    path: Path,
) -> bool:
    """Reject processed files belonging to non-GPU platforms."""
    if frame.empty:
        return False

    if "platform" in frame.columns:
        platforms = {
            normalize_platform_token(value)
            for value in frame["platform"]
            .dropna()
            .unique()
        }
        return not platforms or "vllm" in platforms

    return "vllm" in str(path).lower()


def ensure_gpu_trace_model_column(
    frame: pd.DataFrame,
    path: Path,
) -> pd.DataFrame:
    """Create the raw model identifier expected by resolve_metadata()."""
    out = frame.copy()

    if "model" in out.columns:
        return out

    candidates = [
        column
        for column in [
            "model_name",
            "model_id",
            "model_label",
            "model_alias_dir",
            "served_model_id",
            "repo_id",
            "model_path",
        ]
        if column in out.columns
    ]

    if not candidates:
        raise KeyError(
            f"{path} has no usable model identifier column."
        )

    out["model"] = (
        out[candidates]
        .bfill(axis=1)
        .iloc[:, 0]
    )
    return out


def trace_rows_from_system_traces(
    path: Path,
) -> pd.DataFrame:
    """Load every recorded GPU VRAM observation from system_traces.csv."""
    traces = pd.read_csv(path)

    if not is_gpu_trace_frame(traces, path):
        return pd.DataFrame()

    required = {
        "dataset",
        "vram_used_mb",
    }
    if not required.issubset(traces.columns):
        return pd.DataFrame()

    traces = ensure_gpu_trace_model_column(
        traces,
        path,
    )
    traces["dataset"] = traces["dataset"].map(
        normalize_dataset
    )
    traces["vram_used_mb"] = pd.to_numeric(
        traces["vram_used_mb"],
        errors="coerce",
    )
    traces = traces[
        traces["dataset"].isin(
            BENCHMARK_ORDER_ALPHABETICAL
        )
        & traces["vram_used_mb"].notna()
    ].copy()

    if traces.empty:
        return traces

    if "trace_idx" not in traces.columns:
        traces["trace_idx"] = (
            traces.groupby(
                [
                    column
                    for column in [
                        "run_id",
                        "sample_idx",
                        "request_id",
                        "sample_id",
                    ]
                    if column in traces.columns
                ],
                dropna=False,
            )
            .cumcount()
        )

    traces["trace_source"] = (
        "processed system_traces.csv"
    )
    traces["source_priority"] = 0
    traces["source_file"] = str(path)

    return traces


def trace_rows_from_samples_json(
    path: Path,
) -> pd.DataFrame:
    """Expand every vram_used_mb point from sample trace JSON."""
    samples = pd.read_csv(path)

    if not is_gpu_trace_frame(samples, path):
        return pd.DataFrame()

    if (
        "dataset" not in samples.columns
        or "system_metrics_trace_json"
        not in samples.columns
    ):
        return pd.DataFrame()

    samples = ensure_gpu_trace_model_column(
        samples,
        path,
    )
    samples["dataset"] = samples["dataset"].map(
        normalize_dataset
    )
    samples = samples[
        samples["dataset"].isin(
            BENCHMARK_ORDER_ALPHABETICAL
        )
    ].copy()

    expanded_rows = []

    preserved_columns = [
        column
        for column in [
            "platform",
            "dataset",
            "model",
            "model_label",
            "model_alias_dir",
            "repo_id",
            "served_model_id",
            "run_id",
            "sample_idx",
            "request_id",
            "sample_id",
            "prompt_id",
        ]
        if column in samples.columns
    ]

    for source_row_index, row in samples.iterrows():
        raw_trace = row.get(
            "system_metrics_trace_json"
        )

        if raw_trace is None:
            continue

        try:
            if pd.isna(raw_trace):
                continue
        except Exception:
            pass

        if isinstance(raw_trace, str):
            if not raw_trace.strip():
                continue
            try:
                trace_points = json.loads(raw_trace)
            except Exception:
                continue
        elif isinstance(raw_trace, list):
            trace_points = raw_trace
        else:
            continue

        if not isinstance(trace_points, list):
            continue

        base = {
            column: row.get(column)
            for column in preserved_columns
        }

        for trace_idx, point in enumerate(
            trace_points
        ):
            if not isinstance(point, dict):
                continue

            vram_value = pd.to_numeric(
                point.get("vram_used_mb"),
                errors="coerce",
            )
            if pd.isna(vram_value):
                continue

            expanded_rows.append({
                **base,
                "trace_idx": trace_idx,
                "t_s": pd.to_numeric(
                    point.get("t_s"),
                    errors="coerce",
                ),
                "vram_used_mb": float(vram_value),
                "source_row_index": source_row_index,
                "trace_source": (
                    "samples.csv system_metrics_trace_json"
                ),
                "source_priority": 1,
                "source_file": str(path),
            })

    return pd.DataFrame(expanded_rows)


def create_trace_identity(
    frame: pd.DataFrame,
) -> pd.Series:
    """Create a stable identity for cross-source trace deduplication."""
    identity_columns = [
        column
        for column in [
            "run_id",
            "sample_idx",
            "request_id",
            "sample_id",
            "prompt_id",
            "trace_idx",
            "t_s",
        ]
        if column in frame.columns
    ]

    if not identity_columns:
        return pd.Series(
            [
                f"row::{index}"
                for index in frame.index
            ],
            index=frame.index,
            dtype=object,
        )

    identity_frame = frame[
        identity_columns
    ].astype(str)

    return identity_frame.apply(
        lambda row: "::".join(
            f"{column}={row[column]}"
            for column in identity_columns
        ),
        axis=1,
    )


In [ ]:
def build_gpu_vram_measurements(
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load every recorded GPU VRAM measurement without peak reduction."""
    candidates = gpu_trace_source_candidates()
    trace_parts = []
    source_attempt_rows = []

    loaders = [
        (
            "system_traces.csv",
            candidates["system_traces"],
            trace_rows_from_system_traces,
        ),
        (
            "samples trace JSON",
            candidates["samples"],
            trace_rows_from_samples_json,
        ),
    ]

    selected_source_kind = None
    selected_source_path = None

    # Strict fallback: stop after the first usable source. In a normal run,
    # processed system_traces.csv is authoritative and the much slower JSON
    # expansion is never attempted.
    for source_kind, paths, loader in loaders:
        for path in paths:
            try:
                rows = loader(path)
                usable_count = (
                    int(
                        pd.to_numeric(
                            rows.get(
                                "vram_used_mb",
                                pd.Series(dtype=float),
                            ),
                            errors="coerce",
                        )
                        .notna()
                        .sum()
                    )
                    if not rows.empty
                    else 0
                )

                selected = usable_count > 0
                source_attempt_rows.append({
                    "source_kind": source_kind,
                    "path": str(path),
                    "status": (
                        "selected"
                        if selected
                        else "no usable VRAM measurements"
                    ),
                    "usable_vram_measurements": usable_count,
                    "selected_for_figure": selected,
                })

                if selected:
                    trace_parts = [rows]
                    selected_source_kind = source_kind
                    selected_source_path = str(path)
                    break

            except Exception as exc:
                source_attempt_rows.append({
                    "source_kind": source_kind,
                    "path": str(path),
                    "status": f"error: {exc}",
                    "usable_vram_measurements": 0,
                    "selected_for_figure": False,
                })

        if trace_parts:
            break

    source_audit = pd.DataFrame(
        source_attempt_rows
    )
    source_audit.to_csv(
        CHECK_DIR
        / "figure_03a_gpu_vram_measurement_source_audit.csv",
        index=False,
    )

    if not trace_parts:
        raise FileNotFoundError(
            "Figure 3a could not find recorded GPU VRAM trace "
            "measurements. Run the GPU analysis notebook so that "
            "results/processed/vllm/system_traces.csv is exported, "
            "or retain system_metrics_trace_json in samples.csv."
        )

    traces = pd.concat(
        trace_parts,
        ignore_index=True,
        sort=False,
    )
    traces["dataset"] = traces["dataset"].map(
        normalize_dataset
    )
    traces["vram_used_mb"] = pd.to_numeric(
        traces["vram_used_mb"],
        errors="coerce",
    )
    traces = traces[
        traces["dataset"].isin(
            BENCHMARK_ORDER_ALPHABETICAL
        )
        & traces["vram_used_mb"].notna()
    ].copy()

    metadata = traces.apply(
        resolve_metadata,
        axis=1,
    )
    traces = pd.concat(
        [
            traces.reset_index(drop=True),
            metadata.reset_index(drop=True),
        ],
        axis=1,
    )

    traces["trace_identity"] = (
        create_trace_identity(traces)
    )

    # The same trace may appear in system_traces.csv and samples.csv.
    # Prefer the processed trace table and retain one copy of each point.
    traces = traces.sort_values(
        [
            "source_priority",
            "source_file",
        ]
    )
    before_deduplication = len(traces)
    traces = traces.drop_duplicates(
        subset=[
            "model_display_canonical",
            "dataset",
            "trace_identity",
        ],
        keep="first",
    )
    duplicates_removed = (
        before_deduplication - len(traces)
    )

    valid = traces[
        traces["model_display_canonical"].notna()
        & traces["dataset"].notna()
        & traces["vram_used_mb"].notna()
    ].copy()

    if valid.empty:
        raise ValueError(
            "No valid GPU VRAM trace measurements remain "
            "after normalization."
        )

    expected_models = set(
        context_subset("gpu_server")[
            "model_display_canonical"
        ]
        .dropna()
        .astype(str)
    )
    observed_models = set(
        valid["model_display_canonical"]
        .dropna()
        .astype(str)
    )
    missing_models = sorted(
        expected_models - observed_models
    )

    if missing_models:
        raise ValueError(
            "Figure 3a would omit GPU configurations because no "
            "VRAM trace measurements were recovered for: "
            + "; ".join(missing_models)
        )

    valid["vram_used_gib"] = (
        valid["vram_used_mb"] / 1024.0
    )
    valid["duplicates_removed_across_sources"] = (
        duplicates_removed
    )

    statistics = (
        valid.groupby(
            [
                "model_display_canonical",
                "dataset",
            ],
            dropna=False,
        )["vram_used_gib"]
        .agg(
            measurement_count="count",
            unique_value_count="nunique",
            mean_vram_gib="mean",
            median_vram_gib="median",
            sd_vram_gib="std",
            minimum_vram_gib="min",
            p05_vram_gib=lambda values: (
                values.quantile(0.05)
            ),
            q1_vram_gib=lambda values: (
                values.quantile(0.25)
            ),
            q3_vram_gib=lambda values: (
                values.quantile(0.75)
            ),
            p95_vram_gib=lambda values: (
                values.quantile(0.95)
            ),
            maximum_vram_gib="max",
        )
        .reset_index()
    )
    statistics["iqr_vram_gib"] = (
        statistics["q3_vram_gib"]
        - statistics["q1_vram_gib"]
    )
    statistics["range_vram_gib"] = (
        statistics["maximum_vram_gib"]
        - statistics["minimum_vram_gib"]
    )
    statistics["collapsed_iqr"] = (
        statistics["iqr_vram_gib"].abs()
        <= 1e-9
    )
    statistics["aggregation_definition"] = (
        "all recorded vram_used_mb trace measurements"
    )
    statistics[
        "duplicates_removed_across_sources"
    ] = duplicates_removed
    statistics["selected_trace_source_kind"] = (
        selected_source_kind
    )
    statistics["selected_trace_source_path"] = (
        selected_source_path
    )

    source_counts = (
        valid.groupby(
            [
                "model_display_canonical",
                "dataset",
                "trace_source",
            ],
            dropna=False,
        )
        .size()
        .rename("measurement_count")
        .reset_index()
    )
    source_counts.to_csv(
        CHECK_DIR
        / "figure_03a_gpu_vram_measurement_source_counts.csv",
        index=False,
    )

    return valid, statistics


In [ ]:

def build_gpu_vram_model_order(
    statistics: pd.DataFrame,
) -> pd.DataFrame:
    """Order GPU models by increasing Dolly median absolute VRAM."""
    dolly = statistics[
        statistics["dataset"].eq("dolly")
    ][
        [
            "model_display_canonical",
            "median_vram_gib",
        ]
    ].copy()

    if dolly.empty:
        raise ValueError(
            "Figure 3a cannot use Dolly ordering because no "
            "Dolly VRAM measurements are available."
        )

    order_table = dolly.rename(
        columns={
            "model_display_canonical":
                "model_configuration",
            "median_vram_gib":
                "dolly_median_vram_gib",
        }
    )
    order_table["dolly_measurement_missing"] = (
        order_table["dolly_median_vram_gib"].isna()
    )
    order_table = (
        order_table.sort_values(
            [
                "dolly_measurement_missing",
                "dolly_median_vram_gib",
                "model_configuration",
            ],
            ascending=[True, True, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )
    order_table["model_order_position"] = (
        np.arange(len(order_table)) + 1
    )
    order_table["model_order_basis"] = (
        "increasing Dolly median absolute VRAM"
    )
    return order_table


def plot_gpu_vram_footprint_intervals(
    statistics: pd.DataFrame,
    stem: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Plot median VRAM with central-90% and full observed ranges."""
    available_benchmarks = [
        dataset
        for dataset in BENCHMARK_ORDER_ALPHABETICAL
        if dataset in statistics["dataset"].unique()
    ]
    order_table = build_gpu_vram_model_order(
        statistics
    )
    model_order = order_table[
        "model_configuration"
    ].tolist()

    model_count = len(model_order)
    workload_count = max(
        len(available_benchmarks),
        1,
    )
    x_positions = np.arange(model_count)
    group_width = 0.78
    offset_width = group_width / workload_count

    marker_map = {
        "dolly": "o",
        "gsm8k": "s",
        "hellaswag": "^",
        "mmlu": "D",
    }

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 4.75),
        constrained_layout=True,
    )

    plotted_rows = []

    for workload_index, dataset in enumerate(
        available_benchmarks
    ):
        offset = (
            -group_width / 2
            + offset_width / 2
            + workload_index * offset_width
        )
        dataset_rows = (
            statistics[
                statistics["dataset"].eq(dataset)
            ]
            .set_index("model_display_canonical")
            .reindex(model_order)
        )
        positions = x_positions + offset
        valid_mask = dataset_rows[
            "median_vram_gib"
        ].notna().to_numpy()

        if not valid_mask.any():
            continue

        x_valid = positions[valid_mask]
        rows_valid = dataset_rows.loc[
            dataset_rows["median_vram_gib"].notna()
        ].copy()

        ax.vlines(
            x_valid,
            rows_valid["minimum_vram_gib"],
            rows_valid["maximum_vram_gib"],
            color=DATASET_COLORS[dataset],
            linewidth=0.70,
            alpha=0.72,
            zorder=2,
        )
        ax.vlines(
            x_valid,
            rows_valid["p05_vram_gib"],
            rows_valid["p95_vram_gib"],
            color=DATASET_COLORS[dataset],
            linewidth=2.35,
            alpha=0.96,
            zorder=3,
        )
        ax.scatter(
            x_valid,
            rows_valid["median_vram_gib"],
            marker=marker_map[dataset],
            s=18,
            facecolor=DATASET_COLORS[dataset],
            edgecolor="0.18",
            linewidth=0.48,
            label=DATASET_DISPLAY[dataset],
            zorder=4,
        )

        for model, x_value, (_, row) in zip(
            rows_valid.index,
            x_valid,
            rows_valid.iterrows(),
        ):
            plotted_rows.append({
                "model_configuration": model,
                "dataset": dataset,
                "x_position": x_value,
                "measurement_count": row["measurement_count"],
                "unique_value_count": row["unique_value_count"],
                "minimum_vram_gib": row["minimum_vram_gib"],
                "p05_vram_gib": row["p05_vram_gib"],
                "median_vram_gib": row["median_vram_gib"],
                "p95_vram_gib": row["p95_vram_gib"],
                "maximum_vram_gib": row["maximum_vram_gib"],
                "iqr_vram_gib": row["iqr_vram_gib"],
                "range_vram_gib": row["range_vram_gib"],
                "collapsed_iqr": row["collapsed_iqr"],
            })

    plotted = pd.DataFrame(plotted_rows)
    if plotted.empty:
        raise ValueError(
            "Figure 3a has no plottable VRAM footprint statistics."
        )

    y_min = float(plotted["minimum_vram_gib"].min())
    y_max = float(plotted["maximum_vram_gib"].max())
    y_range = max(y_max - y_min, 0.25)
    padding = max(y_range * 0.10, 0.15)

    ax.set_ylim(
        max(0.0, y_min - padding),
        y_max + padding,
    )
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel(
        "Absolute GPU VRAM footprint (GiB)"
    )
    ax.set_xlabel(
        "Model configuration "
        "(increasing Dolly median VRAM)"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=13,
                max_lines=3,
            )
            for model in model_order
        ],
        rotation=58,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15,
    )
    ax.text(
        0.995,
        0.02,
        (
            "Marker = median; thick line = 5th–95th percentile; "
            "thin line = observed min–max"
        ),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=5.50,
        color="0.35",
    )

    style_axis(ax, grid_axis="y")
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=4,
        frameon=False,
        columnspacing=0.90,
        handletextpad=0.35,
    )

    plotted = plotted.merge(
        order_table[
            [
                "model_configuration",
                "model_order_position",
                "model_order_basis",
            ]
        ],
        on="model_configuration",
        how="left",
    ).sort_values(
        [
            "model_order_position",
            "dataset",
        ]
    )
    plotted.to_csv(
        TABLE_DIR / f"{stem}_statistics.csv",
        index=False,
    )
    order_table.to_csv(
        CHECK_DIR / f"{stem}_model_order_dolly.csv",
        index=False,
    )

    diagnostics = plotted[
        [
            "model_configuration",
            "dataset",
            "measurement_count",
            "unique_value_count",
            "iqr_vram_gib",
            "range_vram_gib",
            "collapsed_iqr",
        ]
    ].copy()
    diagnostics["interpretation"] = np.where(
        diagnostics["collapsed_iqr"],
        (
            "IQR collapsed: median/range presentation is more "
            "informative than a conventional box."
        ),
        "Non-zero IQR.",
    )
    diagnostics.to_csv(
        CHECK_DIR
        / "figure_03a_gpu_vram_distribution_diagnostics.csv",
        index=False,
    )

    save_figure(fig, stem)
    return plotted, order_table


def build_run_min_adjusted_vram(
    measurements: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Subtract the minimum observed VRAM in each model-server run."""
    adjusted = measurements.copy()

    if "run_id" in adjusted.columns:
        run_tokens = (
            adjusted["run_id"]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        run_tokens = pd.Series(
            "",
            index=adjusted.index,
            dtype=object,
        )

    fallback_tokens = (
        adjusted["source_file"]
        .fillna("unknown_source")
        .astype(str)
        if "source_file" in adjusted.columns
        else pd.Series(
            "unknown_source",
            index=adjusted.index,
            dtype=object,
        )
    )

    adjusted["baseline_group_id"] = np.where(
        run_tokens.ne(""),
        run_tokens,
        fallback_tokens,
    )

    baseline_columns = [
        "model_display_canonical",
        "baseline_group_id",
    ]
    baseline = (
        adjusted.groupby(
            baseline_columns,
            dropna=False,
        )["vram_used_gib"]
        .min()
        .rename("run_minimum_vram_gib")
        .reset_index()
    )

    adjusted = adjusted.merge(
        baseline,
        on=baseline_columns,
        how="left",
        validate="many_to_one",
    )
    adjusted["run_min_adjusted_vram_gib"] = (
        adjusted["vram_used_gib"]
        - adjusted["run_minimum_vram_gib"]
    ).clip(lower=0)

    baseline_audit = (
        adjusted.groupby(
            [
                "model_display_canonical",
                "baseline_group_id",
            ],
            dropna=False,
        )
        .agg(
            run_minimum_vram_gib=(
                "run_minimum_vram_gib",
                "first",
            ),
            measurement_count=(
                "run_min_adjusted_vram_gib",
                "count",
            ),
            maximum_delta_vram_gib=(
                "run_min_adjusted_vram_gib",
                "max",
            ),
        )
        .reset_index()
    )
    baseline_audit["baseline_definition"] = (
        "minimum observed VRAM within model and run; "
        "exploratory, not a measured idle baseline"
    )
    return adjusted, baseline_audit


def plot_run_min_adjusted_vram_boxplots(
    adjusted: pd.DataFrame,
    order_table: pd.DataFrame,
    stem: str,
) -> pd.DataFrame:
    """Plot exploratory VRAM deltas above the model-run minimum."""
    available_benchmarks = [
        dataset
        for dataset in BENCHMARK_ORDER_ALPHABETICAL
        if dataset in adjusted["dataset"].unique()
    ]
    model_order = order_table[
        "model_configuration"
    ].tolist()

    model_count = len(model_order)
    workload_count = max(
        len(available_benchmarks),
        1,
    )
    x_positions = np.arange(model_count)
    group_width = 0.82
    box_width = group_width / workload_count

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 4.75),
        constrained_layout=True,
    )

    export_rows = []
    all_values = []

    for workload_index, dataset in enumerate(
        available_benchmarks
    ):
        offsets = (
            x_positions
            - group_width / 2
            + box_width / 2
            + workload_index * box_width
        )
        distributions = []
        positions = []

        for model_index, model in enumerate(model_order):
            values = (
                adjusted.loc[
                    adjusted[
                        "model_display_canonical"
                    ].eq(model)
                    & adjusted["dataset"].eq(dataset),
                    "run_min_adjusted_vram_gib",
                ]
                .dropna()
                .astype(float)
                .to_numpy()
            )
            if len(values) == 0:
                continue

            distributions.append(values)
            positions.append(float(offsets[model_index]))
            all_values.extend(values.tolist())

            export_rows.extend({
                "model_configuration": model,
                "dataset": dataset,
                "run_min_adjusted_vram_gib": value,
            } for value in values)

        if not distributions:
            continue

        boxplot = ax.boxplot(
            distributions,
            positions=positions,
            widths=box_width * 0.76,
            patch_artist=True,
            whis=1.5,
            showfliers=True,
            flierprops={
                "marker": ".",
                "markersize": 1.6,
                "markerfacecolor": "0.35",
                "markeredgecolor": "0.35",
                "alpha": 0.45,
            },
            manage_ticks=False,
            zorder=3,
        )

        for patch in boxplot["boxes"]:
            patch.set_facecolor(DATASET_COLORS[dataset])
            patch.set_edgecolor("0.20")
            patch.set_linewidth(0.60)
            patch.set_hatch(DATASET_HATCHES[dataset])
            patch.set_alpha(0.96)

        for element_name in [
            "whiskers",
            "caps",
            "medians",
        ]:
            for artist in boxplot[element_name]:
                artist.set_color("0.20")
                artist.set_linewidth(
                    0.60
                    if element_name != "medians"
                    else 0.95
                )

    if not all_values:
        raise ValueError(
            "No run-min-adjusted VRAM values are available."
        )

    value_max = float(np.nanmax(all_values))
    ax.set_ylim(0, max(value_max * 1.10, 0.10))
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel(
        "VRAM above model-run minimum (GiB)"
    )
    ax.set_xlabel(
        "Model configuration (same order as Figure 3a)"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=13,
                max_lines=3,
            )
            for model in model_order
        ],
        rotation=58,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15,
    )
    ax.text(
        0.995,
        0.98,
        (
            "Exploratory proxy: baseline = minimum observed VRAM "
            "within each model-server run"
        ),
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=5.50,
        color="0.35",
    )

    style_axis(ax, grid_axis="y")

    legend_handles = [
        Patch(
            facecolor=DATASET_COLORS[dataset],
            edgecolor="0.20",
            hatch=DATASET_HATCHES[dataset],
            linewidth=0.60,
            label=DATASET_DISPLAY[dataset],
        )
        for dataset in available_benchmarks
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=4,
        frameon=False,
        columnspacing=0.8,
        handletextpad=0.35,
    )

    export = pd.DataFrame(export_rows)
    export.to_csv(
        TABLE_DIR / f"{stem}_measurement_values.csv",
        index=False,
    )
    save_figure(fig, stem)
    return export


In [ ]:

GPU_SAMPLE_ENERGY_J_CANDIDATES = [
    "energy_j",
    "gpu_energy_j",
    "request_energy_j",
]

GPU_SAMPLE_POWER_W_CANDIDATES = [
    "power_w_avg",
    "gpu_power_w_avg",
    "mean_power_w",
]

GPU_SAMPLE_LATENCY_MS_CANDIDATES = [
    "latency_ms",
    "end_to_end_latency_ms",
]


def create_energy_request_identity(
    frame: pd.DataFrame,
    source_path: Path,
) -> pd.Series:
    """Create a stable request identity across trace and sample exports."""
    identity_columns = [
        column
        for column in [
            "run_id",
            "request_id",
            "sample_id",
            "sample_idx",
            "prompt_id",
        ]
        if column in frame.columns
    ]

    if not identity_columns:
        return pd.Series(
            [
                f"{source_path}::row::{index}"
                for index in frame.index
            ],
            index=frame.index,
            dtype=object,
        )

    identity_frame = frame[
        identity_columns
    ].astype(str)

    return identity_frame.apply(
        lambda row: "::".join(
            f"{column}={row[column]}"
            for column in identity_columns
        ),
        axis=1,
    )


def integrate_power_trace_joules(
    times,
    powers,
) -> tuple[float, str, int, float]:
    """Integrate GPU power over time; watts × seconds = joules."""
    frame = pd.DataFrame({
        "t_s": pd.to_numeric(
            pd.Series(times),
            errors="coerce",
        ),
        "power_w": pd.to_numeric(
            pd.Series(powers),
            errors="coerce",
        ),
    }).dropna()

    if frame.empty:
        return np.nan, "missing", 0, np.nan

    frame = (
        frame.groupby(
            "t_s",
            as_index=False,
        )["power_w"]
        .mean()
        .sort_values("t_s")
    )

    if len(frame) < 2:
        return (
            np.nan,
            "insufficient trace points",
            int(len(frame)),
            0.0,
        )

    t = frame["t_s"].to_numpy(dtype=float)
    p = frame["power_w"].to_numpy(dtype=float)
    duration_s = float(t[-1] - t[0])

    if duration_s <= 0:
        return (
            np.nan,
            "non-positive trace duration",
            int(len(frame)),
            duration_s,
        )

    intervals = np.diff(t)
    energy_j = float(
        (
            (
                p[:-1] + p[1:]
            ) / 2.0
            * intervals
        ).sum()
    )

    return (
        energy_j,
        "trapezoidal integral of power_w over t_s",
        int(len(frame)),
        duration_s,
    )


def energy_rows_from_system_traces(
    path: Path,
) -> pd.DataFrame:
    """Integrate one energy value per request from system_traces.csv."""
    traces = pd.read_csv(path)

    if not is_gpu_trace_frame(traces, path):
        return pd.DataFrame()

    required = {
        "dataset",
        "power_w",
        "t_s",
    }
    if not required.issubset(traces.columns):
        return pd.DataFrame()

    traces = ensure_gpu_trace_model_column(
        traces,
        path,
    )
    traces["dataset"] = traces["dataset"].map(
        normalize_dataset
    )
    traces = traces[
        traces["dataset"].isin(
            BENCHMARK_ORDER_ALPHABETICAL
        )
    ].copy()

    request_columns = [
        column
        for column in [
            "run_id",
            "request_id",
            "sample_id",
            "sample_idx",
            "prompt_id",
        ]
        if column in traces.columns
    ]

    if not request_columns:
        return pd.DataFrame()

    group_columns = [
        column
        for column in [
            "platform",
            "dataset",
            "model",
            "model_label",
            "model_alias_dir",
            "repo_id",
            "served_model_id",
            *request_columns,
        ]
        if column in traces.columns
    ]

    rows = []
    for group_key, group in traces.groupby(
        group_columns,
        dropna=False,
        sort=False,
    ):
        energy_j, method, point_count, duration_s = (
            integrate_power_trace_joules(
                group["t_s"],
                group["power_w"],
            )
        )

        if pd.isna(energy_j):
            continue

        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        row = dict(
            zip(
                group_columns,
                group_key,
            )
        )
        row.update({
            "energy_j": energy_j,
            "energy_source": method,
            "power_trace_point_count": point_count,
            "integrated_duration_s": duration_s,
            "source_priority": 0,
            "source_file": str(path),
        })
        rows.append(row)

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    result["request_identity"] = (
        create_energy_request_identity(
            result,
            path,
        )
    )
    return result


def parse_energy_from_sample_trace(
    raw_trace,
) -> tuple[float, str, int, float]:
    """Integrate energy from one sample's embedded system trace."""
    if raw_trace is None:
        return np.nan, "missing", 0, np.nan

    try:
        if pd.isna(raw_trace):
            return np.nan, "missing", 0, np.nan
    except Exception:
        pass

    if isinstance(raw_trace, str):
        if not raw_trace.strip():
            return np.nan, "missing", 0, np.nan
        try:
            points = json.loads(raw_trace)
        except Exception:
            return np.nan, "invalid trace JSON", 0, np.nan
    elif isinstance(raw_trace, list):
        points = raw_trace
    else:
        return np.nan, "unsupported trace type", 0, np.nan

    if not isinstance(points, list):
        return np.nan, "invalid trace structure", 0, np.nan

    times = []
    powers = []

    for point in points:
        if not isinstance(point, dict):
            continue
        times.append(point.get("t_s"))
        powers.append(point.get("power_w"))

    energy_j, method, point_count, duration_s = (
        integrate_power_trace_joules(
            times,
            powers,
        )
    )
    return (
        energy_j,
        "sample trace JSON: " + method,
        point_count,
        duration_s,
    )


def energy_rows_from_samples(
    path: Path,
) -> pd.DataFrame:
    """Load direct sample energy or derive it from sample power and latency."""
    samples = pd.read_csv(path)

    if not is_gpu_trace_frame(samples, path):
        return pd.DataFrame()

    if "dataset" not in samples.columns:
        return pd.DataFrame()

    samples = ensure_gpu_trace_model_column(
        samples,
        path,
    )
    samples["dataset"] = samples["dataset"].map(
        normalize_dataset
    )
    samples = samples[
        samples["dataset"].isin(
            BENCHMARK_ORDER_ALPHABETICAL
        )
    ].copy()

    direct_columns = [
        column
        for column in GPU_SAMPLE_ENERGY_J_CANDIDATES
        if column in samples.columns
    ]
    power_columns = [
        column
        for column in GPU_SAMPLE_POWER_W_CANDIDATES
        if column in samples.columns
    ]
    latency_columns = [
        column
        for column in GPU_SAMPLE_LATENCY_MS_CANDIDATES
        if column in samples.columns
    ]

    direct_energy = pd.Series(
        np.nan,
        index=samples.index,
        dtype=float,
    )
    if direct_columns:
        direct_energy = (
            samples[direct_columns]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .bfill(axis=1)
            .iloc[:, 0]
        )

    derived_energy = pd.Series(
        np.nan,
        index=samples.index,
        dtype=float,
    )
    if power_columns and latency_columns:
        average_power = (
            samples[power_columns]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .bfill(axis=1)
            .iloc[:, 0]
        )
        latency_ms = (
            samples[latency_columns]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .bfill(axis=1)
            .iloc[:, 0]
        )
        derived_energy = (
            average_power
            * latency_ms
            / 1000.0
        )

    trace_energy = pd.Series(
        np.nan,
        index=samples.index,
        dtype=float,
    )
    trace_method = pd.Series(
        "",
        index=samples.index,
        dtype=object,
    )
    trace_points = pd.Series(
        np.nan,
        index=samples.index,
        dtype=float,
    )
    trace_duration = pd.Series(
        np.nan,
        index=samples.index,
        dtype=float,
    )

    unresolved = (
        direct_energy.isna()
        & derived_energy.isna()
    )
    if (
        unresolved.any()
        and "system_metrics_trace_json"
        in samples.columns
    ):
        parsed = samples.loc[
            unresolved,
            "system_metrics_trace_json",
        ].map(
            parse_energy_from_sample_trace
        )
        trace_energy.loc[unresolved] = (
            parsed.map(lambda item: item[0])
        )
        trace_method.loc[unresolved] = (
            parsed.map(lambda item: item[1])
        )
        trace_points.loc[unresolved] = (
            parsed.map(lambda item: item[2])
        )
        trace_duration.loc[unresolved] = (
            parsed.map(lambda item: item[3])
        )

    samples["energy_j"] = (
        direct_energy
        .fillna(derived_energy)
        .fillna(trace_energy)
    )
    samples["energy_source"] = np.select(
        [
            direct_energy.notna(),
            derived_energy.notna(),
            trace_energy.notna(),
        ],
        [
            "direct sample energy_j",
            "sample average power × latency",
            trace_method,
        ],
        default="missing",
    )
    samples["power_trace_point_count"] = (
        trace_points
    )
    samples["integrated_duration_s"] = (
        trace_duration
    )
    samples["source_priority"] = np.select(
        [
            direct_energy.notna(),
            derived_energy.notna(),
            trace_energy.notna(),
        ],
        [1, 2, 3],
        default=9,
    )
    samples["source_file"] = str(path)
    samples["request_identity"] = (
        create_energy_request_identity(
            samples,
            path,
        )
    )

    return samples[
        samples["energy_j"].notna()
    ].copy()


def build_gpu_request_energy(
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build one GPU-energy observation per request."""
    candidates = gpu_trace_source_candidates()
    parts = []
    source_attempt_rows = []

    loaders = [
        (
            "system_traces.csv integration",
            candidates["system_traces"],
            energy_rows_from_system_traces,
        ),
        (
            "samples.csv fallback",
            candidates["samples"],
            energy_rows_from_samples,
        ),
    ]

    for source_kind, paths, loader in loaders:
        for path in paths:
            try:
                rows = loader(path)
                usable_count = (
                    int(
                        pd.to_numeric(
                            rows.get(
                                "energy_j",
                                pd.Series(dtype=float),
                            ),
                            errors="coerce",
                        )
                        .notna()
                        .sum()
                    )
                    if not rows.empty
                    else 0
                )
                source_attempt_rows.append({
                    "source_kind": source_kind,
                    "path": str(path),
                    "status": (
                        "usable"
                        if usable_count > 0
                        else "no usable request energy"
                    ),
                    "usable_request_energy_rows": (
                        usable_count
                    ),
                })
                if usable_count > 0:
                    parts.append(rows)

            except Exception as exc:
                source_attempt_rows.append({
                    "source_kind": source_kind,
                    "path": str(path),
                    "status": f"error: {exc}",
                    "usable_request_energy_rows": 0,
                })

    source_audit = pd.DataFrame(
        source_attempt_rows
    )
    source_audit.to_csv(
        CHECK_DIR
        / "gpu_request_energy_source_audit.csv",
        index=False,
    )

    if not parts:
        raise FileNotFoundError(
            "GPU request energy could not be derived. A usable "
            "system_traces.csv with t_s and power_w, or samples.csv "
            "with energy_j / power_w_avg and latency, is required."
        )

    energy = pd.concat(
        parts,
        ignore_index=True,
        sort=False,
    )
    energy["dataset"] = energy["dataset"].map(
        normalize_dataset
    )
    energy["energy_j"] = pd.to_numeric(
        energy["energy_j"],
        errors="coerce",
    )
    energy = energy[
        energy["dataset"].isin(
            BENCHMARK_ORDER_ALPHABETICAL
        )
        & energy["energy_j"].notna()
        & energy["energy_j"].ge(0)
    ].copy()

    metadata = energy.apply(
        resolve_metadata,
        axis=1,
    )
    energy = pd.concat(
        [
            energy.reset_index(drop=True),
            metadata.reset_index(drop=True),
        ],
        axis=1,
    )

    energy = energy.sort_values(
        [
            "source_priority",
            "source_file",
        ]
    )
    before_deduplication = len(energy)
    energy = energy.drop_duplicates(
        subset=[
            "model_display_canonical",
            "dataset",
            "request_identity",
        ],
        keep="first",
    )
    duplicates_removed = (
        before_deduplication - len(energy)
    )

    valid = energy[
        energy["model_display_canonical"].notna()
        & energy["dataset"].notna()
        & energy["energy_j"].notna()
    ].copy()

    if valid.empty:
        raise ValueError(
            "No valid GPU request-energy measurements remain."
        )

    statistics = (
        valid.groupby(
            [
                "model_display_canonical",
                "dataset",
            ],
            dropna=False,
        )["energy_j"]
        .agg(
            request_count="count",
            mean_energy_j="mean",
            median_energy_j="median",
            sd_energy_j="std",
            minimum_energy_j="min",
            p05_energy_j=lambda values: (
                values.quantile(0.05)
            ),
            q1_energy_j=lambda values: (
                values.quantile(0.25)
            ),
            q3_energy_j=lambda values: (
                values.quantile(0.75)
            ),
            p95_energy_j=lambda values: (
                values.quantile(0.95)
            ),
            maximum_energy_j="max",
        )
        .reset_index()
    )
    statistics["iqr_energy_j"] = (
        statistics["q3_energy_j"]
        - statistics["q1_energy_j"]
    )
    statistics["range_energy_j"] = (
        statistics["maximum_energy_j"]
        - statistics["minimum_energy_j"]
    )
    statistics["aggregation_definition"] = (
        "one energy observation per request"
    )
    statistics[
        "duplicates_removed_across_sources"
    ] = duplicates_removed

    source_counts = (
        valid.groupby(
            [
                "model_display_canonical",
                "dataset",
                "energy_source",
            ],
            dropna=False,
        )
        .size()
        .rename("request_count")
        .reset_index()
    )
    source_counts.to_csv(
        CHECK_DIR
        / "gpu_request_energy_source_counts.csv",
        index=False,
    )

    return valid, statistics, source_audit


def build_gpu_energy_model_order(
    statistics: pd.DataFrame,
) -> pd.DataFrame:
    """Shared Figure 2a/3b order: increasing Dolly mean energy."""
    dolly = statistics[
        statistics["dataset"].eq("dolly")
    ][
        [
            "model_display_canonical",
            "mean_energy_j",
        ]
    ].copy()

    if dolly.empty:
        raise ValueError(
            "GPU energy figures require Dolly request-energy values."
        )

    order_table = dolly.rename(
        columns={
            "model_display_canonical":
                "model_configuration",
            "mean_energy_j":
                "dolly_mean_energy_j",
        }
    )
    order_table["dolly_measurement_missing"] = (
        order_table["dolly_mean_energy_j"].isna()
    )
    order_table = (
        order_table.sort_values(
            [
                "dolly_measurement_missing",
                "dolly_mean_energy_j",
                "model_configuration",
            ],
            ascending=[True, True, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )
    order_table["model_order_position"] = (
        np.arange(len(order_table)) + 1
    )
    order_table["model_order_basis"] = (
        "increasing Dolly mean GPU energy per request"
    )
    return order_table


def plot_gpu_energy_grouped_bars(
    statistics: pd.DataFrame,
    order_table: pd.DataFrame,
    stem: str,
) -> pd.DataFrame:
    """Figure 2a: grouped mean-energy bars by model and benchmark."""
    available_benchmarks = [
        dataset
        for dataset in BENCHMARK_ORDER_ALPHABETICAL
        if dataset in statistics["dataset"].unique()
    ]
    model_order = order_table[
        "model_configuration"
    ].tolist()

    pivot = (
        statistics.pivot_table(
            index="model_display_canonical",
            columns="dataset",
            values="mean_energy_j",
            aggfunc="first",
        )
        .reindex(
            index=model_order,
            columns=available_benchmarks,
        )
    )

    model_count = len(model_order)
    workload_count = max(
        len(available_benchmarks),
        1,
    )
    x_positions = np.arange(model_count)
    group_width = 0.82
    bar_width = group_width / workload_count

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 4.35),
        constrained_layout=True,
    )

    for workload_index, dataset in enumerate(
        available_benchmarks
    ):
        values = pivot[
            dataset
        ].to_numpy(dtype=float)
        offsets = (
            x_positions
            - group_width / 2
            + bar_width / 2
            + workload_index * bar_width
        )
        mask = np.isfinite(values)

        ax.bar(
            offsets[mask],
            values[mask],
            width=bar_width * 0.92,
            label=DATASET_DISPLAY[dataset],
            color=DATASET_COLORS[dataset],
            hatch=DATASET_HATCHES[dataset],
            edgecolor="0.20",
            linewidth=0.60,
            zorder=3,
        )

    maximum = float(
        np.nanmax(pivot.to_numpy(dtype=float))
    )
    ax.set_ylim(0, maximum * 1.18)
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel(
        "Mean GPU energy per request (J)"
    )
    ax.set_xlabel(
        "Model configuration "
        "(increasing Dolly mean energy)"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=13,
                max_lines=3,
            )
            for model in model_order
        ],
        rotation=58,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15,
    )

    style_axis(ax, grid_axis="y")
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=4,
        frameon=False,
        columnspacing=0.8,
        handletextpad=0.35,
    )

    export = (
        pivot.reset_index()
        .rename(
            columns={
                "model_display_canonical":
                    "model_configuration"
            }
        )
    )
    export.insert(
        0,
        "model_order_position",
        np.arange(len(export)) + 1,
    )
    export.insert(
        1,
        "model_order_basis",
        "increasing Dolly mean GPU energy per request",
    )
    export.to_csv(
        TABLE_DIR / f"{stem}_values.csv",
        index=False,
    )
    order_table.to_csv(
        CHECK_DIR
        / f"{stem}_model_order_dolly.csv",
        index=False,
    )
    pd.DataFrame({
        "benchmark_order_position": np.arange(
            1,
            len(available_benchmarks) + 1,
        ),
        "benchmark": available_benchmarks,
        "benchmark_display": [
            DATASET_DISPLAY[dataset]
            for dataset in available_benchmarks
        ],
    }).to_csv(
        CHECK_DIR
        / f"{stem}_benchmark_order.csv",
        index=False,
    )

    save_figure(fig, stem)
    return export


def plot_gpu_energy_intervals(
    statistics: pd.DataFrame,
    order_table: pd.DataFrame,
    stem: str,
) -> pd.DataFrame:
    """Figure 3b: median energy with central-90% and min–max ranges."""
    available_benchmarks = [
        dataset
        for dataset in BENCHMARK_ORDER_ALPHABETICAL
        if dataset in statistics["dataset"].unique()
    ]
    model_order = order_table[
        "model_configuration"
    ].tolist()

    model_count = len(model_order)
    workload_count = max(
        len(available_benchmarks),
        1,
    )
    x_positions = np.arange(model_count)
    group_width = 0.78
    offset_width = group_width / workload_count

    marker_map = {
        "dolly": "o",
        "gsm8k": "s",
        "hellaswag": "^",
        "mmlu": "D",
    }

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 4.75),
        constrained_layout=True,
    )

    plotted_rows = []

    for workload_index, dataset in enumerate(
        available_benchmarks
    ):
        offset = (
            -group_width / 2
            + offset_width / 2
            + workload_index * offset_width
        )
        rows = (
            statistics[
                statistics["dataset"].eq(dataset)
            ]
            .set_index("model_display_canonical")
            .reindex(model_order)
        )
        positions = x_positions + offset
        valid_mask = rows[
            "median_energy_j"
        ].notna().to_numpy()

        if not valid_mask.any():
            continue

        x_valid = positions[valid_mask]
        rows_valid = rows.loc[
            rows["median_energy_j"].notna()
        ].copy()

        ax.vlines(
            x_valid,
            rows_valid["minimum_energy_j"],
            rows_valid["maximum_energy_j"],
            color=DATASET_COLORS[dataset],
            linewidth=0.70,
            alpha=0.72,
            zorder=2,
        )
        ax.vlines(
            x_valid,
            rows_valid["p05_energy_j"],
            rows_valid["p95_energy_j"],
            color=DATASET_COLORS[dataset],
            linewidth=2.35,
            alpha=0.96,
            zorder=3,
        )
        ax.scatter(
            x_valid,
            rows_valid["median_energy_j"],
            marker=marker_map[dataset],
            s=18,
            facecolor=DATASET_COLORS[dataset],
            edgecolor="0.18",
            linewidth=0.48,
            label=DATASET_DISPLAY[dataset],
            zorder=4,
        )

        for model, x_value, (_, row) in zip(
            rows_valid.index,
            x_valid,
            rows_valid.iterrows(),
        ):
            plotted_rows.append({
                "model_configuration": model,
                "dataset": dataset,
                "x_position": x_value,
                "request_count": row["request_count"],
                "minimum_energy_j": row["minimum_energy_j"],
                "p05_energy_j": row["p05_energy_j"],
                "median_energy_j": row["median_energy_j"],
                "p95_energy_j": row["p95_energy_j"],
                "maximum_energy_j": row["maximum_energy_j"],
                "iqr_energy_j": row["iqr_energy_j"],
                "range_energy_j": row["range_energy_j"],
            })

    plotted = pd.DataFrame(plotted_rows)
    if plotted.empty:
        raise ValueError(
            "Figure 3b has no plottable request-energy statistics."
        )

    y_min = float(plotted["minimum_energy_j"].min())
    y_max = float(plotted["maximum_energy_j"].max())
    y_range = max(y_max - y_min, 1.0)
    padding = max(y_range * 0.10, 0.50)

    ax.set_ylim(
        max(0.0, y_min - padding),
        y_max + padding,
    )
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel(
        "GPU energy per request (J)"
    )
    ax.set_xlabel(
        "Model configuration "
        "(same energy order as Figure 2a)"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=13,
                max_lines=3,
            )
            for model in model_order
        ],
        rotation=58,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15,
    )
    ax.text(
        0.995,
        0.02,
        (
            "Marker = median; thick line = 5th–95th percentile; "
            "thin line = observed min–max"
        ),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=5.50,
        color="0.35",
    )

    style_axis(ax, grid_axis="y")
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=4,
        frameon=False,
        columnspacing=0.90,
        handletextpad=0.35,
    )

    plotted = plotted.merge(
        order_table[
            [
                "model_configuration",
                "model_order_position",
                "model_order_basis",
            ]
        ],
        on="model_configuration",
        how="left",
    ).sort_values(
        [
            "model_order_position",
            "dataset",
        ]
    )
    plotted.to_csv(
        TABLE_DIR / f"{stem}_statistics.csv",
        index=False,
    )
    save_figure(fig, stem)
    return plotted


ENERGY_SPLIT_A_DATASETS = ["dolly", "gsm8k"]
ENERGY_SPLIT_B_DATASETS = ["hellaswag", "mmlu"]


def ensure_energy_panel_datasets(
    statistics: pd.DataFrame,
    datasets: list[str],
    panel_name: str,
) -> list[str]:
    available = [
        dataset
        for dataset in datasets
        if dataset in statistics["dataset"].unique()
    ]
    if not available:
        raise ValueError(
            f"{panel_name} cannot be generated because none of the requested "
            f"datasets are available: {datasets}"
        )
    return available


def plot_gpu_energy_grouped_bars_split(
    statistics: pd.DataFrame,
    order_table: pd.DataFrame,
    datasets: list[str],
    stem: str,
    xlabel_suffix: str,
) -> pd.DataFrame:
    """Split Figure 2a into panel-specific grouped bars."""
    available_datasets = ensure_energy_panel_datasets(
        statistics,
        datasets,
        stem,
    )
    model_order = order_table["model_configuration"].tolist()

    pivot = (
        statistics.pivot_table(
            index="model_display_canonical",
            columns="dataset",
            values="mean_energy_j",
            aggfunc="first",
        )
        .reindex(
            index=model_order,
            columns=available_datasets,
        )
    )

    model_count = len(model_order)
    workload_count = max(len(available_datasets), 1)
    x_positions = np.arange(model_count)
    group_width = 0.82
    bar_width = group_width / workload_count

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 4.35),
        constrained_layout=True,
    )

    for workload_index, dataset in enumerate(available_datasets):
        values = pivot[dataset].to_numpy(dtype=float)
        offsets = (
            x_positions
            - group_width / 2
            + bar_width / 2
            + workload_index * bar_width
        )
        mask = np.isfinite(values)

        ax.bar(
            offsets[mask],
            values[mask],
            width=bar_width * 0.92,
            label=DATASET_DISPLAY[dataset],
            color=DATASET_COLORS[dataset],
            hatch=DATASET_HATCHES[dataset],
            edgecolor="0.20",
            linewidth=0.60,
            zorder=3,
        )

    finite_values = pivot.to_numpy(dtype=float)
    max_value = float(np.nanmax(finite_values))
    ax.set_ylim(0, max_value * 1.18 if max_value > 0 else 1.0)
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel("Mean GPU energy per request (J)")
    ax.set_xlabel(
        "Model configuration "
        f"({xlabel_suffix})"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=13,
                max_lines=3,
            )
            for model in model_order
        ],
        rotation=58,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15,
    )

    style_axis(ax, grid_axis="y")
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=len(available_datasets),
        frameon=False,
        columnspacing=0.8,
        handletextpad=0.35,
    )

    export = (
        pivot.reset_index()
        .rename(
            columns={
                "model_display_canonical": "model_configuration"
            }
        )
    )
    export.insert(
        0,
        "model_order_position",
        np.arange(len(export)) + 1,
    )
    export.insert(
        1,
        "model_order_basis",
        "increasing Dolly mean GPU energy per request",
    )
    export.to_csv(
        TABLE_DIR / f"{stem}_values.csv",
        index=False,
    )
    order_table.to_csv(
        CHECK_DIR / f"{stem}_model_order_dolly.csv",
        index=False,
    )
    pd.DataFrame({
        "benchmark_order_position": np.arange(
            1,
            len(available_datasets) + 1,
        ),
        "benchmark": available_datasets,
        "benchmark_display": [
            DATASET_DISPLAY[dataset]
            for dataset in available_datasets
        ],
    }).to_csv(
        CHECK_DIR / f"{stem}_benchmark_order.csv",
        index=False,
    )

    save_figure(fig, stem)
    return export


def plot_gpu_energy_boxplots_split(
    request_energy: pd.DataFrame,
    order_table: pd.DataFrame,
    datasets: list[str],
    stem: str,
    xlabel_suffix: str,
) -> pd.DataFrame:
    """Split Figure 3b into panel-specific grouped request-energy boxplots."""
    available_datasets = ensure_energy_panel_datasets(
        request_energy,
        datasets,
        stem,
    )
    model_order = order_table["model_configuration"].tolist()

    model_count = len(model_order)
    workload_count = max(len(available_datasets), 1)
    x_positions = np.arange(model_count)
    group_width = 0.82
    box_width = group_width / workload_count

    fig, ax = plt.subplots(
        figsize=(DOUBLE_COLUMN_WIDTH_IN, 4.55),
        constrained_layout=True,
    )

    export_rows = []
    all_values = []

    for workload_index, dataset in enumerate(available_datasets):
        offsets = (
            x_positions
            - group_width / 2
            + box_width / 2
            + workload_index * box_width
        )

        distributions = []
        positions = []

        for model_index, model in enumerate(model_order):
            values = (
                request_energy.loc[
                    request_energy["model_display_canonical"].eq(model)
                    & request_energy["dataset"].eq(dataset),
                    "energy_j",
                ]
                .dropna()
                .astype(float)
                .to_numpy()
            )

            if len(values) == 0:
                continue

            distributions.append(values)
            positions.append(float(offsets[model_index]))
            all_values.extend(values.tolist())

            export_rows.extend(
                {
                    "model_configuration": model,
                    "dataset": dataset,
                    "energy_j": value,
                }
                for value in values
            )

        if not distributions:
            continue

        boxplot = ax.boxplot(
            distributions,
            positions=positions,
            widths=box_width * 0.76,
            patch_artist=True,
            whis=1.5,
            showfliers=True,
            flierprops={
                "marker": ".",
                "markersize": 1.6,
                "markerfacecolor": "0.35",
                "markeredgecolor": "0.35",
                "alpha": 0.45,
            },
            manage_ticks=False,
            zorder=3,
        )

        for patch in boxplot["boxes"]:
            patch.set_facecolor(DATASET_COLORS[dataset])
            patch.set_edgecolor("0.20")
            patch.set_linewidth(0.60)
            patch.set_hatch(DATASET_HATCHES[dataset])
            patch.set_alpha(0.96)

        for element_name in ["whiskers", "caps", "medians"]:
            for artist in boxplot[element_name]:
                artist.set_color("0.20")
                artist.set_linewidth(
                    0.60 if element_name != "medians" else 0.95
                )

    if not all_values:
        raise ValueError(
            f"{stem} has no request-energy distributions to plot."
        )

    max_value = float(np.nanmax(all_values))
    ax.set_ylim(0, max(max_value * 1.10, 1.0))
    ax.set_xlim(-0.55, model_count - 0.45)
    ax.set_ylabel("GPU energy per request (J)")
    ax.set_xlabel(
        "Model configuration "
        f"({xlabel_suffix})"
    )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            publication_model_label(
                model,
                width=13,
                max_lines=3,
            )
            for model in model_order
        ],
        rotation=58,
        ha="right",
        rotation_mode="anchor",
        fontsize=5.15,
    )

    style_axis(ax, grid_axis="y")

    legend_handles = [
        Patch(
            facecolor=DATASET_COLORS[dataset],
            edgecolor="0.20",
            hatch=DATASET_HATCHES[dataset],
            linewidth=0.60,
            label=DATASET_DISPLAY[dataset],
        )
        for dataset in available_datasets
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=len(available_datasets),
        frameon=False,
        columnspacing=0.8,
        handletextpad=0.35,
    )

    export = pd.DataFrame(export_rows)
    export.to_csv(
        TABLE_DIR / f"{stem}_measurement_values.csv",
        index=False,
    )
    order_table.to_csv(
        CHECK_DIR / f"{stem}_model_order_dolly.csv",
        index=False,
    )
    pd.DataFrame({
        "benchmark_order_position": np.arange(
            1,
            len(available_datasets) + 1,
        ),
        "benchmark": available_datasets,
        "benchmark_display": [
            DATASET_DISPLAY[dataset]
            for dataset in available_datasets
        ],
    }).to_csv(
        CHECK_DIR / f"{stem}_benchmark_order.csv",
        index=False,
    )

    save_figure(fig, stem)
    return export


## 17. Generate split GPU energy figures last and refresh the manifest

The final cell derives per-request GPU energy once and reuses it for four split energy figures: Figure 2a1 (Dolly + GSM8K bars), Figure 2a2 (HellaSwag + MMLU bars), Figure 3b1 (Dolly + GSM8K boxplots), and Figure 3b2 (HellaSwag + MMLU boxplots). Figure 3a is still generated last among the main GPU resource figures.


In [ ]:

from time import perf_counter

GENERATE_FIGURE_2A1 = True
GENERATE_FIGURE_2A2 = True
GENERATE_FIGURE_3B1 = True
GENERATE_FIGURE_3B2 = True
GENERATE_FIGURE_3A = True
GENERATE_EXPLORATORY_VRAM_DELTA = True

gpu_task_start = perf_counter()

print(
    "[GPU resources 1/10] Loading and normalizing GPU VRAM measurements...",
    flush=True,
)
stage_start = perf_counter()
(
    gpu_vram_measurements,
    gpu_vram_measurement_statistics,
) = build_gpu_vram_measurements()
print(
    f"[GPU resources 1/10] Loaded {len(gpu_vram_measurements):,} VRAM measurements in "
    f"{perf_counter() - stage_start:.1f} s.",
    flush=True,
)

print(
    "[GPU resources 2/10] Deriving one GPU-energy value per request...",
    flush=True,
)
stage_start = perf_counter()
(
    gpu_request_energy,
    gpu_request_energy_statistics,
    gpu_request_energy_source_audit,
) = build_gpu_request_energy()
gpu_energy_order = build_gpu_energy_model_order(
    gpu_request_energy_statistics
)
gpu_request_energy_statistics.to_csv(
    TABLE_DIR / "gpu_request_energy_statistics.csv",
    index=False,
)
gpu_request_energy[
    [
        column
        for column in [
            "platform",
            "dataset",
            "model",
            "model_display_canonical",
            "run_id",
            "request_identity",
            "energy_j",
            "energy_source",
            "power_trace_point_count",
            "integrated_duration_s",
            "source_file",
        ]
        if column in gpu_request_energy.columns
    ]
].to_csv(
    CHECK_DIR / "gpu_request_energy_request_audit.csv",
    index=False,
)
print(
    f"[GPU resources 2/10] Derived {len(gpu_request_energy):,} request-energy observations in "
    f"{perf_counter() - stage_start:.1f} s.",
    flush=True,
)

if GENERATE_FIGURE_2A1:
    print("[GPU resources 3/10] Generating Figure 2a1...", flush=True)
    stage_start = perf_counter()
    figure_02a1_values = plot_gpu_energy_grouped_bars_split(
        gpu_request_energy_statistics,
        gpu_energy_order,
        ENERGY_SPLIT_A_DATASETS,
        "figure_02a1_gpu_mean_energy_per_request_dolly_gsm8k",
        "increasing Dolly mean energy; Dolly + GSM8K only",
    )
    print(
        f"[GPU resources 3/10] Figure 2a1 completed in {perf_counter() - stage_start:.1f} s.",
        flush=True,
    )
else:
    print("[GPU resources 3/10] Figure 2a1 skipped.", flush=True)

if GENERATE_FIGURE_2A2:
    print("[GPU resources 4/10] Generating Figure 2a2...", flush=True)
    stage_start = perf_counter()
    figure_02a2_values = plot_gpu_energy_grouped_bars_split(
        gpu_request_energy_statistics,
        gpu_energy_order,
        ENERGY_SPLIT_B_DATASETS,
        "figure_02a2_gpu_mean_energy_per_request_hellaswag_mmlu",
        "same model order; HellaSwag + MMLU only",
    )
    print(
        f"[GPU resources 4/10] Figure 2a2 completed in {perf_counter() - stage_start:.1f} s.",
        flush=True,
    )
else:
    print("[GPU resources 4/10] Figure 2a2 skipped.", flush=True)

if GENERATE_FIGURE_3B1:
    print("[GPU resources 5/10] Generating Figure 3b1...", flush=True)
    stage_start = perf_counter()
    figure_03b1_values = plot_gpu_energy_boxplots_split(
        gpu_request_energy,
        gpu_energy_order,
        ENERGY_SPLIT_A_DATASETS,
        "figure_03b1_gpu_energy_per_request_boxplots_dolly_gsm8k",
        "same model order as Figure 2a1",
    )
    print(
        f"[GPU resources 5/10] Figure 3b1 completed in {perf_counter() - stage_start:.1f} s.",
        flush=True,
    )
else:
    print("[GPU resources 5/10] Figure 3b1 skipped.", flush=True)

if GENERATE_FIGURE_3B2:
    print("[GPU resources 6/10] Generating Figure 3b2...", flush=True)
    stage_start = perf_counter()
    figure_03b2_values = plot_gpu_energy_boxplots_split(
        gpu_request_energy,
        gpu_energy_order,
        ENERGY_SPLIT_B_DATASETS,
        "figure_03b2_gpu_energy_per_request_boxplots_hellaswag_mmlu",
        "same model order as Figure 2a2",
    )
    print(
        f"[GPU resources 6/10] Figure 3b2 completed in {perf_counter() - stage_start:.1f} s.",
        flush=True,
    )
else:
    print("[GPU resources 6/10] Figure 3b2 skipped.", flush=True)

if GENERATE_EXPLORATORY_VRAM_DELTA:
    print(
        "[GPU resources 7/10] Generating exploratory run-min-adjusted VRAM box plots...",
        flush=True,
    )
    stage_start = perf_counter()
    (
        gpu_vram_run_min_adjusted,
        gpu_vram_baseline_audit,
    ) = build_run_min_adjusted_vram(
        gpu_vram_measurements
    )
    figure_s03a_values = plot_run_min_adjusted_vram_boxplots(
        gpu_vram_run_min_adjusted,
        build_gpu_vram_model_order(
            gpu_vram_measurement_statistics
        ),
        "figure_s03a_gpu_run_min_adjusted_vram_boxplots",
    )
    gpu_vram_baseline_audit.to_csv(
        CHECK_DIR / "figure_s03a_gpu_run_min_baseline_audit.csv",
        index=False,
    )
    print(
        f"[GPU resources 7/10] Exploratory supplement completed in {perf_counter() - stage_start:.1f} s.",
        flush=True,
    )
else:
    print("[GPU resources 7/10] Exploratory VRAM-delta figure skipped.", flush=True)

if GENERATE_FIGURE_3A:
    print(
        "[GPU resources 8/10] Generating Figure 3a absolute VRAM footprint last...",
        flush=True,
    )
    stage_start = perf_counter()
    (
        figure_03a_statistics,
        figure_03a_order,
    ) = plot_gpu_vram_footprint_intervals(
        gpu_vram_measurement_statistics,
        "figure_03a_gpu_vram_footprint_intervals",
    )
    print(
        f"[GPU resources 8/10] Figure 3a completed in {perf_counter() - stage_start:.1f} s.",
        flush=True,
    )
else:
    print("[GPU resources 8/10] Figure 3a skipped.", flush=True)

print("[GPU resources 9/10] Writing the VRAM trace audit...", flush=True)
stage_start = perf_counter()
gpu_vram_measurements[
    [
        column
        for column in [
            "platform",
            "dataset",
            "model",
            "model_display_canonical",
            "run_id",
            "sample_idx",
            "request_id",
            "trace_idx",
            "t_s",
            "vram_used_mb",
            "vram_used_gib",
            "trace_source",
            "source_file",
        ]
        if column in gpu_vram_measurements.columns
    ]
].to_csv(
    CHECK_DIR / "figure_03a_gpu_vram_measurement_trace_audit.csv",
    index=False,
)
print(
    f"[GPU resources 9/10] Audit written in {perf_counter() - stage_start:.1f} s.",
    flush=True,
)

print(
    f"[GPU resources] Total time: {perf_counter() - gpu_task_start:.1f} s.",
    flush=True,
)

print("[Final 10/10] Refreshing the output manifest...", flush=True)
manifest_rows = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        manifest_rows.append({
            "relative_path": str(path.relative_to(OUTPUT_DIR)),
            "type": path.suffix.lower().lstrip("."),
            "size_kb": round(path.stat().st_size / 1024, 1),
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(
    CHECK_DIR / "output_manifest.csv",
    index=False,
)
display(manifest)
print(
    f"Generated {len(manifest)} files in {OUTPUT_DIR}",
    flush=True,
)
